In [1]:
# =====================================================================
# Cell 1: Cài đặt thư viện cần thiết cho PIPNet, TorchLM, OpenCV &
#         TensorBoard trên Google Colab và kiểm tra GPU
# =====================================================================
import importlib.util
import platform
import subprocess
import sys

if platform.system() == "Linux":
    print("Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True,
    )

_needed_pkgs = [
    pkg for mod, pkg in [
        ("torchlm", "torchlm"),
        ("kagglehub", "kagglehub"),
        ("cv2", "opencv-python"),
        ("tensorboard", "tensorboard"),
        ("tqdm", "tqdm"),
        ("yaml", "pyyaml"),
        ("scipy", "scipy"),
    ]
    if importlib.util.find_spec(mod) is None
]
if _needed_pkgs:
    print(f"Đang cài đặt các thư viện còn thiếu: {_needed_pkgs}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", *_needed_pkgs
    ])
else:
    print("Tất cả các thư viện cần thiết đã được cài đặt sẵn.")

import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")

Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...
Đang cài đặt các thư viện còn thiếu: ['torchlm']
PyTorch: 2.14.1+cu130 | CUDA available: True
GPU: NVIDIA L4
Cài đặt hoàn tất! Môi trường đã sẵn sàng.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!cp /content/drive/MyDrive/LiteFat_v1/litefat_v1_cache_pt.zip /content
!unzip litefat_v1_cache_pt.zip

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nyvantran6634/litefatv1", output_dir="./kaggle")

print("Path to dataset files:", path)

In [5]:
# =====================================================================
# File: config.py
# Dự án: Driver Guardian - Module LiteFat_v1 (KSS 1-9, Sliding 60s/30s)
# Tác vụ: Tập trung toàn bộ cấu hình (TrainConfig) cho dữ liệu, tiền xử lý,
#         lấy mẫu 10 FPS, kiến trúc LiteFatV1, tối ưu hóa và LiteFatLoss.
# =====================================================================
from __future__ import annotations

import dataclasses
import json
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union


@dataclass
class TrainConfig:
    """
    Cấu hình tập trung cho toàn bộ pipeline huấn luyện, kiểm định và suy luận LiteFatV1 (KSS 1-9):
      - Dữ liệu dạng E:\\LSTM\\data_processed_v1 đọc từ manifest .csv hoặc .json.
      - Lấy mẫu chuẩn 10 FPS (time_interval = 0.1s), giữ toàn bộ chiều dài mẫu (>60s)
        để LiteFatV1 thực thi Sliding Window (60s window / 30s stride) tích lũy đặc trưng.
      - Hỗ trợ đầy đủ cả Landmark-Only và Multimodal (MobileNetV3 + FiLM).
    """

    # =================================================================
    # 1. ĐƯỜNG DẪN DỮ LIỆU, MANIFEST & LƯU TRỮ
    # =================================================================
    dataset_dir: str = f"./kaggle/"
    manifest_filename: str = "dataset_merged_split.csv"
    cache_dir: Optional[str] = "/content/cache_pt"
    use_cache: bool = True
    selected_datasets: Optional[List[str]] = None

    checkpoint_dir: str = "./checkpoints/LiteFat_v1"
    log_dir: str = "./checkpoints/LiteFat_v1/logs"

    # =================================================================
    # 2. CHẾ ĐỘ ĐA PHƯƠNG THỨC & TRÍCH XUẤT PIPNET
    # =================================================================
    use_visual_context: bool = False
    visual_frame_stride: int = 5
    letterbox_size: Tuple[int, int] = (640, 640)
    face_size: Tuple[int, int] = (224, 224)

    pipnet_backbone: str = "resnet18"
    conf_threshold: float = 0.5
    nms_threshold: float = 0.3
    bbox_margin: float = 0.15
    box_smooth_alpha: float = 0.7

    # =================================================================
    # 3. LẤY MẪU THỜI GIAN & SLIDING WINDOW (10 FPS, 60s / 30s)
    # =================================================================
    fps: int = 10
    time_interval: float = 0.1
    # None: lấy toàn bộ video dài >60s để model tự chạy sliding window 60s/30s.
    # Có thể đặt giá trị (vd: 120.0) nếu muốn cắt cửa sổ cố định khi train.
    dataset_window_sec: Optional[float] = None
    eval_window_mode: str = "full"

    # Tham số cửa sổ trượt của kiến trúc LiteFatV1 và LiteFatLoss
    window_sec: float = 60.0
    stride_sec: float = 30.0

    # =================================================================
    # 4. TĂNG CƯỜNG DỮ LIỆU ĐA PHƯƠNG THỨC (DATA AUGMENTATION)
    # =================================================================
    augment_train: bool = True
    flip_prob: float = 0.5
    scale_jitter: float = 0.08
    shift_jitter: float = 0.05
    roll_deg: float = 12.0
    temporal_dropout: float = 0.05
    landmark_noise_std: float = 0.0003
    color_jitter: float = 0.2
    spatial_dropout: float = 0.15

    # =================================================================
    # 5. THAM SỐ KIẾN TRÚC MÔ HÌNH LITEFAT_V1 (KSS 1-9)
    # =================================================================
    num_nodes: int = 68
    feature_dim: int = 64
    mobilenet_proj_hidden: int = 192
    node_embed_dim: int = 16
    residual_channels: int = 64
    dilation_channels: int = 64
    skip_channels: int = 128
    end_channels: int = 256
    num_classes: int = 9
    kss_min: float = 1.0
    kss_max: float = 9.0
    num_blocks: int = 3
    layers_per_block: int = 4
    kernel_size: int = 3
    dropout: float = 0.3
    pretrained_backbone: bool = True
    static_shape_ratio: float = 0.20

    vertical_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (37, 41), (38, 40), (43, 47), (44, 46),
        (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65),
    ])
    cross_region_pairs: List[Tuple[int, int]] = field(default_factory=lambda: [
        (19, 37), (24, 44), (30, 39), (30, 42), (57, 8), (51, 33),
    ])

    # =================================================================
    # 6. THAM SỐ HUẤN LUYỆN, TỐI ƯU HÓA & CHECKPOINT
    # =================================================================
    epochs: int = 30
    batch_size: int = 2
    grad_accum_steps: int = 8
    num_workers: int = 0
    pin_memory: bool = True
    seed: int = 42

    subject_balanced_sampling: bool = True
    use_manifest_sample_weights: bool = True
    drop_single_class_subjects: bool = False

    ema_decay: float = 0.995
    use_ema_eval: bool = True
    early_stop_patience: int = 8

    lr: float = 3e-4
    backbone_lr: float = 3e-5
    weight_decay: float = 2e-3
    freeze_backbone_epochs: int = 5
    unfreeze_last_n_blocks: int = 3
    warmup_epochs: int = 2
    pretrained_lr_scale: float = 0.25
    grad_clip: float = 1.0
    use_amp: bool = True

    save_freq: int = 1
    resume: Optional[str] = None
    pretrained_from: Optional[str] = None

    # =================================================================
    # 7. THAM SỐ HÀM MẤT MÁT HYBRID KSS LOSS (LITEFATLOSS)
    # =================================================================
    clip_reg_weight: float = 1.0
    clip_cls_weight: float = 0.5
    frame_loss_weight: float = 0.3
    window_loss_weight: float = 0.25
    window_smooth_weight: float = 0.05

    huber_beta: float = 0.5
    ordinal_sigma: float = 0.85
    label_smoothing: float = 0.05
    alert_kss_threshold: float = 4.0
    mid_kss_threshold: float = 6.0
    topk_mid_ratio: float = 0.70
    topk_drowsy_ratio: float = 0.45
    smooth_margin: float = 1.0

    # =================================================================
    # 8. THAM SỐ DOMAIN-ADVERSARIAL TRAINING (REGIME-CONDITIONED DANN)
    # =================================================================
    use_dann: bool = True
    dann_weight: float = 0.05
    dann_max_alpha: float = 0.20

    def to_dict(self) -> Dict[str, Any]:
        """Chuyển cấu hình thành từ điển thuần JSON-serializable."""
        return asdict(self)

    def save_json(self, path: Union[str, Path]) -> None:
        """Lưu cấu hình ra tệp JSON để tái lập thực nghiệm."""
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, path: Union[str, Path]) -> TrainConfig:
        """Nạp cấu hình từ tệp JSON, tự động chuẩn hóa kiểu tuple cho kích thước và cặp đỉnh."""
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        valid_fields = {f.name for f in dataclasses.fields(cls)}
        filtered: Dict[str, Any] = {k: v for k, v in data.items() if k in valid_fields}

        for size_key in ("letterbox_size", "face_size"):
            if size_key in filtered and isinstance(filtered[size_key], list):
                filtered[size_key] = tuple(int(x) for x in filtered[size_key])

        for pairs_key in ("vertical_pairs", "cross_region_pairs"):
            if pairs_key in filtered and isinstance(filtered[pairs_key], list):
                filtered[pairs_key] = [(int(p[0]), int(p[1])) for p in filtered[pairs_key]]

        return cls(**filtered)

In [8]:
# =====================================================================
# File: ultis.py
# Dự án: Driver Guardian - Module LiteFat_v1 (KSS 1-9, Sliding 60s/30s)
# Tác vụ: Thư viện hàm phụ trợ dùng chung:
#           1. Tiền xử lý ảnh, cắt vùng mặt & augmentation đối xứng 68 mốc.
#           2. Khởi tạo seed, logging, lưu tệp an toàn (atomic_save) & ModelEMA.
#           3. Tính toán chỉ số đánh giá KSS 1-9 (MAE, RMSE, Within-1, Exact,
#              QWK, Pearson/Spearman, 3-Regime, Binary Drowsy F1/AUC, val_score).
#           4. Nạp & khôi phục checkpoint (load_checkpoint, safe_load_optimizer,
#              load_pretrained_weights).
# =====================================================================
from __future__ import annotations

import json
import logging
import math
import os
import random
import sys
import tempfile
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple, Union

import cv2
import numpy as np
import torch
import torch.nn as nn

# Bảng hoán vị đối xứng trái - phải cho 68 điểm mốc khuôn mặt chuẩn 300-W
FLIP_MAP_68 = [
    16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0,
    26, 25, 24, 23, 22, 21, 20, 19, 18, 17,
    27, 28, 29, 30, 35, 34, 33, 32, 31,
    45, 44, 43, 42, 47, 46, 39, 38, 37, 36, 41, 40,
    54, 53, 52, 51, 50, 49, 48, 59, 58, 57, 56, 55,
    64, 63, 62, 61, 60, 67, 66, 65,
]


# =====================================================================
# 1. TIỆN ÍCH XỬ LÝ ẢNH & TĂNG CƯỜNG DỮ LIỆU ĐA PHƯƠNG THỨC
# =====================================================================
def _parse_size(size: Optional[Union[int, Tuple[int, int], List[int]]]) -> Tuple[int, int]:
    """Chuẩn hóa kích thước về tuple (width, height)."""
    if size is None:
        return (640, 640)
    if isinstance(size, int):
        return (size, size)
    if isinstance(size, (tuple, list)) and len(size) == 2:
        return (int(size[0]), int(size[1]))
    raise ValueError(f"Kích thước không hợp lệ: {size}. Phải là int hoặc tuple (w, h).")


def letterbox(
        image: np.ndarray,
        new_size: Optional[Union[int, Tuple[int, int], List[int]]] = (640, 640),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """Resize ảnh giữ nguyên tỷ lệ khung hình gốc kèm đệm màu viền về kích thước đích (w, h)."""
    target_w, target_h = _parse_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(image, (nw, nh), interpolation=interp)

    if nw == target_w and nh == target_h:
        return resized

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    canvas[y:y + nh, x:x + nw] = resized
    return canvas


def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Tính Intersection over Union (IoU) giữa 2 hộp giới hạn [x1, y1, x2, y2]."""
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])

    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return float(inter / union)


def crop_faces_from_boxes(
        frames: List[np.ndarray],
        crop_boxes: Union[np.ndarray, torch.Tensor],
        face_size: Tuple[int, int] = (224, 224),
) -> torch.Tensor:
    """Cắt vùng khuôn mặt từ crop_boxes [T, 4], chuyển RGB và chuẩn hóa ImageNet."""
    T = len(frames)
    crop_w, crop_h = _parse_size(face_size)
    if T == 0:
        return torch.empty((0, 3, crop_h, crop_w), dtype=torch.float32)

    if isinstance(crop_boxes, torch.Tensor):
        crop_boxes = crop_boxes.detach().cpu().numpy()
    crop_boxes = np.asarray(crop_boxes, dtype=np.int32)

    face_crops = np.zeros((T, crop_h, crop_w, 3), dtype=np.uint8)
    for t in range(T):
        if t < len(crop_boxes):
            x1, y1, x2, y2 = crop_boxes[t]
        else:
            x1, y1, x2, y2 = 0, 0, frames[t].shape[1], frames[t].shape[0]

        crop = frames[t][y1:y2, x1:x2, :]
        if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
            crop = frames[t]

        resized = cv2.resize(crop, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
        face_crops[t] = resized[:, :, ::-1]

    face_tensor = torch.from_numpy(face_crops).permute(0, 3, 1, 2).float() / 255.0
    norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (face_tensor - norm_mean) / norm_std


def augment_multimodal_clip(
        face_frames: Optional[torch.Tensor],
        landmarks: torch.Tensor,
        is_train: bool = True,
        flip_prob: float = 0.5,
        scale_jitter: float = 0.08,
        shift_jitter: float = 0.05,
        roll_deg: float = 12.0,
        temporal_dropout: float = 0.05,
        landmark_noise_std: float = 0.002,
        color_jitter: float = 0.2,
        spatial_dropout: float = 0.15,
) -> Tuple[Optional[torch.Tensor], torch.Tensor]:
    """Tăng cường dữ liệu không gian - thời gian đồng bộ cho cả chuỗi điểm mốc và ảnh mặt."""
    if not is_train:
        return face_frames, landmarks

    lms = landmarks.clone()
    frames = face_frames.clone() if face_frames is not None else None
    valid_pts = (lms < 0.999).all(dim=-1, keepdim=True)

    if random.random() < flip_prob:
        lms[..., 0] = torch.where(valid_pts.squeeze(-1), 1.0 - lms[..., 0], lms[..., 0])
        flip_idx = torch.as_tensor(FLIP_MAP_68, dtype=torch.long, device=lms.device)
        lms = lms[:, flip_idx, :]
        if frames is not None:
            frames = torch.flip(frames, dims=[-1])

    if roll_deg > 0 and random.random() < 0.5:
        angle_deg = random.uniform(-roll_deg, roll_deg)
        theta = math.radians(angle_deg)
        cos_t, sin_t = math.cos(theta), math.sin(theta)
        valid_coords = lms[valid_pts.squeeze(-1)]
        if valid_coords.numel() > 0:
            center_x = valid_coords[:, 0].mean().item()
            center_y = valid_coords[:, 1].mean().item()
            dx = lms[..., 0] - center_x
            dy = lms[..., 1] - center_y
            rot_x = torch.clamp(dx * cos_t - dy * sin_t + center_x, 0.0, 1.0)
            rot_y = torch.clamp(dx * sin_t + dy * cos_t + center_y, 0.0, 1.0)
            lms[..., 0] = torch.where(valid_pts.squeeze(-1), rot_x, lms[..., 0])
            lms[..., 1] = torch.where(valid_pts.squeeze(-1), rot_y, lms[..., 1])

    if scale_jitter > 0 or shift_jitter > 0:
        base_scale = 1.0 + random.uniform(-scale_jitter, scale_jitter)
        aspect_ratio = 1.0 + random.uniform(-0.015, 0.015)
        scale_x, scale_y = base_scale, base_scale * aspect_ratio
        shift_x = random.uniform(-shift_jitter, shift_jitter)
        shift_y = random.uniform(-shift_jitter, shift_jitter)
        valid_coords = lms[valid_pts.squeeze(-1)]
        if valid_coords.numel() > 0:
            center_x = valid_coords[:, 0].mean().item()
            center_y = valid_coords[:, 1].mean().item()
            new_x = torch.clamp((lms[..., 0] - center_x) * scale_x + center_x + shift_x, 0.0, 1.0)
            new_y = torch.clamp((lms[..., 1] - center_y) * scale_y + center_y + shift_y, 0.0, 1.0)
            lms[..., 0] = torch.where(valid_pts.squeeze(-1), new_x, lms[..., 0])
            lms[..., 1] = torch.where(valid_pts.squeeze(-1), new_y, lms[..., 1])

    if landmark_noise_std > 0:
        raw_noise = torch.randn_like(lms) * landmark_noise_std
        if lms.shape[0] >= 3:
            smooth_noise = 0.5 * raw_noise
            smooth_noise[1:-1] += 0.25 * (raw_noise[:-2] + raw_noise[2:])
            smooth_noise[0] += 0.25 * raw_noise[0]
            smooth_noise[-1] += 0.25 * raw_noise[-1]
            noise = smooth_noise
        else:
            noise = raw_noise

        noise[:, 36, :] = 0.0
        noise[:, 45, :] = 0.0
        noisy_lms = torch.clamp(lms + noise, 0.0, 1.0)
        lms = torch.where(valid_pts, noisy_lms, lms)

    if temporal_dropout > 0 and random.random() < 0.3:
        T_len = lms.shape[0]
        num_drop = min(max(1, int(round(T_len * temporal_dropout))), 4)
        if T_len > num_drop:
            start_drop = random.randint(0, T_len - num_drop)
            lms[start_drop:start_drop + num_drop] = 1.0
            if frames is not None and frames.shape[0] > 0:
                T_v = frames.shape[0]
                f_start = min(T_v - 1, int(round(start_drop * T_v / float(T_len))))
                f_num = max(1, int(round(num_drop * T_v / float(T_len))))
                frames[f_start:min(T_v, f_start + f_num)] = 0.0

    if frames is not None and frames.numel() > 0:
        if color_jitter > 0 and random.random() < 0.5:
            brightness_factor = 1.0 + random.uniform(-color_jitter, color_jitter)
            contrast_factor = 1.0 + random.uniform(-color_jitter, color_jitter)
            mean_val = frames.mean(dim=(-2, -1), keepdim=True)
            frames = (frames - mean_val) * contrast_factor + mean_val
            frames = frames * brightness_factor

        if spatial_dropout > 0 and random.random() < spatial_dropout:
            _, _, H, W = frames.shape
            cut_h = int(H * random.uniform(0.1, 0.25))
            cut_w = int(W * random.uniform(0.1, 0.25))
            y1 = random.randint(0, max(0, H - cut_h))
            x1 = random.randint(0, max(0, W - cut_w))
            frames[:, :, y1:y1 + cut_h, x1:x1 + cut_w] = 0.0

    return frames, lms


# =====================================================================
# 2. HỆ THỐNG, LOGGING, ATOMIC SAVE & MODEL EMA
# =====================================================================
def seed_everything(seed: int = 42) -> None:
    """Thiết lập hạt giống ngẫu nhiên để tái lập kết quả thực nghiệm."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def configure_logging(
        output_dir: Union[str, Path],
        logger_name: str = "DriverGuardian.LiteFat_v1",
) -> logging.Logger:
    """Cấu hình logging đồng thời ra Console và File log `train.log`."""
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    log_file = out_path / "train.log"

    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for h in list(logger.handlers):
        try:
            h.close()
        except Exception:
            pass
        logger.removeHandler(h)

    formatter = logging.Formatter(
        fmt="[%(asctime)s] [%(levelname)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console_h = logging.StreamHandler(sys.stdout)
    console_h.setLevel(logging.INFO)
    console_h.setFormatter(formatter)
    logger.addHandler(console_h)

    file_h = logging.FileHandler(str(log_file), mode="a", encoding="utf-8")
    file_h.setLevel(logging.INFO)
    file_h.setFormatter(formatter)
    logger.addHandler(file_h)

    return logger


def atomic_save(obj: Any, path: Union[str, Path]) -> None:
    """Lưu tệp an toàn (Atomic Save) chống hỏng tệp khi bị ngắt giữa chừng."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp_file = tempfile.NamedTemporaryFile(delete=False, dir=str(p.parent), suffix=".tmp")
    tmp_path = Path(tmp_file.name)
    try:
        tmp_file.close()
        torch.save(obj, tmp_path)
        tmp_path.replace(p)
    except Exception as e:
        if tmp_path.exists():
            tmp_path.unlink()
        raise e


class ModelEMA:
    """Theo dõi trọng số mô hình bằng Exponential Moving Average (EMA) có warmup và đồng bộ buffer."""

    def __init__(self, model: nn.Module, decay: float = 0.995):
        self.decay = float(decay)
        self.num_updates = 0
        self.param_keys: Set[str] = {k for k, _ in model.named_parameters()}
        self.shadow: Dict[str, torch.Tensor] = {
            k: v.clone().detach() for k, v in model.state_dict().items()
        }

    def _effective_decay(self) -> float:
        """Warmup decay để tránh giữ trọng số khởi tạo ngẫu nhiên quá lâu khi số step/epoch ít."""
        return min(self.decay, float(1 + self.num_updates) / float(10 + self.num_updates))

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        self.num_updates += 1
        eff_decay = self._effective_decay()
        for k, v in model.state_dict().items():
            if k not in self.shadow:
                continue
            target = v.detach().to(self.shadow[k].device)
            if v.dtype.is_floating_point:
                # Làm mượt EMA cho cả tham số và floating-point buffers (BatchNorm running_mean/running_var)
                self.shadow[k].lerp_(target, 1.0 - eff_decay)
            else:
                # Các buffer số nguyên (num_batches_tracked, ...) sao chép trực tiếp
                self.shadow[k].copy_(target)

    def state_dict(self) -> Dict[str, Any]:
        return {"shadow": self.shadow, "num_updates": self.num_updates}

    def load_state_dict(self, state_dict: Dict[str, Any]) -> None:
        if isinstance(state_dict, dict) and "shadow" in state_dict and isinstance(state_dict["shadow"], dict):
            src = state_dict["shadow"]
            self.num_updates = int(state_dict.get("num_updates", 0))
        else:
            src = state_dict
        for k, v in src.items():
            if k in self.shadow and isinstance(v, torch.Tensor) and self.shadow[k].shape == v.shape:
                self.shadow[k].copy_(v.to(self.shadow[k].device))

    def apply_shadow(self, model: nn.Module) -> Dict[str, torch.Tensor]:
        """Sao lưu trọng số hiện tại và nạp trọng số EMA shadow vào mô hình."""
        backup = {k: v.clone().detach() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow, strict=False)
        return backup

    def restore(self, model: nn.Module, backup: Dict[str, torch.Tensor]) -> None:
        """Khôi phục lại trọng số ban đầu từ backup."""
        model.load_state_dict(backup, strict=False)


# =====================================================================
# 3. CÁC HÀM PHỤ TRỢ TÍNH TOÁN CHỈ SỐ ĐÁNH GIÁ KSS 1-9 & NHỊ PHÂN
# =====================================================================
def _average_ranks(arr: np.ndarray) -> np.ndarray:
    """Tính thứ hạng trung bình có xử lý giá trị trùng (ties)."""
    x = np.asarray(arr, dtype=np.float64).ravel()
    n = len(x)
    if n == 0:
        return np.empty(0, dtype=np.float64)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(n, dtype=np.float64)
    ranks[order] = np.arange(1, n + 1, dtype=np.float64)
    unique_vals, inv, counts = np.unique(x, return_inverse=True, return_counts=True)
    if len(unique_vals) < n:
        sum_ranks = np.bincount(inv, weights=ranks)
        ranks = (sum_ranks / counts)[inv]
    return ranks


def binary_auc(labels: np.ndarray, probs: np.ndarray) -> float:
    """Tính ROC-AUC nhị phân thuần NumPy tương đương kiểm định Mann-Whitney U."""
    y = np.asarray(labels, dtype=int).ravel()
    p = np.asarray(probs, dtype=np.float64).ravel()
    n_pos = int(np.sum(y == 1))
    n_neg = int(np.sum(y == 0))
    if n_pos == 0 or n_neg == 0:
        return 0.5

    ranks = _average_ranks(p)
    sum_pos_ranks = float(ranks[y == 1].sum())
    u_stat = sum_pos_ranks - (n_pos * (n_pos + 1)) / 2.0
    return float(u_stat / float(n_pos * n_neg))


def pearson_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Tính hệ số tương quan tuyến tính Pearson."""
    a = np.asarray(y_true, dtype=np.float64).ravel()
    b = np.asarray(y_pred, dtype=np.float64).ravel()
    if len(a) < 2:
        return 0.0
    a_c = a - np.mean(a)
    b_c = b - np.mean(b)
    denom = math.sqrt(float(np.sum(a_c ** 2) * np.sum(b_c ** 2)))
    if denom <= 1e-12:
        return 0.0
    return float(np.sum(a_c * b_c) / denom)


def spearman_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Tính hệ số tương quan thứ hạng Spearman."""
    a = np.asarray(y_true, dtype=np.float64).ravel()
    b = np.asarray(y_pred, dtype=np.float64).ravel()
    if len(a) < 2:
        return 0.0
    return pearson_correlation(_average_ranks(a), _average_ranks(b))


def quadratic_weighted_kappa(
        y_true: np.ndarray,
        y_pred: np.ndarray,
        min_rating: int = 1,
        max_rating: int = 9,
) -> float:
    """
    Tính Quadratic Weighted Kappa (QWK) cho thang điểm thứ tự KSS [min_rating..max_rating].
    """
    yt = np.clip(np.rint(np.asarray(y_true, dtype=np.float64)).astype(int), min_rating, max_rating) - min_rating
    yp = np.clip(np.rint(np.asarray(y_pred, dtype=np.float64)).astype(int), min_rating, max_rating) - min_rating
    n_samples = len(yt)
    n_ratings = max(2, max_rating - min_rating + 1)
    if n_samples == 0:
        return 0.0

    O = np.zeros((n_ratings, n_ratings), dtype=np.float64)
    np.add.at(O, (yt, yp), 1.0)

    idx = np.arange(n_ratings, dtype=np.float64)
    W = ((idx[:, None] - idx[None, :]) ** 2) / float((n_ratings - 1) ** 2)

    hist_t = O.sum(axis=1)
    hist_p = O.sum(axis=0)
    E = np.outer(hist_t, hist_p) / float(n_samples)

    denom = float(np.sum(W * E))
    if denom <= 1e-12:
        return 1.0 if np.array_equal(yt, yp) else 0.0
    return float(1.0 - (np.sum(W * O) / denom))


def compute_binary_metrics(
        labels: np.ndarray,
        preds: np.ndarray,
        probs: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """Tính các chỉ số phân loại nhị phân (Accuracy, Precision, Recall, Specificity, F1, F2, AUC)."""
    y = np.asarray(labels, dtype=int).ravel()
    p = np.asarray(preds, dtype=int).ravel()

    tp = int(np.sum((y == 1) & (p == 1)))
    fp = int(np.sum((y == 0) & (p == 1)))
    fn = int(np.sum((y == 1) & (p == 0)))
    tn = int(np.sum((y == 0) & (p == 0)))
    total = len(y)

    acc = (tp + tn) / max(1, total)
    prec = tp / max(1, tp + fp)
    rec = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    f1 = (2.0 * prec * rec) / max(1e-8, prec + rec) if (prec + rec) > 0 else 0.0
    f2 = (5.0 * prec * rec) / max(1e-8, 4.0 * prec + rec) if (4.0 * prec + rec) > 0 else 0.0

    metrics: Dict[str, float] = {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "specificity": float(spec),
        "f1": float(f1),
        "f2": float(f2),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
        "auc": 0.5,
    }

    if probs is not None:
        try:
            probs_arr = np.asarray(probs, dtype=np.float64).ravel()
            if len(np.unique(y)) > 1:
                metrics["auc"] = binary_auc(y, probs_arr)
        except Exception:
            metrics["auc"] = 0.5

    return metrics


def _softmax_np(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64)
    x_shift = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(x_shift)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True).clip(min=1e-12)


def compute_kss_metrics(
        targets: np.ndarray,
        pred_kss: np.ndarray,
        pred_logits: Optional[np.ndarray] = None,
        pred_probs: Optional[np.ndarray] = None,
        drowsy_threshold: float = 7.0,
        kss_min: float = 1.0,
        kss_max: float = 9.0,
) -> Dict[str, float]:
    """
    Tính bộ chỉ số toàn diện cho bài toán dự đoán KSS (thang 1..9):
      - Hồi quy liên tục: mae, rmse, within_1_acc, within_05_acc, exact_acc, pearson_r, spearman_r.
      - Thứ tự & Phân nhóm: qwk (Quadratic Weighted Kappa), regime_acc, regime_macro_f1 (3 vùng 1..4 / 5..6 / 7..9).
      - Phát hiện buồn ngủ nhị phân (KSS >= drowsy_threshold): accuracy, precision, recall, specificity, f1, f2, auc.
      - Điểm tổng hợp val_score = 0.40*(1 - mae/4) + 0.30*within_1_acc + 0.30*max(0, qwk).
    """
    y_true = np.asarray(targets, dtype=np.float64).ravel()
    y_pred = np.asarray(pred_kss, dtype=np.float64).ravel()
    n = len(y_true)
    if n == 0:
        return {
            "mae": 0.0,
            "rmse": 0.0,
            "within_1_acc": 0.0,
            "within_05_acc": 0.0,
            "exact_acc": 0.0,
            "qwk": 0.0,
            "pearson_r": 0.0,
            "spearman_r": 0.0,
            "regime_acc": 0.0,
            "regime_macro_f1": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "specificity": 0.0,
            "f1": 0.0,
            "f2": 0.0,
            "auc": 0.5,
            "val_score": 0.0,
            "pred_mean": 0.0,
            "pred_std": 0.0,
            "pred_min": 0.0,
            "pred_max": 0.0,
            "num_samples": 0,
        }

    abs_err = np.abs(y_pred - y_true)
    mae = float(np.mean(abs_err))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    within_1_acc = float(np.mean(abs_err <= 1.0))
    within_05_acc = float(np.mean(abs_err <= 0.5))

    min_int = int(round(kss_min))
    max_int = int(round(kss_max))
    y_true_int = np.clip(np.rint(y_true), min_int, max_int).astype(int)
    y_pred_int = np.clip(np.rint(y_pred), min_int, max_int).astype(int)
    exact_acc = float(np.mean(y_pred_int == y_true_int))

    qwk = quadratic_weighted_kappa(y_true_int, y_pred_int, min_rating=min_int, max_rating=max_int)
    pearson_r = pearson_correlation(y_true, y_pred)
    spearman_r = spearman_correlation(y_true, y_pred)

    # 3 vùng KSS: 0=Alert (<=4), 1=Mid (5..6), 2=Drowsy (>=7)
    reg_true = np.where(y_true >= 6.5, 2, np.where(y_true >= 4.5, 1, 0))
    reg_pred = np.where(y_pred >= 6.5, 2, np.where(y_pred >= 4.5, 1, 0))
    regime_acc = float(np.mean(reg_true == reg_pred))
    reg_f1s: List[float] = []
    for r_id in (0, 1, 2):
        if np.any(reg_true == r_id) or np.any(reg_pred == r_id):
            tp_r = float(np.sum((reg_true == r_id) & (reg_pred == r_id)))
            fp_r = float(np.sum((reg_true != r_id) & (reg_pred == r_id)))
            fn_r = float(np.sum((reg_true == r_id) & (reg_pred != r_id)))
            denom_r = 2.0 * tp_r + fp_r + fn_r
            reg_f1s.append((2.0 * tp_r / denom_r) if denom_r > 0 else 0.0)
    regime_macro_f1 = float(np.mean(reg_f1s)) if reg_f1s else 0.0

    # Chỉ số nhị phân (KSS >= drowsy_threshold)
    y_bin_true = (y_true >= drowsy_threshold).astype(int)
    y_bin_pred = (y_pred >= (drowsy_threshold - 0.5)).astype(int)

    probs_arr = None
    if pred_probs is not None:
        probs_mat = np.asarray(pred_probs, dtype=np.float64)
        if probs_mat.ndim == 2 and probs_mat.shape[1] >= 2:
            drowsy_start_idx = max(0, min(probs_mat.shape[1] - 1, int(round(drowsy_threshold - kss_min))))
            probs_arr = probs_mat[:, drowsy_start_idx:].sum(axis=-1)
    elif pred_logits is not None:
        logits_mat = np.asarray(pred_logits, dtype=np.float64)
        if logits_mat.ndim == 2 and logits_mat.shape[1] >= 2:
            probs_mat = _softmax_np(logits_mat)
            drowsy_start_idx = max(0, min(probs_mat.shape[1] - 1, int(round(drowsy_threshold - kss_min))))
            probs_arr = probs_mat[:, drowsy_start_idx:].sum(axis=-1)

    if probs_arr is None:
        probs_arr = np.clip((y_pred - kss_min) / max(1e-6, kss_max - kss_min), 0.0, 1.0)

    bin_m = compute_binary_metrics(y_bin_true, y_bin_pred, probs_arr)

    val_score = (
        0.40 * max(0.0, 1.0 - mae / 4.0)
        + 0.30 * within_1_acc
        + 0.30 * max(0.0, qwk)
    )

    metrics: Dict[str, float] = {
        "mae": mae,
        "rmse": rmse,
        "within_1_acc": within_1_acc,
        "within_05_acc": within_05_acc,
        "exact_acc": exact_acc,
        "qwk": qwk,
        "pearson_r": pearson_r,
        "spearman_r": spearman_r,
        "regime_acc": regime_acc,
        "regime_macro_f1": regime_macro_f1,
        "val_score": float(val_score),
        "pred_mean": float(np.mean(y_pred)),
        "pred_std": float(np.std(y_pred)),
        "pred_min": float(np.min(y_pred)),
        "pred_max": float(np.max(y_pred)),
        "num_samples": int(n),
    }
    metrics.update(bin_m)
    return metrics


def compute_metrics(
        labels: np.ndarray,
        preds: np.ndarray,
        probs: Optional[np.ndarray] = None,
        pred_logits: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Hàm bọc tương thích ngược:
      - Nếu mảng labels/preds nằm trong thang KSS (giá trị > 1 hoặc float), gọi compute_kss_metrics.
      - Nếu là nhãn nhị phân {0, 1}, gọi compute_binary_metrics.
    """
    y_arr = np.asarray(labels).ravel()
    p_arr = np.asarray(preds).ravel()
    is_kss = (
        pred_logits is not None
        or np.any(y_arr > 1)
        or np.any(p_arr > 1)
        or np.issubdtype(p_arr.dtype, np.floating)
    )
    if is_kss:
        return compute_kss_metrics(y_arr, p_arr, pred_logits=pred_logits, pred_probs=probs)
    return compute_binary_metrics(y_arr, p_arr, probs=probs)


# =====================================================================
# 4. CÁC HÀM PHỤ TRỢ NẠP CHECKPOINT & WARM-START
# =====================================================================
def safe_load_optimizer(
        optimizer: torch.optim.Optimizer,
        opt_state_dict: Dict[str, Any],
        logger: Optional[logging.Logger] = None,
) -> bool:
    """
    Nạp an toàn optimizer state dict, xử lý tương thích khi số lượng param_groups
    hoặc cấu trúc nhóm tham số giữa checkpoint và mô hình hiện tại khác nhau.
    """
    saved_groups = len(opt_state_dict.get("param_groups", []))
    curr_groups = len(optimizer.param_groups)

    try:
        if saved_groups == curr_groups:
            optimizer.load_state_dict(opt_state_dict)
            return True
        if saved_groups == 1 and curr_groups == 2:
            if logger is not None:
                logger.warning(
                    "Checkpoint optimizer state chỉ có 1 param_group. "
                    "Tự động ánh xạ nhóm 0 và giữ nguyên khởi tạo mới cho nhóm 1."
                )
            adapted_dict = {
                "state": opt_state_dict.get("state", {}),
                "param_groups": [
                    opt_state_dict["param_groups"][0],
                    optimizer.state_dict()["param_groups"][1],
                ],
            }
            optimizer.load_state_dict(adapted_dict)
            return True
        if logger is not None:
            logger.warning(
                f"Không tương thích số lượng param_groups (checkpoint: {saved_groups}, hiện tại: {curr_groups}). "
                "Bỏ qua khôi phục optimizer state."
            )
        return False
    except Exception as e:
        if logger is not None:
            logger.warning(f"Không thể khôi phục optimizer_state_dict ({e}). Sử dụng optimizer mới.")
        return False


def _resolve_checkpoint_path(
        checkpoint_path: Union[str, Path],
        save_dir: Optional[Union[str, Path]] = None,
) -> Path:
    ckpt_path = Path(checkpoint_path)
    if ckpt_path.is_file():
        return ckpt_path

    if save_dir is not None:
        save_p = Path(save_dir)
        alt_path = save_p / checkpoint_path
        if alt_path.is_file():
            return alt_path
        available_names = [c.name for c in sorted(save_p.glob("*.pt"))] if save_p.is_dir() else []
        raise FileNotFoundError(
            f"Không tìm thấy tệp checkpoint tại '{checkpoint_path}' hoặc '{alt_path}'. "
            f"Các checkpoint hiện có trong {save_p}: {available_names}"
        )

    raise FileNotFoundError(f"Không tìm thấy tệp checkpoint tại: '{checkpoint_path}'")


def load_checkpoint(
        checkpoint_path: Union[str, Path],
        model: nn.Module,
        optimizer: Optional[torch.optim.Optimizer] = None,
        scheduler: Optional[Any] = None,
        scaler: Optional[Any] = None,
        ema: Optional[ModelEMA] = None,
        save_dir: Optional[Union[str, Path]] = None,
        cfg: Optional[Any] = None,
        device: Optional[Union[str, torch.device]] = None,
        logger: Optional[logging.Logger] = None,
        strict: bool = True,
) -> Dict[str, Any]:
    """
    Nạp toàn bộ trạng thái checkpoint (model, EMA, optimizer, scheduler, GradScaler,
    best metrics & trạng thái đóng băng backbone) để tiếp tục huấn luyện hoặc đánh giá.
    """
    ckpt_path = _resolve_checkpoint_path(checkpoint_path, save_dir=save_dir)
    map_loc = device if device is not None else next(model.parameters()).device

    if logger is not None:
        logger.info(f"Đang nạp checkpoint từ: {ckpt_path}")

    checkpoint = torch.load(ckpt_path, map_location=map_loc, weights_only=False)

    # 1. Nạp trọng số mô hình
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        try:
            model.load_state_dict(checkpoint["model_state_dict"], strict=strict)
        except RuntimeError as e:
            if strict:
                incompat = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
                non_aux_missing = [k for k in incompat.missing_keys if not k.startswith("domain_discriminator.")]
                if non_aux_missing or incompat.unexpected_keys:
                    raise e
                if logger is not None:
                    logger.warning(
                        f"Checkpoint cũ không chứa nhánh phụ trợ ({incompat.missing_keys}); "
                        "đã nạp tương thích với strict=False."
                    )
            else:
                raise
    elif isinstance(checkpoint, dict) and all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        model.load_state_dict(checkpoint, strict=strict)
        checkpoint = {"model_state_dict": checkpoint, "epoch": 0}
    else:
        if logger is not None:
            logger.warning("Checkpoint không chứa 'model_state_dict' hợp lệ!")

    # 2. Khôi phục EMA shadow nếu có
    if ema is not None and "ema_state_dict" in checkpoint:
        ema.load_state_dict(checkpoint["ema_state_dict"])
        if "ema_num_updates" in checkpoint:
            ema.num_updates = int(checkpoint["ema_num_updates"])

    # 3. Khôi phục epoch
    ckpt_epoch = int(checkpoint.get("epoch", 0))
    start_epoch = ckpt_epoch + 1

    # 4. Khôi phục Optimizer, Scheduler, GradScaler
    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        safe_load_optimizer(optimizer, checkpoint["optimizer_state_dict"], logger=logger)

    if scheduler is not None and "scheduler_state_dict" in checkpoint:
        saved_cfg_epochs = checkpoint.get("config", {}).get("epochs", None)
        target_epochs = getattr(cfg, "epochs", None) if cfg is not None else None
        if (
            optimizer is not None
            and saved_cfg_epochs is not None
            and target_epochs is not None
            and int(saved_cfg_epochs) != int(target_epochs)
        ):
            if logger is not None:
                logger.warning(
                    f"Phát hiện thay đổi tổng số epochs ({saved_cfg_epochs} -> {target_epochs}). "
                    f"Tái lập lịch trình Cosine mượt từ Epoch {start_epoch} đến Epoch {target_epochs}."
                )
            remaining_epochs = max(1, int(target_epochs) - start_epoch + 1)

            def resume_lr_lambda(ep_offset: int) -> float:
                progress = float(ep_offset) / float(max(1, remaining_epochs))
                return 0.5 * (1.0 + math.cos(math.pi * progress))

            scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=resume_lr_lambda)
        else:
            try:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
            except Exception as e:
                if logger is not None:
                    logger.warning(f"Không thể khôi phục scheduler_state_dict ({e}).")

    if scaler is not None and "scaler_state_dict" in checkpoint and getattr(scaler, "is_enabled", lambda: False)():
        try:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
        except Exception as e:
            if logger is not None:
                logger.warning(f"Không thể khôi phục scaler_state_dict ({e}).")

    # 5. Khôi phục Best Metrics
    best_epoch = int(checkpoint.get("best_epoch", -1))
    best_val_score = float(checkpoint.get("best_val_score", -1.0))
    best_val_mae = float(checkpoint.get("best_val_mae", float("inf")))
    best_val_f1 = float(checkpoint.get("best_val_f1", -1.0))

    if best_epoch < 0 and save_dir is not None:
        best_json_path = Path(save_dir) / "best_metrics.json"
        if best_json_path.is_file():
            try:
                with open(best_json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                if int(data.get("best_epoch", 0)) <= ckpt_epoch:
                    best_epoch = int(data["best_epoch"])
                    m = data.get("metrics", {})
                    best_val_f1 = float(m.get("f1", -1.0))
                    best_val_mae = float(m.get("mae", float("inf")))
                    best_val_score = float(m.get("val_score", -1.0))
            except Exception:
                pass

    if best_val_score < 0 and isinstance(checkpoint.get("val_metrics"), dict):
        vm = checkpoint["val_metrics"]
        best_val_f1 = float(vm.get("f1", -1.0))
        best_val_mae = float(vm.get("mae", float("inf")))
        best_val_score = float(vm.get("val_score", -1.0))
        best_epoch = ckpt_epoch

    # 6. Đồng bộ trạng thái đóng/mở băng backbone MobileNetV3 theo start_epoch
    if cfg is not None and hasattr(model, "set_backbone_trainable"):
        freeze_epochs = int(getattr(cfg, "freeze_backbone_epochs", 0))
        unfreeze_blocks = getattr(cfg, "unfreeze_last_n_blocks", 3)
        if start_epoch <= freeze_epochs:
            model.set_backbone_trainable(False)
        else:
            model.set_backbone_trainable(True, unfreeze_last_n_blocks=unfreeze_blocks)

    if logger is not None:
        logger.info("=" * 70)
        logger.info(f"===> KHÔI PHỤC CHECKPOINT THÀNH CÔNG TỪ: {ckpt_path.name}")
        logger.info(f"===> Đã hoàn thành Epoch {ckpt_epoch} | Tiếp tục từ Epoch {start_epoch}")
        logger.info(
            f"===> Best Val Score: {best_val_score:.4f} | Best MAE: {best_val_mae:.4f} | Best Epoch: {best_epoch}"
        )
        logger.info("=" * 70)

    return {
        "checkpoint_path": str(ckpt_path),
        "ckpt_epoch": ckpt_epoch,
        "start_epoch": start_epoch,
        "best_epoch": best_epoch,
        "best_val_score": best_val_score,
        "best_val_mae": best_val_mae,
        "best_val_f1": best_val_f1,
        "scheduler": scheduler,
        "history": checkpoint.get("history", []),
        "checkpoint": checkpoint,
    }


def load_pretrained_weights(
        pretrained_path: Union[str, Path],
        model: nn.Module,
        use_visual_context: bool = False,
        device: Optional[Union[str, torch.device]] = None,
        logger: Optional[logging.Logger] = None,
) -> Set[str]:
    """
    Nạp trọng số từng phần (Warm-start) từ checkpoint trước đó cho các tầng khớp kích thước.
    Hỗ trợ:
      - Ánh xạ block STGL theo cặp (block_idx, layer_idx).
      - Thích nghi kernel Conv2d cho TCN khi kernel_size thay đổi.
      - Tự động reset FiLM về identity khi chuyển từ Stage 1 (Landmark-Only) sang Stage 2 (Visual).
    Trả về tập tên các tham số đã được nạp từ checkpoint tiền huấn luyện.
    """
    p = Path(pretrained_path)
    if not p.is_file():
        if logger is not None:
            logger.warning(f"Không tìm thấy tệp pretrained_from tại '{pretrained_path}'. Bỏ qua warm-start.")
        return set()

    map_loc = device if device is not None else next(model.parameters()).device
    if logger is not None:
        logger.info(f"Đang nạp trọng số tiền huấn luyện (Warm-start) từ: {p}")

    checkpoint = torch.load(p, map_location=map_loc, weights_only=False)
    saved_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    curr_dict = model.state_dict()

    ckpt_cfg = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
    ckpt_has_visual = bool(ckpt_cfg.get("use_visual_context", False))
    is_landmark_to_visual = bool(use_visual_context) and (not ckpt_has_visual)
    if is_landmark_to_visual and hasattr(model, "fusion") and hasattr(model.fusion, "reset_film_to_identity"):
        if logger is not None:
            logger.info(
                "===> [Warm-Start] Chuyển từ Landmark-Only sang Visual Context: "
                "Giữ ImageNet cho MobileNetV3 và kích hoạt Zero-init Identity FiLM."
            )
        model.fusion.reset_film_to_identity()

    saved_stgl_indices = sorted({int(k.split(".")[1]) for k in saved_dict if k.startswith("stgl_blocks.")})
    curr_stgl_indices = sorted({int(k.split(".")[1]) for k in curr_dict if k.startswith("stgl_blocks.")})

    num_blocks = int(getattr(model, "num_blocks", 3))
    saved_lpb = len(saved_stgl_indices) // max(1, num_blocks) if saved_stgl_indices else 0
    curr_lpb = len(curr_stgl_indices) // max(1, num_blocks) if curr_stgl_indices else 0

    block_mapping: Dict[int, int] = {}
    if saved_lpb > 0 and curr_lpb > 0:
        for b in range(num_blocks):
            for l in range(min(saved_lpb, curr_lpb)):
                block_mapping[b * curr_lpb + l] = b * saved_lpb + l

    matched_keys: List[str] = []
    adapted_keys: List[str] = []
    skipped_keys: List[str] = []
    pretrained_param_names: Set[str] = set()

    with torch.no_grad():
        for name, curr_param in curr_dict.items():
            if is_landmark_to_visual and (name.startswith("context_encoder.") or "film_" in name):
                skipped_keys.append(f"{name} (bảo toàn ImageNet / zero-init FiLM)")
                continue

            if name.startswith("stgl_blocks."):
                parts = name.split(".")
                c_idx = int(parts[1])
                suffix = ".".join(parts[2:])
                saved_name = f"stgl_blocks.{block_mapping[c_idx]}.{suffix}" if c_idx in block_mapping else None
            else:
                saved_name = name

            if saved_name is not None and saved_name in saved_dict:
                saved_param = saved_dict[saved_name]
                if curr_param.shape == saved_param.shape:
                    curr_param.copy_(saved_param.to(curr_param.device))
                    matched_keys.append(name)
                    pretrained_param_names.add(name)
                elif (
                    curr_param.ndim == 4
                    and saved_param.ndim == 4
                    and curr_param.shape[:3] == saved_param.shape[:3]
                ):
                    curr_param.zero_()
                    k_saved = saved_param.shape[-1]
                    k_curr = curr_param.shape[-1]
                    k_copy = min(k_saved, k_curr)
                    curr_param[..., -k_copy:].copy_(saved_param[..., -k_copy:].to(curr_param.device))
                    adapted_keys.append(f"{name} ({k_saved}->{k_curr})")
                    pretrained_param_names.add(name)
                else:
                    skipped_keys.append(f"{name} (shape lệch: {tuple(saved_param.shape)} vs {tuple(curr_param.shape)})")
            else:
                if name.startswith("stgl_blocks.") and "skip_conv" in name:
                    curr_param.zero_()
                elif name.startswith("stgl_blocks.") and "res_scale" in name:
                    curr_param.fill_(1.0)
                skipped_keys.append(f"{name} (mới)")

    if logger is not None:
        logger.info("=" * 70)
        logger.info(f"===> KẾT QUẢ WARM-START TỪ {p.name}:")
        logger.info(f"===> Khớp hoàn toàn: {len(matched_keys)} | TCN adaptation: {len(adapted_keys)} | Mới/Bỏ qua: {len(skipped_keys)}")
        logger.info("=" * 70)

    return pretrained_param_names

In [7]:
# =====================================================================
# File: dataset.py
# Dự án: Driver Guardian - Module LiteFat_v1 (KSS 1-9, 10 FPS)
# Tác vụ: Nạp tập dữ liệu dạng E:\LSTM\data_processed_v1 từ manifest
#         (.csv hoặc .json), trích xuất 68 điểm mốc khuôn mặt bằng
#         PIPNet (FaceBoxesV2 + ResNet18 300W) và cắt vùng khuôn mặt
#         cho mô hình LiteFat_v1 (cửa sổ 60s @ 10 FPS, sliding 30s).
# =====================================================================
from __future__ import annotations

import csv
import hashlib
import json
import logging
import math
import os
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Monkey patch cho scipy >= 1.14 để tương thích với torchlm metrics
import scipy.integrate

if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

# Cấu hình môi trường Windows và nạp DLL PyTorch CUDA
torch_lib_path = os.path.join(os.path.dirname(sys.executable), "Library", "bin")
if os.path.exists(torch_lib_path):
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset

# Import các hàm tiện ích dùng chung từ ultis.py
try:
    from ultis import (
        _parse_size,
        atomic_save,
        augment_multimodal_clip,
        compute_iou,
        crop_faces_from_boxes,
        letterbox,
    )
except ImportError:
    pass

logger = logging.getLogger("DriverGuardian.LiteFat_v1")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)
logger.propagate = False


# =====================================================================
# 0. HẰNG SỐ VÙNG GIẢI PHẪU & TIỆN ÍCH PIPNET 68 ĐIỂM
#    (Các hàm _parse_size, letterbox, compute_iou, crop_faces_from_boxes,
#     augment_multimodal_clip, atomic_save & FLIP_MAP_68 tập trung ở ultis.py)
# =====================================================================
KINEMATIC_REGIONS = [
    (0, 17, False, (255, 190, 0)),   # Đường viền hàm (0 - 16)
    (17, 22, False, (0, 230, 255)),  # Lông mày trái (17 - 21)
    (22, 27, False, (0, 230, 255)),  # Lông mày phải (22 - 26)
    (27, 31, False, (220, 0, 255)),  # Sống mũi (27 - 30)
    (31, 36, False, (220, 0, 255)),  # Chóp mũi dưới (31 - 35)
    (36, 42, True, (0, 255, 100)),   # Mắt trái (36 - 41)
    (42, 48, True, (0, 255, 100)),   # Mắt phải (42 - 47)
    (48, 60, True, (0, 160, 255)),   # Viền môi ngoài (48 - 59)
    (60, 68, True, (0, 70, 255)),    # Viền môi trong (60 - 67)
]


def init_pipnet_68_gpu(backbone: str = "resnet18") -> None:
    """Khởi tạo pipeline FaceBoxesV2 + PIPNet (68 điểm chuẩn 300W) qua torchlm."""
    import torchlm
    from torchlm.models import pipnet
    from torchlm.tools import faceboxesv2

    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Đang khởi tạo PIPNet ({backbone} - 68 điểm 300W) trên thiết bị: {device.upper()}")
    torchlm.runtime.bind(faceboxesv2(device=device))
    torchlm.runtime.bind(
        pipnet(
            backbone=backbone,
            pretrained=True,
            num_nb=10,
            num_lms=68,
            net_stride=32,
            input_size=256,
            meanface_type="300w",
            map_location=device,
        )
    )


def draw_68_landmarks_custom(
        frame: np.ndarray,
        landmarks: np.ndarray,
        bboxes: np.ndarray,
        conf_threshold: float = 0.6,
) -> None:
    """Vẽ Bounding Box và 68 điểm mốc theo từng vùng giải phẫu (chuẩn 300W / iBUG 68)."""
    for i in range(len(bboxes)):
        x1, y1, x2, y2, score = bboxes[i]
        if score < conf_threshold:
            continue

        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(
            frame,
            f"Face: {score:.2f}",
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 0),
            2,
        )

        pts = landmarks[i].astype(np.int32)
        for start, end, is_closed, color in KINEMATIC_REGIONS:
            cv2.polylines(
                frame,
                [pts[start:end]],
                isClosed=is_closed,
                color=color,
                thickness=1,
                lineType=cv2.LINE_AA,
            )

        for (x, y) in pts:
            cv2.circle(frame, (x, y), radius=2, color=(0, 255, 0), thickness=-1, lineType=cv2.LINE_AA)


# =====================================================================
# 1. BỘ TRÍCH XUẤT PIPNET TỐI ƯU HÓA CAO CẤP (FAST PIPNET EXTRACTOR)
# =====================================================================
class FastPIPNetExtractor:
    """
    Bộ trích xuất 68 điểm mốc khuôn mặt và cắt khung hình cho LiteFat_v1:
      1. Batched FaceBoxesV2 trên GPU với cached PriorBox anchors và torchvision GPU NMS.
      2. Temporal Driver Tracking: Chọn khuôn mặt tài xế ổn định qua score, diện tích & IoU.
      3. Batched PIPNet 300W trên GPU với giải mã NRM vector hóa.
      4. Landmark-Guided Box Refinement + nội suy tuyến tính + làm mượt EMA.
    """

    def __init__(
            self,
            backbone: str = "resnet18",
            device: Optional[str] = None,
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
            use_fp16: bool = False,
    ):
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.bbox_margin = float(bbox_margin)
        self.box_smooth_alpha = float(box_smooth_alpha)
        self.use_fp16 = bool(use_fp16) and (self.device.type == "cuda")

        from torchlm.runtime._wrappers import RuntimeWrapper
        if getattr(RuntimeWrapper, "face_base", None) is None or getattr(RuntimeWrapper, "landmarks_base", None) is None:
            init_pipnet_68_gpu(backbone=backbone)

        self.face_base = RuntimeWrapper.face_base
        self.face_net = self.face_base.net.to(self.device)
        self.face_net.eval()

        self.lmk_base = RuntimeWrapper.landmarks_base
        self.lmk_net = self.lmk_base.to(self.device)
        self.lmk_net.eval()

        target_w, target_h = self.letterbox_size
        from torchlm.tools._faceboxesv2 import PriorBox
        pbox = PriorBox(self.face_base.cfg, image_size=(target_h, target_w))
        self.cached_priors = pbox.forward().to(self.device)
        self.cached_variances = self.face_base.cfg["variance"]
        self.cached_scale = torch.tensor(
            [target_w, target_h, target_w, target_h], dtype=torch.float32, device=self.device
        )
        self.faceboxes_mean = torch.tensor(
            [104.0, 117.0, 123.0], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        self.pipnet_mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)
        self.pipnet_std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        rev1 = self.lmk_base.reverse_index1
        rev2 = self.lmk_base.reverse_index2
        num_nb = self.lmk_base.num_nb
        self.rev_indices = torch.tensor(
            [r1 * num_nb + r2 for r1, r2 in zip(rev1, rev2)],
            dtype=torch.long,
            device=self.device,
        )

    @torch.no_grad()
    def _detect_faces_batched(self, frames: List[np.ndarray], max_chunk: int = 32) -> List[np.ndarray]:
        """Chạy phát hiện khuôn mặt theo lô T khung hình trên GPU."""
        T = len(frames)
        if T == 0:
            return []

        all_dets: List[np.ndarray] = []
        priors_xy = self.cached_priors[None, :, :2]
        priors_wh = self.cached_priors[None, :, 2:]
        v0, v1 = self.cached_variances[0], self.cached_variances[1]

        for s in range(0, T, max_chunk):
            chunk = frames[s:s + max_chunk]
            cur_T = len(chunk)

            frames_np = np.stack(chunk, axis=0)
            tensor = torch.from_numpy(frames_np).permute(0, 3, 1, 2).to(self.device).float()
            tensor = tensor - self.faceboxes_mean

            loc, conf = self.face_net(tensor)
            scores = conf[:, :, 1]

            boxes_xy = priors_xy + loc[:, :, :2] * v0 * priors_wh
            boxes_wh = priors_wh * torch.exp(loc[:, :, 2:] * v1)
            b_x1y1 = boxes_xy - boxes_wh / 2.0
            b_x2y2 = b_x1y1 + boxes_wh
            decoded_boxes = torch.cat([b_x1y1, b_x2y2], dim=-1) * self.cached_scale

            for b in range(cur_T):
                sc = scores[b]
                mask = sc >= self.conf_threshold
                if not mask.any():
                    all_dets.append(np.empty((0, 5), dtype=np.float32))
                    continue

                bx = decoded_boxes[b, mask]
                sc_valid = sc[mask]
                keep = torchvision.ops.nms(bx, sc_valid, iou_threshold=self.nms_threshold)[:50]
                res = torch.cat([bx[keep], sc_valid[keep, None]], dim=-1).cpu().numpy()
                all_dets.append(res)

        return all_dets

    def _select_driver_faces(
            self,
            all_dets: List[np.ndarray],
    ) -> Tuple[List[Optional[np.ndarray]], List[bool]]:
        """Theo dõi và chọn duy nhất 1 khuôn mặt tài xế ổn định qua các khung hình."""
        T = len(all_dets)
        chosen_bboxes: List[Optional[np.ndarray]] = [None] * T
        found: List[bool] = [False] * T

        prev_box: Optional[np.ndarray] = None
        prev_box_area = 1.0

        for t in range(T):
            dets = all_dets[t]
            if len(dets) == 0:
                continue

            best_idx = 0
            if prev_box is None:
                best_score = -1.0
                for i, d in enumerate(dets):
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = d[4] * (area ** 0.5)
                    if metric > best_score:
                        best_score = metric
                        best_idx = i
            else:
                best_match = -1.0
                for i, d in enumerate(dets):
                    iou = compute_iou(d[:4], prev_box[:4])
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = (iou * 2.0) + (d[4] * 0.5) + (min(1.0, area / (prev_box_area + 1e-6)) * 0.3)
                    if metric > best_match:
                        best_match = metric
                        best_idx = i

            chosen = dets[best_idx].copy()
            chosen_bboxes[t] = chosen
            found[t] = True
            prev_box = chosen
            prev_box_area = max(1.0, (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1]))

        return chosen_bboxes, found

    @torch.no_grad()
    def _extract_landmarks_batched(
            self,
            frames: List[np.ndarray],
            chosen_bboxes: List[Optional[np.ndarray]],
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Trích xuất 68 điểm mốc khuôn mặt chuẩn 300W bằng PIPNet theo lô trên GPU."""
        T = len(frames)
        target_w, target_h = self.letterbox_size
        extend = 0.2

        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]
        landmarks_norm = np.ones((T, 68, 2), dtype=np.float32)
        landmarks_px = np.zeros((T, 68, 2), dtype=np.float32)

        if not valid_indices:
            return landmarks_norm, landmarks_px

        crop_coords = []
        crops_resized = []

        for idx in valid_indices:
            bx1, by1, bx2, by2 = chosen_bboxes[idx][:4]
            bw = bx2 - bx1 + 1.0
            bh = by2 - by1 + 1.0

            cx1 = int(bx1 - bw * extend / 2.0)
            cy1 = int(by1 + bh * extend / 2.0)
            cx2 = int(bx2 + bw * extend / 2.0)
            cy2 = int(by2 + bh * extend / 2.0)

            cx1 = max(0, min(target_w - 2, cx1))
            cy1 = max(0, min(target_h - 2, cy1))
            cx2 = max(cx1 + 1, min(target_w, cx2))
            cy2 = max(cy1 + 1, min(target_h, cy2))

            crop_coords.append((cx1, cy1, float(cx2 - cx1), float(cy2 - cy1)))
            face_crop = frames[idx][cy1:cy2, cx1:cx2, ::-1]
            crops_resized.append(cv2.resize(face_crop, (256, 256), interpolation=cv2.INTER_LINEAR))

        max_crop_chunk = 64
        batched_lms_list = []
        norm_factor = 1.0 * self.lmk_base.input_size / self.lmk_base.net_stride
        num_nb = self.lmk_base.num_nb
        max_len = self.lmk_base.max_len

        for s in range(0, len(crops_resized), max_crop_chunk):
            crops_np = np.stack(crops_resized[s:s + max_crop_chunk], axis=0)
            crops_tensor = torch.from_numpy(crops_np).permute(0, 3, 1, 2).to(self.device).float() / 255.0
            crops_tensor = (crops_tensor - self.pipnet_mean) / self.pipnet_std

            out_cls, out_x, out_y, out_nb_x, out_nb_y = self.lmk_net(crops_tensor)
            cur_B, num_lms, _, W_map = out_cls.shape

            out_cls = out_cls.view(cur_B, num_lms, -1)
            max_ids = torch.argmax(out_cls, dim=2)
            max_ids_col = max_ids.unsqueeze(-1)

            x_sel = torch.gather(out_x.view(cur_B, num_lms, -1), 2, max_ids_col).squeeze(2)
            y_sel = torch.gather(out_y.view(cur_B, num_lms, -1), 2, max_ids_col).squeeze(2)

            max_ids_nb = max_ids_col.repeat(1, 1, num_nb).view(cur_B, num_lms * num_nb, 1)
            nb_x_sel = torch.gather(out_nb_x.view(cur_B, num_lms * num_nb, -1), 2, max_ids_nb).view(cur_B, num_lms, num_nb)
            nb_y_sel = torch.gather(out_nb_y.view(cur_B, num_lms * num_nb, -1), 2, max_ids_nb).view(cur_B, num_lms, num_nb)

            pred_x = ((max_ids % W_map).float() + x_sel) / norm_factor
            pred_y = (torch.floor(max_ids.float() / W_map) + y_sel) / norm_factor

            pred_nb_x = ((max_ids % W_map).float().unsqueeze(-1) + nb_x_sel) / norm_factor
            pred_nb_y = (torch.floor(max_ids.float() / W_map).unsqueeze(-1) + nb_y_sel) / norm_factor

            nb_x_rev = pred_nb_x.view(cur_B, -1)[:, self.rev_indices].view(cur_B, num_lms, max_len)
            nb_y_rev = pred_nb_y.view(cur_B, -1)[:, self.rev_indices].view(cur_B, num_lms, max_len)

            final_x = torch.mean(torch.cat([pred_x.unsqueeze(-1), nb_x_rev], dim=-1), dim=-1)
            final_y = torch.mean(torch.cat([pred_y.unsqueeze(-1), nb_y_rev], dim=-1), dim=-1)
            batched_lms_list.append(torch.stack([final_x, final_y], dim=-1).cpu().numpy())

        batched_lms = (
            np.concatenate(batched_lms_list, axis=0)
            if batched_lms_list
            else np.empty((0, 68, 2), dtype=np.float32)
        )

        for i, idx in enumerate(valid_indices):
            cx1, cy1, cw, ch = crop_coords[i]
            lm_crop = batched_lms[i]
            abs_x = lm_crop[:, 0] * cw + cx1
            abs_y = lm_crop[:, 1] * ch + cy1
            landmarks_px[idx, :, 0] = abs_x
            landmarks_px[idx, :, 1] = abs_y
            landmarks_norm[idx, :, 0] = abs_x / float(target_w)
            landmarks_norm[idx, :, 1] = abs_y / float(target_h)

        return landmarks_norm, landmarks_px

    def compute_face_crop_boxes(
            self,
            chosen_bboxes: List[Optional[np.ndarray]],
            landmarks_px: np.ndarray,
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Tính toán hộp cắt khuôn mặt vuông kèm nội suy mất mặt và làm mượt EMA."""
        T = len(chosen_bboxes)
        target_w, target_h = self.letterbox_size
        raw_boxes = np.zeros((T, 4), dtype=np.float32)

        for t in range(T):
            if found[t] and chosen_bboxes[t] is not None:
                bx1, by1, bx2, by2 = chosen_bboxes[t][:4]
                lms = landmarks_px[t]
                fx1 = min(bx1, float(np.min(lms[:, 0])))
                fy1 = min(by1, float(np.min(lms[:, 1])))
                fx2 = max(bx2, float(np.max(lms[:, 0])))
                fy2 = max(by2, float(np.max(lms[:, 1])))

                fw, fh = fx2 - fx1, fy2 - fy1
                cx1, cy1 = fx1 - fw * self.bbox_margin, fy1 - fh * self.bbox_margin
                cx2, cy2 = fx2 + fw * self.bbox_margin, fy2 + fh * self.bbox_margin

                center_x, center_y = (cx1 + cx2) / 2.0, (cy1 + cy2) / 2.0
                side = max(cx2 - cx1, cy2 - cy1)
                raw_boxes[t] = [
                    center_x - side / 2.0,
                    center_y - side / 2.0,
                    center_x + side / 2.0,
                    center_y + side / 2.0,
                ]

        valid_indices = [t for t in range(T) if found[t]]
        if not valid_indices:
            side = min(target_w, target_h) * 0.8
            cx, cy = target_w / 2.0, target_h / 2.0
            default_box = np.array([cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0], dtype=np.float32)
            raw_boxes[:] = default_box
        else:
            first_idx, last_idx = valid_indices[0], valid_indices[-1]
            raw_boxes[:first_idx] = raw_boxes[first_idx]
            raw_boxes[last_idx + 1:] = raw_boxes[last_idx]
            for i in range(len(valid_indices) - 1):
                idx_a, idx_b = valid_indices[i], valid_indices[i + 1]
                if idx_b - idx_a > 1:
                    box_a, box_b = raw_boxes[idx_a], raw_boxes[idx_b]
                    for t in range(idx_a + 1, idx_b):
                        alpha = float(t - idx_a) / float(idx_b - idx_a)
                        raw_boxes[t] = (1.0 - alpha) * box_a + alpha * box_b

        smoothed_boxes = np.zeros_like(raw_boxes)
        smoothed_boxes[0] = raw_boxes[0]
        alpha = self.box_smooth_alpha
        for t in range(1, T):
            smoothed_boxes[t] = alpha * raw_boxes[t] + (1.0 - alpha) * smoothed_boxes[t - 1]

        crop_boxes = np.zeros((T, 4), dtype=np.int32)
        for t in range(T):
            bx1, by1, bx2, by2 = smoothed_boxes[t]
            x1 = max(0, min(target_w - 2, int(round(bx1))))
            y1 = max(0, min(target_h - 2, int(round(by1))))
            x2 = max(x1 + 1, min(target_w, int(round(bx2))))
            y2 = max(y1 + 1, min(target_h, int(round(by2))))
            crop_boxes[t] = [x1, y1, x2, y2]

        return crop_boxes, smoothed_boxes

    def extract_from_frames(
            self,
            frames: List[np.ndarray],
            crop_faces: bool = True,
            return_crop_boxes: bool = False,
            visual_frame_stride: int = 1,
    ) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool]],
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool], np.ndarray],
    ]:
        """Pipeline trích xuất 68 điểm mốc + ảnh cắt khuôn mặt từ chuỗi khung hình BGR."""
        target_w, target_h = self.letterbox_size
        uniform_frames = [
            letterbox(f, new_size=(target_w, target_h)) if (f.shape[1] != target_w or f.shape[0] != target_h) else f
            for f in frames
        ]

        all_dets = self._detect_faces_batched(uniform_frames)
        chosen_bboxes, found = self._select_driver_faces(all_dets)
        landmarks_norm, landmarks_px = self._extract_landmarks_batched(uniform_frames, chosen_bboxes, found)
        crop_boxes, _ = self.compute_face_crop_boxes(chosen_bboxes, landmarks_px, found)

        face_frames_tensor = None
        if crop_faces:
            v_stride = max(1, int(visual_frame_stride))
            if v_stride > 1:
                face_frames_tensor = crop_faces_from_boxes(
                    uniform_frames[::v_stride], crop_boxes[::v_stride], face_size=self.face_size
                )
            else:
                face_frames_tensor = crop_faces_from_boxes(uniform_frames, crop_boxes, face_size=self.face_size)

        landmarks_tensor = torch.from_numpy(landmarks_norm).float()
        if return_crop_boxes:
            return face_frames_tensor, landmarks_tensor, chosen_bboxes, found, crop_boxes
        return face_frames_tensor, landmarks_tensor, chosen_bboxes, found


# =====================================================================
# 2. ĐỌC NHANH KHUNG HÌNH TỪ VIDEO (FAST VIDEO SEEKER)
# =====================================================================
def fast_read_video_frames(
        video_path: Union[str, Path],
        target_indices: np.ndarray,
        letterbox_size: Tuple[int, int] = (640, 640),
) -> Tuple[List[np.ndarray], float, int]:
    """Đọc chính xác các chỉ số `target_indices` từ video bằng seek + cap.grab() + letterbox."""
    if len(target_indices) == 0:
        target_indices = np.array([0], dtype=int)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở tệp video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_w, target_h = _parse_size(letterbox_size)

    frames: List[np.ndarray] = []
    curr_frame = int(target_indices[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

    try:
        for target_idx in target_indices:
            target_idx = int(target_idx)
            if target_idx < curr_frame and frames:
                frames.append(frames[-1].copy())
                continue

            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1

            ret, frame = cap.read()
            if not ret or frame is None:
                break

            frames.append(letterbox(frame, new_size=(target_w, target_h)))
            curr_frame += 1
    finally:
        cap.release()

    if not frames:
        raise IOError(f"Không đọc được khung hình nào từ: {video_path}")

    while len(frames) < len(target_indices):
        frames.append(frames[-1].copy())

    return frames, fps, total_frames


# =====================================================================
# 3. LỚP LITEFAT DATASET (NẠP MANIFEST .CSV / .JSON CHO KSS 1-9 @ 10 FPS)
# =====================================================================
def _safe_int(val: Any, default: int = 0) -> int:
    if val is None or str(val).strip() == "":
        return default
    return int(round(float(val)))


def _safe_float(val: Any, default: float = 0.0) -> float:
    if val is None or str(val).strip() == "":
        return default
    return float(val)


class LiteFatDataset(Dataset):
    """
    PyTorch Dataset nạp dữ liệu KSS (thang điểm 1-9) cho LiteFat_v1 từ cấu trúc
    như E:\\LSTM\\data_processed_v1:
      - Chỉ đọc đường dẫn file và metadata mẫu từ file manifest (.csv hoặc .json)
        trong thư mục dataset (không quét thư mục video trực tiếp).
      - Mặc định lấy mẫu ở 10 FPS (`time_interval=0.1`) với toàn bộ chiều dài video
        (`window_sec=None`) để hỗ trợ chạy Sliding Window 60s (stride 30s) tích lũy đặc trưng.
      - Trả về `label` kiểu `torch.long` (giá trị nguyên 1..9 từ cột `label`/`kss_score`)
        và lưu kèm `kss_raw`, `kss_class`, `sample_weight` trong `meta`.
    """

    _SHARED_CACHE: Dict[str, Dict[str, Any]] = {}
    _SHARED_META: Dict[str, Dict[str, Any]] = {}

    def __init__(
            self,
            dataset_dir: Union[str, Path] = "/content/data_processed_v1",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = None,
            manifest_filename: str = "dataset_merged_split.csv",
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            bbox_margin: float = 0.15,
            return_meta: bool = False,
            return_raw_frames: bool = False,
            use_cache: bool = False,
            cache_dir: Optional[Union[str, Path]] = None,
            extractor: Optional[FastPIPNetExtractor] = None,
            device: Optional[str] = None,
            extract_crops_only: bool = False,
            pipnet_backbone: str = "resnet18",
            nms_threshold: float = 0.3,
            box_smooth_alpha: float = 0.7,
            selected_datasets: Optional[Union[List[str], str]] = None,
            use_visual_context: bool = True,
            augment_train: bool = True,
            flip_prob: float = 0.5,
            scale_jitter: float = 0.08,
            shift_jitter: float = 0.05,
            roll_deg: float = 12.0,
            temporal_dropout: float = 0.05,
            landmark_noise_std: float = 0.002,
            color_jitter: float = 0.2,
            spatial_dropout: float = 0.15,
            eval_window_mode: str = "center",
            drop_single_class_subjects: bool = False,
            visual_frame_stride: int = 1,
    ):
        split = split.lower().strip()
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        if time_interval is not None and float(time_interval) < 0:
            raise ValueError(f"time_interval phải là số không âm (>= 0): {time_interval}")
        if window_sec is not None and float(window_sec) <= 0:
            raise ValueError(f"window_sec phải là số dương hoặc None: {window_sec}")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval) if time_interval is not None else 0.1
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.manifest_filename = str(manifest_filename)
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
        self.bbox_margin = bbox_margin
        self.box_smooth_alpha = box_smooth_alpha
        self.pipnet_backbone = pipnet_backbone
        self.device = device
        self.return_meta = bool(return_meta)
        self.return_raw_frames = bool(return_raw_frames)
        self.use_cache = bool(use_cache)
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        self.eval_window_mode = str(eval_window_mode).lower().strip()
        self.drop_single_class_subjects = bool(drop_single_class_subjects)
        self.visual_frame_stride = max(1, int(visual_frame_stride))

        if self.cache_dir is not None:
            self.crop_cache_dir = self.cache_dir / f"face_crops_stride{self.visual_frame_stride}"
            try:
                self.crop_cache_dir.mkdir(parents=True, exist_ok=True)
            except Exception:
                pass
        else:
            self.crop_cache_dir = None

        self.use_visual_context = bool(use_visual_context)
        self.extract_crops_only = bool(extract_crops_only) or (not self.use_visual_context)

        if selected_datasets is not None:
            if isinstance(selected_datasets, str):
                self.selected_datasets = [selected_datasets.lower()]
            else:
                self.selected_datasets = [str(s).lower() for s in selected_datasets if str(s).strip()]
                if not self.selected_datasets or any(s in ("all", "*") for s in self.selected_datasets):
                    self.selected_datasets = None
        else:
            self.selected_datasets = None

        self.augment_train = bool(augment_train)
        self.flip_prob = float(flip_prob)
        self.scale_jitter = float(scale_jitter)
        self.shift_jitter = float(shift_jitter)
        self.roll_deg = float(roll_deg)
        self.temporal_dropout = float(temporal_dropout)
        self.landmark_noise_std = float(landmark_noise_std)
        self.color_jitter = float(color_jitter)
        self.spatial_dropout = float(spatial_dropout)

        self.manifest_path: Optional[Path] = None
        self.manifest_summary: Dict[str, Any] = {}
        self.cache_meta: Dict[str, Any] = {}
        self.cached_data: Optional[Dict[str, Any]] = None

        if self.use_cache and self.cache_dir is not None:
            self._load_cache()

        self.samples = self._load_samples()
        if len(self.samples) == 0:
            logger.warning(f"Không tìm thấy mẫu hợp lệ nào trong {self.manifest_path} cho split='{self.split}'")

        self.extractor = extractor
        if self.extractor is None and not (self.use_cache and self.cached_data is not None):
            self.extractor = self._get_or_create_extractor()

    def _get_or_create_extractor(self) -> FastPIPNetExtractor:
        """Khởi tạo trễ (Lazy init) FastPIPNetExtractor khi cần trích xuất trực tiếp từ video."""
        if self.extractor is None:
            self.extractor = FastPIPNetExtractor(
                backbone=self.pipnet_backbone,
                device=self.device,
                letterbox_size=self.letterbox_size,
                face_size=self.face_size,
                conf_threshold=self.conf_threshold,
                nms_threshold=self.nms_threshold,
                bbox_margin=self.bbox_margin,
                box_smooth_alpha=self.box_smooth_alpha,
            )
        return self.extractor

    def _is_dataset_selected(self, key: str, dataset_field: Optional[str] = None) -> bool:
        """Kiểm tra mẫu có thuộc danh sách selected_datasets hay không."""
        if self.selected_datasets is None:
            return True
        if dataset_field and any(ds in str(dataset_field).lower() for ds in self.selected_datasets):
            return True
        key_lower = key.lower()
        return any(ds in key_lower for ds in self.selected_datasets)

    def _resolve_manifest_path(self) -> Path:
        """
        Xác định đường dẫn file manifest (.csv hoặc .json) trong thư mục dataset.
        Ưu tiên manifest_filename, fallback sang phần mở rộng đối ứng (.csv <-> .json)
        hoặc tệp .csv/.json đầu tiên có trong thư mục gốc.
        """
        candidate = Path(self.manifest_filename)
        if not candidate.is_absolute():
            candidate = self.root / self.manifest_filename

        if candidate.is_file() and candidate.suffix.lower() in (".csv", ".json"):
            return candidate

        alt_suffix = ".json" if candidate.suffix.lower() == ".csv" else ".csv"
        alt_candidate = candidate.with_suffix(alt_suffix)
        if alt_candidate.is_file():
            logger.info(f"[Manifest] Chuyển sang đọc tệp đối ứng: {alt_candidate.name}")
            return alt_candidate

        if self.root.is_dir():
            for ext in ("*.csv", "*.json"):
                matches = sorted(self.root.glob(ext))
                if matches:
                    logger.info(f"[Manifest] Tự động sử dụng manifest tìm thấy: {matches[0].name}")
                    return matches[0]

        raise FileNotFoundError(
            f"Không tìm thấy tệp manifest (.csv hoặc .json) '{self.manifest_filename}' trong thư mục: {self.root}"
        )

    def _parse_manifest_record(
            self,
            row: Dict[str, Any],
            default_split: Optional[str] = None,
    ) -> Optional[Dict[str, Any]]:
        """Chuẩn hóa 1 bản ghi từ CSV hoặc JSON sang cấu trúc sample thống nhất cho KSS 1-9."""
        raw_split = row.get("split")
        split_val = str(raw_split if raw_split is not None else (default_split or "")).strip().lower()
        if split_val != self.split:
            return None

        rel_path = row.get("rel_path") or row.get("processed_path") or row.get("path")
        if not rel_path or str(rel_path).strip() == "":
            return None

        norm_rel = Path(str(rel_path).strip().replace("\\", "/")).as_posix()
        dataset_name = str(row.get("dataset", "") or "unknown")
        if not self._is_dataset_selected(norm_rel, dataset_name):
            return None

        rel_obj = Path(norm_rel)
        full_path = rel_obj if rel_obj.is_absolute() else (self.root / norm_rel)

        raw_lbl = row.get("label")
        if raw_lbl is None or str(raw_lbl).strip() == "":
            raw_lbl = row.get("kss_score", row.get("kss_raw", 1))
        label_int = _safe_int(raw_lbl, default=1)
        kss_score = _safe_int(row.get("kss_score"), default=label_int)
        kss_raw = _safe_float(row.get("kss_raw"), default=float(kss_score))
        kss_class = max(0, min(8, label_int - 1))
        sample_weight = _safe_float(row.get("sample_weight"), default=1.0)

        return {
            "key": norm_rel,
            "path": full_path,
            "orig_path": str(row.get("orig_path", "") or ""),
            "label": label_int,
            "kss_score": kss_score,
            "kss_raw": kss_raw,
            "kss_class": kss_class,
            "sample_weight": sample_weight,
            "video_id": str(row.get("video_id") or rel_obj.stem),
            "subject_id": str(row.get("subject_id", "") or "unknown"),
            "session_id": str(row.get("session_id", "") or ""),
            "condition": str(row.get("condition", "") or ""),
            "dataset": dataset_name,
            "fps": _safe_float(row.get("fps"), default=30.0),
            "frame_count": _safe_int(row.get("frame_count"), default=0),
            "width": _safe_int(row.get("width"), default=0),
            "height": _safe_int(row.get("height"), default=0),
            "duration_sec": _safe_float(row.get("duration_sec"), default=0.0),
        }

    def _load_samples(self) -> List[Dict[str, Any]]:
        """Chỉ đọc danh sách mẫu và đường dẫn file từ manifest .csv hoặc .json."""
        manifest_path = self._resolve_manifest_path()
        self.manifest_path = manifest_path
        ext = manifest_path.suffix.lower()
        samples: List[Dict[str, Any]] = []

        if ext == ".csv":
            with open(manifest_path, mode="r", encoding="utf-8-sig", newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    parsed = self._parse_manifest_record(row)
                    if parsed is not None:
                        samples.append(parsed)
        elif ext == ".json":
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                data = json.load(f)

            default_split: Optional[str] = None
            if isinstance(data, dict) and isinstance(data.get("splits"), dict):
                self.manifest_summary = data.get("summary", {}) if isinstance(data.get("summary"), dict) else {}
                records = data["splits"].get(self.split, [])
                default_split = self.split
            elif isinstance(data, dict) and ("samples" in data or "data" in data):
                self.manifest_summary = data.get("summary", {}) if isinstance(data.get("summary"), dict) else {}
                records = data.get("samples") or data.get("data") or []
            elif isinstance(data, list):
                records = data
            else:
                raise ValueError(f"Cấu trúc tệp JSON manifest không hợp lệ: {manifest_path}")

            for row in records:
                if isinstance(row, dict):
                    parsed = self._parse_manifest_record(row, default_split=default_split)
                    if parsed is not None:
                        samples.append(parsed)
        else:
            raise ValueError(f"Chỉ hỗ trợ đọc manifest .csv hoặc .json, không hỗ trợ: {manifest_path}")

        if self.split == "train" and self.drop_single_class_subjects:
            sub_labels: Dict[str, set] = defaultdict(set)
            for s in samples:
                sub_labels[s["subject_id"]].add(s["label"])
            single_class_subs = {sub for sub, lbls in sub_labels.items() if len(lbls) < 2}
            if single_class_subs:
                orig_cnt = len(samples)
                samples = [s for s in samples if s["subject_id"] not in single_class_subs]
                logger.info(
                    f"[Dataset] Đã loại {orig_cnt - len(samples)} mẫu từ {len(single_class_subs)} "
                    f"subject chỉ có 1 mức KSS trong train: {sorted(single_class_subs)}"
                )

        return samples

    def _load_cache(self) -> None:
        """Nạp dữ liệu cache từ tệp .pt theo từng cache_dir (hỗ trợ Shared Cross-Split Cache Pool)."""
        assert self.cache_dir is not None
        pool_key = str(self.cache_dir.resolve()) if self.cache_dir.exists() else str(self.cache_dir)
        cache_pool = self._SHARED_CACHE.setdefault(pool_key, {})
        meta_pool = self._SHARED_META.setdefault(pool_key, {})

        cache_path = self.cache_dir / f"{self.split}.pt"
        if not cache_path.is_file():
            logger.warning(f"[Cache] Không tìm thấy tệp cache {cache_path}. Sẽ trích xuất on-the-fly nếu mẫu không có trong cache.")

        for split_name in (self.split, "train", "val", "test"):
            split_file = f"{split_name}.pt"
            split_path = self.cache_dir / split_file
            if split_file not in meta_pool and split_path.is_file():
                try:
                    payload = torch.load(split_path, map_location="cpu", weights_only=False)
                    if isinstance(payload, dict) and "samples" in payload:
                        items = payload["samples"]
                        header_meta = {k: v for k, v in payload.items() if k not in ("samples", "meta")}
                        nested_meta = payload.get("meta") if isinstance(payload.get("meta"), dict) else {}
                        meta_pool[split_file] = {**header_meta, **nested_meta}
                    else:
                        items = payload
                        meta_pool[split_file] = {}
                    cache_pool.update(items)
                    logger.info(
                        f"[Cache] Đã nạp {len(items)} mẫu từ {split_file} vào Shared Cache Pool "
                        f"(Tổng: {len(cache_pool)})."
                    )
                except Exception as e:
                    logger.warning(f"[Cache] Lỗi khi nạp cache {split_path}: {e}")

        if cache_pool:
            self.cached_data = cache_pool
            split_meta = meta_pool.get(f"{self.split}.pt") or next(iter(meta_pool.values()), {})
            raw_dt = split_meta.get("time_interval", self.time_interval)
            raw_lbs = split_meta.get("letterbox_size", self.letterbox_size)
            self.cache_meta = {
                "format_version": 1,
                "split": self.split,
                "time_interval": float(self.time_interval if raw_dt is None else raw_dt),
                "window_sec": self.window_sec,
                "letterbox_size": self.letterbox_size if raw_lbs is None else raw_lbs,
                "face_size": self.face_size,
                "num_samples": len(cache_pool),
            }

    def _compute_window_span(self, duration_sec: float) -> Tuple[float, float]:
        """Xác định (start_sec, win_sec) thống nhất cho cả đọc video trực tiếp và đọc cache."""
        if self.window_sec is None:
            return 0.0, duration_sec

        if self.split != "train":
            if self.eval_window_mode == "full":
                return 0.0, duration_sec
            if self.eval_window_mode == "legacy18s":
                eval_win = min(max(self.window_sec, 18.0), duration_sec)
                return 0.0, eval_win
            if self.eval_window_mode == "start":
                eval_win = min(self.window_sec, duration_sec)
                return 0.0, eval_win
            # "center"
            eval_win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - eval_win)
            return max_start / 2.0, eval_win

        win = min(self.window_sec, duration_sec)
        max_start = max(0.0, duration_sec - win)
        start_sec = random.uniform(0.0, max_start) if max_start > 0 else 0.0
        return start_sec, win

    def _build_timestamps(self, start_sec: float, span_sec: float) -> np.ndarray:
        """Tạo mảng mốc thời gian lấy mẫu tránh lỗi làm tròn dấu phẩy động (vd: 60.0s @ 0.1s -> đúng 600)."""
        if span_sec <= 0.0:
            return np.array([start_sec], dtype=np.float64)
        end_bound = max(self.time_interval * 0.5, span_sec) - 1e-6
        timestamps = start_sec + np.arange(0.0, max(1e-7, end_bound), self.time_interval, dtype=np.float64)
        if len(timestamps) == 0:
            timestamps = np.array([start_sec], dtype=np.float64)
        return timestamps

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        """Tính toán chỉ số khung hình cần đọc từ video theo time_interval và window_sec."""
        if total_frames <= 0:
            return np.array([0], dtype=int)

        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        if self.time_interval == 0:
            if self.window_sec is None or (self.split != "train" and self.eval_window_mode == "full"):
                return np.arange(total_frames, dtype=int)
            start_sec, win_sec = self._compute_window_span(duration_sec)
            num_consec = max(1, min(int(round(win_sec * fps)), total_frames))
            start_idx = max(0, min(total_frames - num_consec, int(round(start_sec * fps))))
            return np.arange(start_idx, start_idx + num_consec, dtype=int)

        start_sec, win_sec = self._compute_window_span(duration_sec)
        timestamps = self._build_timestamps(start_sec, win_sec)
        indices = np.rint(timestamps * fps).astype(int)
        return np.clip(indices, 0, total_frames - 1)

    def _slice_cached_sequence(self, item: Dict[str, Any], fps: float) -> np.ndarray:
        """Xác định chỉ số cắt trên chuỗi cache .pt theo time_interval và window_sec."""
        landmarks = item["landmarks"]
        T_cache = int(landmarks.shape[0])
        if T_cache <= 0:
            return np.array([0], dtype=int)

        cache_dt = float(self.cache_meta.get("time_interval", 0.1))
        if cache_dt <= 0:
            cache_dt = 1.0 / (fps if fps > 0 else 30.0)

        meta = item.get("meta", {})
        duration_sec = float(meta.get("duration_sec", T_cache * cache_dt))
        if duration_sec <= 0:
            duration_sec = T_cache * cache_dt

        if self.time_interval == 0:
            if self.window_sec is None or (self.split != "train" and self.eval_window_mode == "full"):
                return np.arange(T_cache, dtype=int)
            start_sec, win_sec = self._compute_window_span(duration_sec)
            num_consec = max(1, min(int(round(win_sec / cache_dt)), T_cache))
            start_idx = max(0, min(T_cache - num_consec, int(round(start_sec / cache_dt))))
            return np.arange(start_idx, start_idx + num_consec, dtype=int)

        if (self.window_sec is None or (self.split != "train" and self.eval_window_mode == "full")) and abs(
            self.time_interval - cache_dt
        ) < 1e-5:
            return np.arange(T_cache, dtype=int)

        start_sec, win_sec = self._compute_window_span(duration_sec)
        timestamps = self._build_timestamps(start_sec, win_sec)
        pos = np.rint(timestamps / cache_dt).astype(int)
        return np.clip(pos, 0, T_cache - 1)

    def _build_kss_meta(
            self,
            sample: Dict[str, Any],
            fps: float,
            total_frames: int,
            duration_sec: float,
            selected_target_indices: np.ndarray,
            num_frames: int,
            found_ratio: float,
            letterbox_size: Tuple[int, int],
            cached: bool,
    ) -> Dict[str, Any]:
        """Khởi tạo dictionary metadata thống nhất chứa đầy đủ thông tin KSS (1-9)."""
        return {
            "key": sample["key"],
            "video_id": sample["video_id"],
            "subject_id": sample["subject_id"],
            "session_id": sample["session_id"],
            "condition": sample["condition"],
            "dataset": sample["dataset"],
            "domain_id": (0 if str(sample.get("dataset", "")).upper().startswith("UL") else 1),
            "path": str(sample["path"]),
            "orig_path": sample["orig_path"],
            "split": self.split,
            "label": int(sample["label"]),
            "kss_score": int(sample["kss_score"]),
            "kss_raw": float(sample["kss_raw"]),
            "kss_class": int(sample["kss_class"]),
            "sample_weight": float(sample["sample_weight"]),
            "fps": float(fps),
            "width": int(sample["width"]),
            "height": int(sample["height"]),
            "total_frames": int(total_frames),
            "duration_sec": float(duration_sec),
            "target_indices": torch.as_tensor(selected_target_indices, dtype=torch.long),
            "num_frames": int(num_frames),
            "found_faces_ratio": float(found_ratio),
            "letterbox_size": letterbox_size,
            "cached": bool(cached),
        }

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any], List[np.ndarray]],
    ]:
        sample = self.samples[index]
        key = sample["key"]
        label = torch.tensor(sample["label"], dtype=torch.long)

        # 1. Đường nhanh: Nạp từ RAM Cache .pt nếu có
        if self.cached_data is not None and key in self.cached_data:
            item = self.cached_data[key]
            fps = float(sample.get("fps") or item.get("meta", {}).get("fps", 30.0))
            pos = self._slice_cached_sequence(item, fps)

            landmarks = item["landmarks"][pos]
            if landmarks.shape[-1] > 2:
                landmarks = landmarks[..., :2]
            crop_boxes = item["crop_boxes"][pos] if "crop_boxes" in item else None
            bboxes = item["bboxes"][pos] if "bboxes" in item else None
            found = item["found"][pos] if "found" in item else None

            cached_target_indices = item.get("target_indices")
            if cached_target_indices is None and isinstance(item.get("meta"), dict):
                cached_target_indices = item["meta"].get("target_indices")

            if cached_target_indices is not None:
                if isinstance(cached_target_indices, torch.Tensor):
                    selected_target_indices = cached_target_indices[pos].long().cpu().numpy()
                else:
                    selected_target_indices = np.asarray(cached_target_indices, dtype=int)[pos]
            else:
                cache_dt = float(self.cache_meta.get("time_interval", 0.1))
                selected_target_indices = np.rint(pos * (cache_dt * fps)).astype(int)

            cache_lbs = _parse_size(self.cache_meta.get("letterbox_size", self.letterbox_size))
            face_frames = item.get("face_frames", None)
            if face_frames is not None:
                face_frames = face_frames[pos]

            raw_frames: Optional[List[np.ndarray]] = None
            v_stride = self.visual_frame_stride
            expected_v_len = len(range(0, len(selected_target_indices), v_stride))

            if (
                not self.extract_crops_only
                and face_frames is None
                and not self.return_raw_frames
                and self.crop_cache_dir is not None
                and self.window_sec is None
            ):
                cache_key = hashlib.md5(key.encode()).hexdigest()
                crop_file = self.crop_cache_dir / f"{cache_key}.pt"
                if crop_file.is_file():
                    try:
                        cached_uint8 = torch.load(crop_file, map_location="cpu", weights_only=True)
                        if cached_uint8.shape[0] == expected_v_len:
                            norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
                            norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
                            face_frames = (cached_uint8.float() / 255.0 - norm_mean) / norm_std
                    except Exception:
                        face_frames = None

            need_video_read = (not self.extract_crops_only and face_frames is None) or self.return_raw_frames
            if need_video_read:
                if v_stride > 1 and not self.return_raw_frames:
                    v_pos = np.arange(0, len(selected_target_indices), v_stride)
                    v_indices = selected_target_indices[v_pos]
                    v_crop_boxes = crop_boxes[v_pos] if crop_boxes is not None else None
                else:
                    v_indices = selected_target_indices
                    v_crop_boxes = crop_boxes

                raw_frames, actual_fps, _ = fast_read_video_frames(
                    sample["path"], v_indices, letterbox_size=cache_lbs
                )
                fps = actual_fps or fps
                if not self.extract_crops_only and face_frames is None and v_crop_boxes is not None:
                    face_frames = crop_faces_from_boxes(raw_frames, v_crop_boxes, face_size=self.face_size)
                    if self.crop_cache_dir is not None and self.window_sec is None:
                        try:
                            cache_key = hashlib.md5(key.encode()).hexdigest()
                            crop_file = self.crop_cache_dir / f"{cache_key}.pt"
                            norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
                            norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
                            uint8_crops = ((face_frames * norm_std + norm_mean).clamp(0.0, 1.0) * 255.0).to(torch.uint8)
                            atomic_save(uint8_crops, crop_file)
                        except Exception:
                            pass

            if not self.use_visual_context:
                face_frames = None

            if self.split == "train" and self.augment_train:
                face_frames, landmarks = augment_multimodal_clip(
                    face_frames,
                    landmarks,
                    is_train=True,
                    flip_prob=self.flip_prob,
                    scale_jitter=self.scale_jitter,
                    shift_jitter=self.shift_jitter,
                    roll_deg=self.roll_deg,
                    temporal_dropout=self.temporal_dropout,
                    landmark_noise_std=self.landmark_noise_std,
                    color_jitter=self.color_jitter,
                    spatial_dropout=self.spatial_dropout,
                )

            if found is not None:
                found_t = torch.as_tensor(found, dtype=torch.bool)
                found_ratio = float(found_t.sum().item()) / float(max(1, len(found_t)))
            else:
                found_t = torch.ones(len(pos), dtype=torch.bool)
                found_ratio = 1.0

            item_meta = item.get("meta", {}) if isinstance(item.get("meta"), dict) else {}
            meta = self._build_kss_meta(
                sample=sample,
                fps=fps,
                total_frames=int(sample.get("frame_count") or item_meta.get("total_frames", 0)),
                duration_sec=float(sample.get("duration_sec") or item_meta.get("duration_sec", 0.0)),
                selected_target_indices=selected_target_indices,
                num_frames=int(landmarks.shape[0]),
                found_ratio=found_ratio,
                letterbox_size=cache_lbs,
                cached=True,
            )
            if crop_boxes is not None:
                meta["crop_boxes"] = torch.as_tensor(crop_boxes, dtype=torch.int32)
            if bboxes is not None:
                meta["bboxes"] = torch.as_tensor(bboxes, dtype=torch.float32)
            if found is not None:
                meta["found"] = found_t

            for k_meta, v_meta in item_meta.items():
                if k_meta not in meta and k_meta != "path":
                    meta[k_meta] = v_meta
            if "path" in item_meta and str(item_meta["path"]) != str(sample["path"]):
                meta["remote_path"] = str(item_meta["path"])

            if self.return_raw_frames and self.return_meta:
                return face_frames, landmarks, label, meta, (raw_frames if raw_frames is not None else [])
            if self.return_meta:
                return face_frames, landmarks, label, meta
            return face_frames, landmarks, label

        # 2. Đường chuẩn: Đọc trực tiếp từ đường dẫn file trong manifest (.csv / .json)
        extractor = self._get_or_create_extractor()
        fps = sample["fps"]
        total_frames = sample["frame_count"]

        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 300)
            cap.release()
            sample["fps"] = fps
            sample["frame_count"] = total_frames

        target_indices = self._get_frame_indices(total_frames, fps)
        raw_frames, actual_fps, _ = fast_read_video_frames(
            sample["path"], target_indices, letterbox_size=self.letterbox_size
        )

        crop_faces = not self.extract_crops_only
        face_frames, landmarks, bboxes, found, crop_boxes = extractor.extract_from_frames(
            raw_frames,
            crop_faces=crop_faces,
            return_crop_boxes=True,
            visual_frame_stride=self.visual_frame_stride,
        )

        if not self.use_visual_context:
            face_frames = None

        if self.split == "train" and self.augment_train:
            face_frames, landmarks = augment_multimodal_clip(
                face_frames,
                landmarks,
                is_train=True,
                flip_prob=self.flip_prob,
                scale_jitter=self.scale_jitter,
                shift_jitter=self.shift_jitter,
                roll_deg=self.roll_deg,
                temporal_dropout=self.temporal_dropout,
                landmark_noise_std=self.landmark_noise_std,
                color_jitter=self.color_jitter,
                spatial_dropout=self.spatial_dropout,
            )

        T_frames = len(found)
        bboxes_tensor = torch.zeros((T_frames, 5), dtype=torch.float32)
        for t_idx, b in enumerate(bboxes):
            if b is not None and len(b) >= 4:
                b_arr = np.asarray(b, dtype=np.float32)
                bboxes_tensor[t_idx, :min(5, len(b_arr))] = torch.from_numpy(b_arr[:min(5, len(b_arr))])

        duration_sec = float(sample["duration_sec"]) if sample["duration_sec"] > 0 else (
            float(total_frames / actual_fps) if actual_fps > 0 else 0.0
        )
        meta = self._build_kss_meta(
            sample=sample,
            fps=actual_fps,
            total_frames=total_frames,
            duration_sec=duration_sec,
            selected_target_indices=target_indices,
            num_frames=int(landmarks.shape[0]),
            found_ratio=float(sum(found)) / float(max(1, len(found))),
            letterbox_size=self.letterbox_size,
            cached=False,
        )
        meta["crop_boxes"] = torch.from_numpy(crop_boxes).int()
        meta["bboxes"] = bboxes_tensor
        meta["found"] = torch.tensor(found, dtype=torch.bool)
        meta["raw_bboxes_list"] = bboxes

        if self.return_raw_frames and self.return_meta:
            return face_frames, landmarks, label, meta, raw_frames
        if self.return_meta:
            return face_frames, landmarks, label, meta
        return face_frames, landmarks, label


# =====================================================================
# 4. HÀM COLLATE & XÂY DỰNG DATALOADER CHO LITEFAT_V1 (KSS 1-9)
# =====================================================================
def collate_litefat(batch: List[Tuple[Any, ...]]):
    """
    Collate function cho DataLoader LiteFat_v1:
      - Tự động đệm (padding) theo chiều thời gian T khi các mẫu có độ dài khác nhau.
      - Hỗ trợ face_frames=None (chế độ Landmark-Only).
      - batch_labels có kiểu torch.long [B] chứa nhãn KSS (1..9).
    """
    num_elements = len(batch[0])
    face_frames_list = [item[0] for item in batch]
    landmarks_list = [item[1] for item in batch]
    labels_list = [item[2] for item in batch]

    lengths = [lm.shape[0] for lm in landmarks_list]
    max_len = max(lengths)

    has_face_frames = all(f is not None for f in face_frames_list)
    v_lengths = [f.shape[0] for f in face_frames_list] if has_face_frames else []
    max_v_len = max(v_lengths) if has_face_frames else 0

    padded_frames = []
    padded_landmarks = []
    for idx, (lm, cur_len) in enumerate(zip(landmarks_list, lengths)):
        f = face_frames_list[idx]
        if cur_len < max_len:
            pad_lm = torch.ones((max_len - cur_len, lm.shape[1], lm.shape[2]), dtype=lm.dtype)
            padded_landmarks.append(torch.cat([lm, pad_lm], dim=0))
        else:
            padded_landmarks.append(lm)

        if has_face_frames and f is not None:
            cur_v_len = f.shape[0]
            if cur_v_len < max_v_len:
                last_frame = f[-1:].repeat(max_v_len - cur_v_len, 1, 1, 1)
                padded_frames.append(torch.cat([f, last_frame], dim=0))
            else:
                padded_frames.append(f)

    batch_frames = torch.stack(padded_frames, dim=0) if has_face_frames else None
    batch_landmarks = torch.stack(padded_landmarks, dim=0)
    batch_labels = torch.stack(labels_list, dim=0)
    batch_lengths = torch.tensor(lengths, dtype=torch.long)

    if num_elements >= 4:
        metas = [item[3] for item in batch]
        if num_elements == 5:
            raw_frames_list = [item[4] for item in batch]
            return batch_frames, batch_landmarks, batch_labels, batch_lengths, metas, raw_frames_list
        return batch_frames, batch_landmarks, batch_labels, batch_lengths, metas

    return batch_frames, batch_landmarks, batch_labels, batch_lengths


def build_litefat_dataloaders(
        dataset_dir: Union[str, Path] = "/content/data_processed_v1",
        batch_size: int = 8,
        num_workers: int = 0,
        time_interval: float = 0.1,
        window_sec: Optional[float] = None,
        letterbox_size: Tuple[int, int] = (640, 640),
        face_size: Tuple[int, int] = (224, 224),
        manifest_filename: str = "dataset_merged_split.csv",
        use_cache: bool = False,
        cache_dir: Optional[Union[str, Path]] = None,
        pin_memory: bool = True,
        shuffle_train: bool = True,
        device: Optional[str] = None,
        pipnet_backbone: str = "resnet18",
        conf_threshold: float = 0.5,
        nms_threshold: float = 0.3,
        bbox_margin: float = 0.15,
        box_smooth_alpha: float = 0.7,
        extract_crops_only: bool = False,
        selected_datasets: Optional[Union[List[str], str]] = None,
        use_visual_context: bool = True,
        augment_train: bool = True,
        flip_prob: float = 0.5,
        scale_jitter: float = 0.08,
        shift_jitter: float = 0.05,
        roll_deg: float = 12.0,
        temporal_dropout: float = 0.05,
        landmark_noise_std: float = 0.0003,
        color_jitter: float = 0.2,
        spatial_dropout: float = 0.15,
        subject_balanced_sampling: bool = True,
        use_manifest_sample_weights: bool = True,
        eval_window_mode: str = "center",
        drop_single_class_subjects: bool = False,
        visual_frame_stride: int = 1,
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Khởi tạo bộ 3 DataLoader (train, val, test) cho mô hình LiteFat_v1 (KSS 1-9 @ 10 FPS)."""
    shared_extractor = (
        FastPIPNetExtractor(
            backbone=pipnet_backbone,
            device=device,
            letterbox_size=letterbox_size,
            face_size=face_size,
            conf_threshold=conf_threshold,
            nms_threshold=nms_threshold,
            bbox_margin=bbox_margin,
            box_smooth_alpha=box_smooth_alpha,
        )
        if not use_cache
        else None
    )

    common_cfg = dict(
        dataset_dir=dataset_dir,
        time_interval=time_interval,
        window_sec=window_sec,
        manifest_filename=manifest_filename,
        letterbox_size=letterbox_size,
        face_size=face_size,
        conf_threshold=conf_threshold,
        nms_threshold=nms_threshold,
        bbox_margin=bbox_margin,
        box_smooth_alpha=box_smooth_alpha,
        pipnet_backbone=pipnet_backbone,
        use_cache=use_cache,
        cache_dir=cache_dir,
        extractor=shared_extractor,
        device=device,
        extract_crops_only=extract_crops_only,
        selected_datasets=selected_datasets,
        use_visual_context=use_visual_context,
        eval_window_mode=eval_window_mode,
        visual_frame_stride=visual_frame_stride,
    )

    train_ds = LiteFatDataset(
        split="train",
        augment_train=augment_train,
        flip_prob=flip_prob,
        scale_jitter=scale_jitter,
        shift_jitter=shift_jitter,
        roll_deg=roll_deg,
        temporal_dropout=temporal_dropout,
        landmark_noise_std=landmark_noise_std,
        color_jitter=color_jitter,
        spatial_dropout=spatial_dropout,
        drop_single_class_subjects=drop_single_class_subjects,
        **common_cfg,
    )
    val_ds = LiteFatDataset(split="val", augment_train=False, **common_cfg)
    test_ds = LiteFatDataset(split="test", augment_train=False, **common_cfg)

    train_sampler = None
    if len(train_ds) > 0:
        if subject_balanced_sampling:
            ds_label_counts: Counter = Counter()
            sub_label_counts: Counter = Counter()
            for s in train_ds.samples:
                ds_name = str(s.get("dataset", "unknown"))
                sub = str(s.get("subject_id", "unknown"))
                lbl = int(s.get("label", 1))
                ds_label_counts[(ds_name, lbl)] += 1
                sub_label_counts[(ds_name, sub, lbl)] += 1

            raw_weights: List[float] = []
            ds_totals: Counter = Counter()
            for s in train_ds.samples:
                ds_name = str(s.get("dataset", "unknown"))
                sub = str(s.get("subject_id", "unknown"))
                lbl = int(s.get("label", 1))
                w = 1.0 / (
                    float(max(1, ds_label_counts[(ds_name, lbl)]))
                    * math.sqrt(float(max(1, sub_label_counts[(ds_name, sub, lbl)])))
                )
                if use_manifest_sample_weights:
                    w *= max(1e-6, float(s.get("sample_weight", 1.0)))
                raw_weights.append(w)
                ds_totals[ds_name] += w

            sample_weights = [
                w / max(1e-12, float(ds_totals[str(s.get("dataset", "unknown"))]))
                for s, w in zip(train_ds.samples, raw_weights)
            ]
            train_sampler = torch.utils.data.WeightedRandomSampler(
                weights=torch.as_tensor(sample_weights, dtype=torch.double),
                num_samples=len(sample_weights),
                replacement=True,
            )
        elif use_manifest_sample_weights:
            sample_weights = [max(1e-6, float(s.get("sample_weight", 1.0))) for s in train_ds.samples]
            train_sampler = torch.utils.data.WeightedRandomSampler(
                weights=torch.as_tensor(sample_weights, dtype=torch.double),
                num_samples=len(sample_weights),
                replacement=True,
            )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=(shuffle_train if train_sampler is None else False),
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_litefat,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_litefat,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_litefat,
    )

    return train_loader, val_loader, test_loader


# =====================================================================
# 5. TRỰC QUAN HÓA MẪU KSS (1-9) & MÔ PHỎNG KINEMATIC FEATURE EXTRACTOR
# =====================================================================
def _format_kss_label(label: Union[torch.Tensor, int, float]) -> Tuple[str, Tuple[int, int, int]]:
    """Định dạng nhãn KSS (1..9) và màu hiển thị BGR."""
    val = float(label.item()) if isinstance(label, torch.Tensor) else float(label)
    if val >= 7.0:
        return f"KSS {val:.1f}/9 (DROWSY)", (0, 0, 255)
    if val >= 5.0:
        return f"KSS {val:.1f}/9 (MID-FATIGUE)", (0, 180, 255)
    return f"KSS {val:.1f}/9 (ALERT)", (0, 255, 0)


def render_kinematic_frame(
        pos_t: np.ndarray,
        vel_t: np.ndarray,
        acc_t: np.ndarray,
        vel_all: np.ndarray,
        acc_all: np.ndarray,
        t: int,
        T: int,
        is_missing: bool = False,
        video_id: str = "sample",
        label_str: str = "",
        label_color: Tuple[int, int, int] = (0, 255, 0),
        raw_iod: float = 0.0,
        raw_landmarks: Optional[np.ndarray] = None,
        target_w: int = 640,
        target_h: int = 640,
        zoom_scale: float = 260.0,
        coord_mode: str = "nose",
) -> np.ndarray:
    """Dựng khung hình trực quan hóa mô phỏng cho lớp KinematicFeatureExtractor."""
    canvas = np.full((720, 1280, 3), (22, 26, 32), dtype=np.uint8)

    cv2.rectangle(canvas, (0, 0), (1280, 48), (14, 17, 22), -1)
    cv2.putText(canvas, "DRIVER GUARDIAN | KINEMATIC FEATURE EXTRACTOR (10 FPS)", (20, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 0), 2, cv2.LINE_AA)
    top_info = f"Clip: {video_id} | Frame {t + 1}/{T} | {label_str} | Zoom: {zoom_scale:.0f}px"
    cv2.putText(canvas, top_info, (680, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, label_color, 2, cv2.LINE_AA)
    cv2.line(canvas, (0, 48), (1280, 48), (45, 55, 70), 1)
    cv2.line(canvas, (720, 48), (720, 720), (45, 55, 70), 1)

    vx1, vy1, vx2, vy2 = 15, 55, 705, 705
    viewport_rect = (vx1, vy1, vx2 - vx1, vy2 - vy1)
    cx, cy = 360, 380
    S = float(zoom_scale)
    is_model_mode = (coord_mode == "model")

    if is_model_mode:
        cv2.putText(canvas, "MODEL INPUT SPACE (ORGAN-CENTERED, static_shape_ratio=0.25) | [M]: Nose IOD=1.0",
                    (25, 78), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (130, 160, 190), 1, cv2.LINE_AA)
        organ_groups = [
            ("L-Eye/Brow (0,0)", (cx - 170, cy - 165)),
            ("R-Eye/Brow (0,0)", (cx + 170, cy - 165)),
            ("Jaw/Nose (0,0)", (cx, cy + 5)),
            ("Mouth (0,0)", (cx, cy + 205)),
        ]
        for g_label, (ox, oy) in organ_groups:
            cv2.line(canvas, (ox - 55, oy), (ox + 55, oy), (38, 48, 60), 1, cv2.LINE_AA)
            cv2.line(canvas, (ox, oy - 55), (ox, oy + 55), (38, 48, 60), 1, cv2.LINE_AA)
            cv2.circle(canvas, (ox, oy), 4, (0, 255, 255), -1, cv2.LINE_AA)
            cv2.putText(canvas, g_label, (ox - 55, oy - 62), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (130, 160, 190), 1, cv2.LINE_AA)
    else:
        cv2.putText(canvas, "LOCAL NOSE-CENTERED SPACE (0,0) & IOD = 1.0 | [M]: Model Input", (25, 78),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.46, (130, 160, 190), 1, cv2.LINE_AA)
        cv2.line(canvas, (vx1, cy), (vx2, cy), (38, 48, 60), 1, cv2.LINE_AA)
        cv2.line(canvas, (cx, vy1), (cx, vy2), (38, 48, 60), 1, cv2.LINE_AA)
        cv2.circle(canvas, (cx, cy), int(S), (32, 40, 52), 1, cv2.LINE_AA)
        cv2.circle(canvas, (cx, cy), int(S * 0.5), (28, 35, 45), 1, cv2.LINE_AA)
        cv2.circle(canvas, (cx, cy), 6, (0, 255, 255), -1, cv2.LINE_AA)

    num_visible = 0
    if is_missing:
        cv2.putText(canvas, "[CANH BAO: MAT MAT / HEAD TURNED]", (120, cy - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2, cv2.LINE_AA)
    else:
        if is_model_mode:
            node_anchors = [(cx, cy + 5)] * 68
            for i in (*range(17, 22), *range(36, 42)):
                node_anchors[i] = (cx - 170, cy - 165)
            for i in (*range(22, 27), *range(42, 48)):
                node_anchors[i] = (cx + 170, cy - 165)
            for i in range(48, 68):
                node_anchors[i] = (cx, cy + 205)
        else:
            node_anchors = [(cx, cy)] * 68

        pts_raw = [
            (int(round(node_anchors[i][0] + pos_t[i, 0] * S)), int(round(node_anchors[i][1] + pos_t[i, 1] * S)))
            for i in range(68)
        ]
        for start, end, is_closed, color in KINEMATIC_REGIONS:
            pts_region = pts_raw[start:end]
            for j in range(len(pts_region) - 1):
                ret, c1, c2 = cv2.clipLine(viewport_rect, pts_region[j], pts_region[j + 1])
                if ret:
                    cv2.line(canvas, c1, c2, color, 2, cv2.LINE_AA)
            if is_closed and len(pts_region) > 2:
                ret, c1, c2 = cv2.clipLine(viewport_rect, pts_region[-1], pts_region[0])
                if ret:
                    cv2.line(canvas, c1, c2, color, 2, cv2.LINE_AA)

        for px, py in pts_raw:
            if vx1 <= px <= vx2 and vy1 <= py <= vy2:
                cv2.circle(canvas, (px, py), 3, (0, 255, 120), -1, cv2.LINE_AA)
                num_visible += 1

        if (not is_model_mode) and raw_landmarks is not None and len(raw_landmarks) > 0 and raw_iod > 1.0:
            trail_pts = []
            for k in range(max(0, t - 10), t + 1):
                rdx = (raw_landmarks[k, 30, 0] - raw_landmarks[t, 30, 0]) * target_w / max(1.0, raw_iod)
                rdy = (raw_landmarks[k, 30, 1] - raw_landmarks[t, 30, 1]) * target_h / max(1.0, raw_iod)
                trail_pts.append((int(round(cx + rdx * S)), int(round(cy + rdy * S))))
            for k in range(len(trail_pts) - 1):
                ret, c1, c2 = cv2.clipLine(viewport_rect, trail_pts[k], trail_pts[k + 1])
                if ret:
                    alpha = int(90 + (k / max(1, len(trail_pts) - 1)) * 165)
                    cv2.line(canvas, c1, c2, (0, alpha, 255), 2, cv2.LINE_AA)

    cv2.rectangle(canvas, (vx1, vy1), (vx2, vy2), (45, 55, 70), 1)

    hud_x1, hud_x2, hud_y1, hud_y2 = 740, 1260, 60, 225
    cv2.rectangle(canvas, (hud_x1, hud_y1), (hud_x2, hud_y2), (16, 20, 26), -1)
    cv2.rectangle(canvas, (hud_x1, hud_y1), (hud_x2, hud_y2), (40, 50, 65), 1)
    cv2.putText(canvas, "REAL-TIME KINEMATIC TELEMETRY", (hud_x1 + 18, hud_y1 + 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 0), 1, cv2.LINE_AA)

    nvx_val, nvy_val = float(vel_t[30, 0]), float(vel_t[30, 1])
    nax_val, nay_val = float(acc_t[30, 0]), float(acc_t[30, 1])
    cv2.putText(canvas, f"Head Velocity (Nose): |V|={np.hypot(nvx_val, nvy_val):.4f} [Vx:{nvx_val:+.3f}, Vy:{nvy_val:+.3f}]",
                (hud_x1 + 18, hud_y1 + 58), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (220, 220, 220), 1, cv2.LINE_AA)
    cv2.putText(canvas, f"Head Acceleration   : |A|={np.hypot(nax_val, nay_val):.4f} [Ax:{nax_val:+.3f}, Ay:{nay_val:+.3f}]",
                (hud_x1 + 18, hud_y1 + 86), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (220, 220, 220), 1, cv2.LINE_AA)
    cv2.putText(canvas, f"Raw IOD: {raw_iod:.1f} px | Visible Nodes: {num_visible}/68",
                (hud_x1 + 18, hud_y1 + 114), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (0, 255, 255), 1, cv2.LINE_AA)

    for series, title, color, y1_g, y2_g in [
        (vel_all[:, 30, 1], "Vertical Velocity Vy(t)", (0, 255, 255), 250, 465),
        (acc_all[:, 30, 1], "Vertical Acceleration Ay(t)", (255, 100, 220), 490, 705),
    ]:
        gx1, gx2 = 740, 1260
        cv2.rectangle(canvas, (gx1, y1_g), (gx2, y2_g), (12, 14, 18), -1)
        cv2.rectangle(canvas, (gx1, y1_g), (gx2, y2_g), (40, 50, 65), 1)
        cv2.putText(canvas, title, (gx1 + 15, y1_g - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)
        zy = (y1_g + y2_g) // 2
        cv2.line(canvas, (gx1, zy), (gx2, zy), (35, 45, 60), 1, cv2.LINE_AA)

        max_amp = 0.15
        pts = [
            (
                int(gx1 + (i / max(1, T - 1)) * (gx2 - gx1)),
                int(zy - (np.clip(float(series[i]), -max_amp, max_amp) / max_amp) * ((y2_g - y1_g) // 2 - 15)),
            )
            for i in range(T)
        ]
        for i in range(len(pts) - 1):
            cv2.line(canvas, pts[i], pts[i + 1], color, 2, cv2.LINE_AA)
        cur_x = int(gx1 + (t / max(1, T - 1)) * (gx2 - gx1))
        cv2.line(canvas, (cur_x, y1_g), (cur_x, y2_g), (0, 0, 255), 2, cv2.LINE_AA)

    return canvas


def visualize_kinematic_simulation(
        landmarks: torch.Tensor,
        label: Union[torch.Tensor, int, float] = 1,
        video_id: str = "sample",
        delay_ms: int = 100,
        window_name: str = "Driver Guardian - KinematicFeatureExtractor Simulation",
) -> None:
    """Hiển thị trực quan hóa mô phỏng KinematicFeatureExtractor từ tensor điểm mốc."""
    if landmarks.dim() == 4:
        landmarks = landmarks.squeeze(0)
    if landmarks.shape[-1] > 2:
        landmarks = landmarks[..., :2]

    T = landmarks.shape[0]
    lms_np = landmarks.cpu().numpy()
    label_str, label_color = _format_kss_label(label)

    from model import KinematicFeatureExtractor
    kfe = KinematicFeatureExtractor(fps=10.0)
    kfe.eval()
    lms_in = landmarks.unsqueeze(0).permute(0, 3, 1, 2).float()
    with torch.no_grad():
        kfe_out = kfe(lms_in)
        p_left_t = lms_in[:, :, :, kfe.left_eye_idx:kfe.left_eye_idx + 1]
        p_right_t = lms_in[:, :, :, kfe.right_eye_idx:kfe.right_eye_idx + 1]
        raw_iod_t = torch.norm(p_right_t - p_left_t, p=2, dim=1, keepdim=True).clamp(min=kfe.eps)
        clip_iod_t = raw_iod_t.mean(dim=2, keepdim=True).clamp(min=kfe.eps)
        iod_scale_t = torch.maximum(0.5 * raw_iod_t + 0.5 * clip_iod_t, 0.65 * clip_iod_t).clamp(min=kfe.eps)
        nose_pos_t = lms_in[:, :, :, kfe.nose_idx:kfe.nose_idx + 1]
        pos_norm_t = torch.clamp((lms_in - nose_pos_t) / iod_scale_t, min=-8.0, max=8.0)
    pos_norm_all = pos_norm_t[0].permute(1, 2, 0).cpu().numpy()
    pos_feat_all = kfe_out[0, :2].permute(1, 2, 0).cpu().numpy()
    vel_norm_all = kfe_out[0, 2:4].permute(1, 2, 0).cpu().numpy()
    acc_norm_all = kfe_out[0, 4:6].permute(1, 2, 0).cpu().numpy()

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1280, 720)

    t, paused, zoom_scale, coord_mode = 0, False, 260.0, "nose"
    while True:
        is_missing = np.allclose(lms_np[t], 1.0)
        p_left = lms_np[t, 36, :2] * np.array([640, 640])
        p_right = lms_np[t, 45, :2] * np.array([640, 640])
        raw_iod = float(np.linalg.norm(p_right - p_left)) if not is_missing else 0.0
        pos_cur = pos_norm_all[t] if coord_mode == "nose" else pos_feat_all[t]

        canvas = render_kinematic_frame(
            pos_t=pos_cur,
            vel_t=vel_norm_all[t],
            acc_t=acc_norm_all[t],
            vel_all=vel_norm_all,
            acc_all=acc_norm_all,
            t=t,
            T=T,
            is_missing=is_missing,
            video_id=video_id,
            label_str=label_str,
            label_color=label_color,
            raw_iod=raw_iod,
            raw_landmarks=lms_np,
            target_w=640,
            target_h=640,
            zoom_scale=zoom_scale,
            coord_mode=coord_mode,
        )
        cv2.imshow(window_name, canvas)
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        key = cv2.waitKey(0 if paused else delay_ms) & 0xFF
        if key in (27, ord("q"), ord("Q")):
            break
        if key == 32:
            paused = not paused
        elif key in (ord("r"), ord("R")):
            t = 0
            continue
        elif key in (ord("d"), ord("D"), 83):
            t = (t + 1) % T
            continue
        elif key in (ord("a"), ord("A"), 81):
            t = (t - 1 + T) % T
            continue
        elif key in (ord("+"), ord("=")):
            zoom_scale = min(600.0, zoom_scale + 30.0)
            continue
        elif key in (ord("-"), ord("_")):
            zoom_scale = max(100.0, zoom_scale - 30.0)
            continue
        elif key == ord("0"):
            zoom_scale = 260.0
            continue
        elif key in (ord("m"), ord("M")):
            coord_mode = "model" if coord_mode == "nose" else "nose"
            continue

        if not paused:
            t = (t + 1) % T

    cv2.destroyAllWindows()


def visualize_litefat_sample(
        face_frames: Optional[torch.Tensor],
        landmarks: torch.Tensor,
        label: Union[torch.Tensor, int, float],
        raw_frames: List[np.ndarray],
        meta: Optional[Dict[str, Any]] = None,
        delay_ms: int = 100,
        window_name: str = "Driver Guardian - LiteFat_v1 KSS Preview",
) -> None:
    """Hiển thị trực quan mẫu KSS (1-9) kèm landmarks, bounding box và ảnh mặt cắt (nếu có)."""
    T = len(raw_frames)
    if T == 0:
        return

    label_str, label_color = _format_kss_label(label)
    video_id = meta.get("video_id", "sample") if meta else "sample"
    bboxes_list = meta.get("bboxes", [None] * T) if meta else [None] * T
    crop_boxes_list = meta.get("crop_boxes", None) if meta else None

    has_faces = face_frames is not None and face_frames.shape[0] > 0
    faces_bgr = []
    if has_faces and face_frames is not None:
        norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        unnorm = (face_frames * norm_std + norm_mean).clamp(0.0, 1.0)
        faces_rgb = (unnorm.permute(0, 2, 3, 1).cpu().numpy() * 255.0).astype(np.uint8)
        faces_bgr = [cv2.cvtColor(faces_rgb[i], cv2.COLOR_RGB2BGR) for i in range(faces_rgb.shape[0])]

    lms_np = landmarks.cpu().numpy()
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    t, paused = 0, False
    while True:
        left_img = raw_frames[t].copy()
        target_h, target_w = left_img.shape[:2]
        is_missing = np.allclose(lms_np[t], 1.0)

        if not is_missing:
            pts_px = lms_np[t, :, :2].copy()
            pts_px[:, 0] *= float(target_w)
            pts_px[:, 1] *= float(target_h)
            bbox = bboxes_list[t] if t < len(bboxes_list) else None
            bbox_arr = bbox.detach().cpu().numpy() if isinstance(bbox, torch.Tensor) else (
                np.asarray(bbox, dtype=np.float32) if bbox is not None else None
            )
            if bbox_arr is not None and len(bbox_arr) >= 5 and float(bbox_arr[4]) > 0:
                bboxes_to_draw = np.array([bbox_arr[:5]], dtype=np.float32)
            else:
                bboxes_to_draw = np.array(
                    [[np.min(pts_px[:, 0]), np.min(pts_px[:, 1]), np.max(pts_px[:, 0]), np.max(pts_px[:, 1]), 1.0]],
                    dtype=np.float32,
                )
            draw_68_landmarks_custom(left_img, np.expand_dims(pts_px, 0), bboxes_to_draw, conf_threshold=0.1)

        if crop_boxes_list is not None and t < len(crop_boxes_list):
            cb = crop_boxes_list[t]
            if isinstance(cb, torch.Tensor):
                cb = cb.detach().cpu().numpy()
            if len(cb) >= 4:
                cv2.rectangle(left_img, (int(cb[0]), int(cb[1])), (int(cb[2]), int(cb[3])), (0, 255, 255), 1)

        if has_faces and faces_bgr:
            f_idx = min(len(faces_bgr) - 1, int(round(t * len(faces_bgr) / float(max(1, T)))))
            right_panel = np.zeros((target_h, target_h, 3), dtype=np.uint8)
            face_disp = cv2.resize(faces_bgr[f_idx], (target_h - 45, target_h - 45), interpolation=cv2.INTER_LINEAR)
            right_panel[45:, 22:22 + target_h - 45] = face_disp
            combined = np.hstack([left_img, right_panel])
        else:
            combined = left_img

        top_bar = f"Video: {video_id} | {label_str} | Frame {t + 1}/{T}" + (" [PAUSED]" if paused else "")
        cv2.putText(combined, top_bar, (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, label_color, 2, cv2.LINE_AA)
        cv2.imshow(window_name, combined)

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        key = cv2.waitKey(0 if paused else delay_ms) & 0xFF
        if key in (27, ord("q"), ord("Q")):
            break
        if key == 32:
            paused = not paused
        elif key in (ord("r"), ord("R")):
            t = 0
            continue
        elif key in (ord("d"), ord("D"), 83):
            t = (t + 1) % T
            continue
        elif key in (ord("a"), ord("A"), 81):
            t = (t - 1 + T) % T
            continue

        if not paused:
            t = (t + 1) % T

    cv2.destroyAllWindows()

In [9]:
# =====================================================================
# File: model.py
# Dự án: Driver Guardian - Module LiteFat (IEEE/RSJ IROS 2025 Upgraded)
# Tác vụ: Định nghĩa kiến trúc LiteFat_v1 dự đoán điểm buồn ngủ KSS (1-9)
#         với cửa sổ mặc định 60s (10 FPS) và cơ chế trượt cửa sổ
#         stride 30s tích lũy đặc trưng kép (Overlap + Gated GRU Memory)
#         cho các chuỗi có độ dài lớn hơn 60s.
# =====================================================================
from __future__ import annotations

import math
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small


class MaskedBatchNorm2d(nn.Module):
    """
    BatchNorm2d có mặt nạ (Masked BatchNorm2d):
      - Khi training và có mask: chỉ tính mean/var và cập nhật running stats trên các vị trí hợp lệ (mask > 0.5),
        triệt tiêu hoàn toàn nhiễu từ các khung hình đệm (padding).
      - Khi eval hoặc không có mask: dùng running stats cố định và che lại đầu ra theo mask (bất biến batch_size & padding).
    """

    def __init__(
            self,
            num_features: int,
            eps: float = 1e-5,
            momentum: float = 0.01,
            affine: bool = True,
    ):
        super().__init__()
        self.bn = nn.BatchNorm2d(num_features, eps=eps, momentum=momentum, affine=affine)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if (not self.training) or (mask is None) or (not self.bn.track_running_stats):
            out = self.bn(x)
            if mask is not None:
                out = out * mask.to(dtype=out.dtype)
            return out

        B, C, H, W = x.shape
        dev_type = x.device.type if x.device.type in ("cuda", "cpu") else "cpu"
        with torch.amp.autocast(dev_type, enabled=False):
            x_32 = x.float()
            m_broad = (mask > 0.5).float().expand(B, 1, H, W)
            valid_count = m_broad.sum()

            if valid_count.item() < 2.0:
                out_32 = F.batch_norm(
                    x_32,
                    self.bn.running_mean,
                    self.bn.running_var,
                    self.bn.weight.float() if self.bn.affine else None,
                    self.bn.bias.float() if self.bn.affine else None,
                    training=False,
                    momentum=0.0,
                    eps=self.bn.eps,
                ) * m_broad
            else:
                mean = (x_32 * m_broad).sum(dim=(0, 2, 3)) / valid_count
                mean_4d = mean.view(1, C, 1, 1)
                var = (((x_32 - mean_4d) * m_broad).pow(2)).sum(dim=(0, 2, 3)) / valid_count

                with torch.no_grad():
                    mom = float(self.bn.momentum) if self.bn.momentum is not None else 0.1
                    self.bn.num_batches_tracked.add_(1)
                    var_unbiased = var * (valid_count / (valid_count - 1.0))
                    self.bn.running_mean.mul_(1.0 - mom).add_(mean.detach() * mom)
                    self.bn.running_var.mul_(1.0 - mom).add_(var_unbiased.detach() * mom)

                inv_std = torch.rsqrt(var.view(1, C, 1, 1) + self.bn.eps)
                out_32 = (x_32 - mean_4d) * inv_std
                if self.bn.affine:
                    out_32 = (
                        out_32 * self.bn.weight.float().view(1, C, 1, 1)
                        + self.bn.bias.float().view(1, C, 1, 1)
                    )
                out_32 = out_32 * m_broad

        return out_32.to(x.dtype)


class KinematicFeatureExtractor(nn.Module):
    """
    Khối tiền xử lý chuẩn hóa động học 68 điểm mốc khuôn mặt (LiteFat v5):
    1. Chuẩn hóa hình thái đa cơ quan độc lập (Organ-Centered Local Coordinates):
       - Mắt & Lông mày: Quy chiếu về tâm khóe mắt, bảo toàn 100% độ mở mắt tĩnh/động (EAR).
       - Miệng: Quy chiếu về tâm khóe miệng, bảo toàn 100% độ mở miệng tĩnh/động (MAR).
       - Hàm & Mũi: Khử cấu trúc tĩnh (pos_dyn + 0.25*pos_norm) để ghi nhận tư thế đầu mà không học thuộc lòng xương sọ.
    2. Ổn định IOD tránh nổ thương số khi quay đầu sang ngang (Head Yaw).
    3. Đạo hàm vận tốc d(nose)/iod và gia tốc đúng chuẩn giải tích, loại bỏ triệt để gai giả ở t=1.
    4. Cân bằng năng lượng động học từng clip (Instance Kinematic Normalization) trước BatchNorm2d(6)
       để đồng bộ biên độ chuyển động giữa các bộ dữ liệu khác biệt FPS và độ rung cabin (UL-DD, SUST, VBDDD).
    """

    def __init__(
            self,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-2,
            static_shape_ratio: float = 0.20,
            fps: float = 10.0,
            num_nodes: int = 68,
    ):
        super().__init__()
        self.nose_idx = nose_idx
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps
        self.static_shape_ratio = float(static_shape_ratio)
        self.fps = float(max(fps, 1.0))
        self.num_nodes = int(num_nodes)

        # Chuẩn hóa phân phối riêng cho 6 kênh [x, y, dx, dy, d2x, d2y] chỉ tính trên frame hợp lệ
        self.channel_norm = MaskedBatchNorm2d(6, momentum=0.01, affine=True)

    @staticmethod
    def build_temporal_mask(
            x: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
            eps: float = 1e-6,
    ) -> torch.Tensor:
        """
        Tạo binary float mask kích thước (B, 1, T, 1) biểu diễn các frame hợp lệ (1.0) và frame đệm (0.0).
        """
        B = x.size(0)
        T = x.size(2) if x.dim() == 4 else x.size(1)
        device = x.device
        if lengths is not None:
            t_idx = torch.arange(T, device=device).unsqueeze(0)  # (1, T)
            mask = (t_idx < lengths.to(device).unsqueeze(1)).float()  # (B, T)
            return mask.view(B, 1, T, 1)
        else:
            frame_activity = x[:, :2].abs().sum(dim=(1, 3), keepdim=True) if x.dim() == 4 and x.size(1) == 2 else x.abs().sum(dim=(-2, -1), keepdim=True)
            return (frame_activity > eps).float().view(B, 1, T, 1)

    def forward(
            self,
            x: torch.Tensor,
            mask: Optional[torch.Tensor] = None,
            kin_state: Optional[Dict[str, torch.Tensor]] = None,
            return_state: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Input : (B, 2, T, V=68) - Tọa độ [0, 1] trong hệ tọa độ khung ảnh.
        mask  : (B, 1, T, 1)    - Float mask (1.0 = valid, 0.0 = padded/missing).
        kin_state: Trạng thái chuẩn hóa nhân quả truyền từ cửa sổ trước (dùng cho streaming/mobile).
        Output: (B, 6, T, V=68) - [x_norm, y_norm, dx_norm, dy_norm, d2x_norm, d2y_norm].
        """
        if x.size(1) != 2:
            out_passthru = x * mask if mask is not None else x
            if return_state:
                return out_passthru, (kin_state if kin_state is not None else {})
            return out_passthru

        B, _, T, V = x.shape

        # 1. Khoảng cách giải phẫu 2 mắt (Inter-Ocular Distance - IOD giữa điểm 36 và 45)
        p_left_eye = x[:, :, :, self.left_eye_idx: self.left_eye_idx + 1]    # (B, 2, T, 1)
        p_right_eye = x[:, :, :, self.right_eye_idx: self.right_eye_idx + 1]  # (B, 2, T, 1)
        raw_iod = torch.norm(p_right_eye - p_left_eye, p=2, dim=1, keepdim=True).clamp(min=self.eps)  # (B, 1, T, 1)

        # Ổn định IOD nhân quả: lưu baseline_iod từ cửa sổ đầu và cập nhật EMA chậm theo luồng streaming
        if mask is not None:
            v_mask = mask.clamp(min=0.0, max=1.0)
            v_sum = v_mask.sum(dim=2, keepdim=True)
            has_valid = v_sum > 0.5
            cur_clip_iod = (raw_iod * v_mask).sum(dim=2, keepdim=True) / v_sum.clamp(min=1.0)
        else:
            has_valid = torch.ones((B, 1, 1, 1), dtype=torch.bool, device=x.device)
            cur_clip_iod = raw_iod.mean(dim=2, keepdim=True)
        cur_clip_iod = cur_clip_iod.clamp(min=self.eps)

        if kin_state is not None and "baseline_iod" in kin_state:
            ref_iod = kin_state["baseline_iod"].to(device=x.device, dtype=raw_iod.dtype).clamp(min=self.eps)
            next_iod = torch.where(has_valid, 0.90 * ref_iod + 0.10 * cur_clip_iod, ref_iod)
        else:
            ref_iod = cur_clip_iod
            next_iod = cur_clip_iod

        iod_scale = torch.maximum(0.5 * raw_iod + 0.5 * ref_iod, 0.65 * ref_iod).clamp(min=self.eps)

        # 2. Tọa độ chuẩn hóa theo Tâm Mũi (pos_norm) dùng cho biến dạng động lực học
        nose_pos = x[:, :, :, self.nose_idx: self.nose_idx + 1]  # (B, 2, T, 1)
        pos_norm = (x - nose_pos) / iod_scale                    # (B, 2, T, V)
        pos_norm = torch.clamp(pos_norm, min=-8.0, max=8.0)

        # 3. Chuẩn hóa hình thái đa cơ quan độc lập & tỷ lệ nội cơ quan (Intra-Organ Aspect Ratio):
        c_left_eye = 0.5 * (x[:, :, :, 36:37] + x[:, :, :, 39:40])
        c_right_eye = 0.5 * (x[:, :, :, 42:43] + x[:, :, :, 45:46])
        c_mouth = 0.5 * (x[:, :, :, 48:49] + x[:, :, :, 54:55])

        w_left_eye = torch.norm(x[:, :, :, 39:40] - x[:, :, :, 36:37], p=2, dim=1, keepdim=True)
        w_right_eye = torch.norm(x[:, :, :, 45:46] - x[:, :, :, 42:43], p=2, dim=1, keepdim=True)
        w_mouth = torch.norm(x[:, :, :, 54:55] - x[:, :, :, 48:49], p=2, dim=1, keepdim=True)

        scale_left_eye = torch.maximum(w_left_eye, 0.22 * iod_scale).clamp(min=self.eps)
        scale_right_eye = torch.maximum(w_right_eye, 0.22 * iod_scale).clamp(min=self.eps)
        scale_mouth = torch.maximum(w_mouth, 0.32 * iod_scale).clamp(min=self.eps)

        pos_feat = pos_norm.clone()
        # Lông mày trái & phải (quy chiếu về tâm khóe mắt theo iod_scale)
        pos_feat[:, :, :, 17:22] = (x[:, :, :, 17:22] - c_left_eye) / iod_scale
        pos_feat[:, :, :, 22:27] = (x[:, :, :, 22:27] - c_right_eye) / iod_scale

        # Mắt trái (36..41): trục ngang theo iod_scale, trục dọc theo bề rộng hốc mắt (bảo toàn EAR)
        rel_le = x[:, :, :, 36:42] - c_left_eye
        pos_feat[:, 0:1, :, 36:42] = rel_le[:, 0:1] / iod_scale
        pos_feat[:, 1:2, :, 36:42] = rel_le[:, 1:2] / (scale_left_eye * 0.35)

        # Mắt phải (42..47): trục ngang theo iod_scale, trục dọc theo bề rộng hốc mắt (bảo toàn EAR)
        rel_re = x[:, :, :, 42:48] - c_right_eye
        pos_feat[:, 0:1, :, 42:48] = rel_re[:, 0:1] / iod_scale
        pos_feat[:, 1:2, :, 42:48] = rel_re[:, 1:2] / (scale_right_eye * 0.35)

        # Miệng (48..67): trục ngang theo iod_scale, trục dọc theo bề rộng miệng (bảo toàn MAR)
        rel_mo = x[:, :, :, 48:68] - c_mouth
        pos_feat[:, 0:1, :, 48:68] = rel_mo[:, 0:1] / iod_scale
        pos_feat[:, 1:2, :, 48:68] = rel_mo[:, 1:2] / (scale_mouth * 0.45)

        # Baseline nhân quả cho cơ quan (mắt, lông mày, miệng) và khung đầu (hàm, mũi)
        organ_indices = list(range(17, 27)) + list(range(36, 68))
        pos_org = pos_feat[:, :, :, organ_indices]
        if mask is not None:
            denom_org = mask.sum(dim=2, keepdim=True).clamp(min=1.0)
            cur_org_mean = (pos_org * mask).sum(dim=2, keepdim=True) / denom_org
        else:
            cur_org_mean = pos_org.mean(dim=2, keepdim=True)

        jaw_nose_indices = list(range(0, 17)) + list(range(27, 36))
        pos_jn = pos_norm[:, :, :, jaw_nose_indices]
        if mask is not None:
            denom_jn = mask.sum(dim=2, keepdim=True).clamp(min=1.0)
            cur_jn_mean = (pos_jn * mask).sum(dim=2, keepdim=True) / denom_jn
        else:
            cur_jn_mean = pos_jn.mean(dim=2, keepdim=True)

        if kin_state is not None and "baseline_org_mean" in kin_state and "baseline_jn_mean" in kin_state:
            ref_org_mean = kin_state["baseline_org_mean"].to(device=x.device, dtype=pos_org.dtype)
            ref_jn_mean = kin_state["baseline_jn_mean"].to(device=x.device, dtype=pos_jn.dtype)
            next_org_mean = torch.where(has_valid, 0.90 * ref_org_mean + 0.10 * cur_org_mean, ref_org_mean)
            next_jn_mean = torch.where(has_valid, 0.90 * ref_jn_mean + 0.10 * cur_jn_mean, ref_jn_mean)
        else:
            ref_org_mean = cur_org_mean
            ref_jn_mean = cur_jn_mean
            next_org_mean = cur_org_mean
            next_jn_mean = cur_jn_mean

        if self.static_shape_ratio < 0.999:
            pos_feat[:, :, :, organ_indices] = (pos_org - ref_org_mean) + self.static_shape_ratio * ref_org_mean

        pos_feat[:, :, :, jaw_nose_indices] = (pos_jn - ref_jn_mean) + 0.20 * pos_jn
        # Khuếch đại nhẹ vùng mắt (36..47) và miệng (48..67)
        pos_feat[:, :, :, 36:48] = pos_feat[:, :, :, 36:48] * 1.8
        pos_feat[:, :, :, 48:68] = pos_feat[:, :, :, 48:68] * 1.4
        pos_feat = torch.clamp(pos_feat, min=-8.0, max=8.0)

        # 4. Tính Vận tốc (dx, dy) và Gia tốc (d2x, d2y) chuẩn hóa giải tích
        if T <= 1:
            vel_norm = torch.zeros_like(pos_norm)
            acc_norm = torch.zeros_like(pos_norm)
        else:
            # Vận tốc biến dạng cục bộ giữa các khung hình liên tiếp
            vel_local = torch.diff(pos_norm, n=1, dim=2)  # (B, 2, T-1, V)

            # Vận tốc tịnh tiến toàn đầu tại đỉnh tâm mũi (node 30):
            # Tính d(nose)/iod thay vì d(nose/iod) để triệt tiêu hoàn toàn lỗi nổ vận tốc do dao động mẫu số
            delta_nose = torch.diff(nose_pos, n=1, dim=2)  # (B, 2, T-1, 1)
            iod_mid = 0.5 * (iod_scale[:, :, 1:, :] + iod_scale[:, :, :-1, :])
            nose_vel = delta_nose / iod_mid.clamp(min=self.eps)
            nose_vel = torch.clamp(nose_vel, min=-4.0, max=4.0)
            vel_local = vel_local.clone()
            vel_local[:, :, :, self.nose_idx: self.nose_idx + 1] = nose_vel

            if mask is not None:
                pair_valid = mask[:, :, 1:, :] * mask[:, :, :-1, :]
                vel_local = vel_local * pair_valid

            # Gia tốc: Đạo hàm bậc 2 trên chuỗi vận tốc trước khi chèn số 0 (loại bỏ gai giả ở t=1)
            if T >= 3:
                acc_raw = torch.diff(vel_local, n=1, dim=2)  # (B, 2, T-2, V)
                if mask is not None:
                    triplet_valid = mask[:, :, 2:, :] * mask[:, :, 1:-1, :] * mask[:, :, :-2, :]
                    acc_raw = acc_raw * triplet_valid
                acc_raw = torch.clamp(acc_raw, min=-4.0, max=4.0)
                acc_norm = F.pad(acc_raw, (0, 0, 2, 0), mode="constant", value=0.0)  # (B, 2, T, V)
            else:
                acc_norm = torch.zeros_like(pos_norm)

            vel_norm = F.pad(vel_local, (0, 0, 1, 0), mode="constant", value=0.0)  # (B, 2, T, V)
            vel_norm = torch.clamp(vel_norm, min=-4.0, max=4.0)

        # 5. Chuẩn hóa năng lượng động học bảo toàn biên độ sinh lý (Soft-Ceiling Kinematic Normalization)
        # Không chia nén ngược vận tốc chớp mắt/gật đầu bình thường; chỉ kẹp mềm khi có giật cực đoan
        if mask is not None:
            m_valid = mask.clamp(min=0.0, max=1.0)
            n_valid = m_valid.sum(dim=(2, 3), keepdim=True).clamp(min=1.0) * float(V)
            vel_rms = torch.sqrt((vel_norm.pow(2) * m_valid).sum(dim=(2, 3), keepdim=True) / n_valid + 1e-6)
            acc_rms = torch.sqrt((acc_norm.pow(2) * m_valid).sum(dim=(2, 3), keepdim=True) / n_valid + 1e-6)
        else:
            vel_rms = torch.sqrt(vel_norm.pow(2).mean(dim=(2, 3), keepdim=True) + 1e-6)
            acc_rms = torch.sqrt(acc_norm.pow(2).mean(dim=(2, 3), keepdim=True) + 1e-6)

        fps_ratio = 20.0 / self.fps
        vel_ref = 0.012 * fps_ratio
        acc_ref = 0.010 * fps_ratio
        vel_ceil = (vel_rms / (0.060 * fps_ratio)).clamp(min=1.0)
        acc_ceil = (acc_rms / (0.050 * fps_ratio)).clamp(min=1.0)
        vel_norm = (vel_norm / (vel_ref * vel_ceil)) * 0.35
        acc_norm = (acc_norm / (acc_ref * acc_ceil)) * 0.35

        out = torch.cat([pos_feat, vel_norm, acc_norm], dim=1)  # (B, 6, T, V)
        out = torch.nan_to_num(out, nan=0.0, posinf=4.0, neginf=-4.0)

        # 6. Chuẩn hóa kênh MaskedBatchNorm2d(6) chỉ tính trên các khung hình hợp lệ
        out = self.channel_norm(out, mask=mask)
        if return_state:
            new_kin_state = {
                "baseline_iod": next_iod.detach(),
                "baseline_org_mean": next_org_mean.detach(),
                "baseline_jn_mean": next_jn_mean.detach(),
            }
            return out, new_kin_state
        return out


class MobileNetV3ContextEncoder(nn.Module):
    """
    Trích xuất vector ngữ cảnh thị giác d in R^{D} cho mỗi khung hình khuôn mặt
    bằng xương sống MobileNetV3-Small.
    """

    def __init__(
            self,
            out_dim: int = 64,
            proj_hidden_dim: int = 192,
            pretrained: bool = True,
            dropout: float = 0.2,
            freeze_backbone: bool = False,
    ):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        base_model = mobilenet_v3_small(weights=weights)

        self.features = base_model.features
        self.avgpool = base_model.avgpool

        self.projector = nn.Sequential(
            nn.Linear(576, proj_hidden_dim),
            nn.Hardswish(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(proj_hidden_dim, out_dim),
        )

        if freeze_backbone:
            self.freeze_backbone(freeze=True)

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        """Bật/tắt đóng băng tham số của xương sống MobileNetV3-Small."""
        for param in self.features.parameters():
            param.requires_grad = not freeze
        for m in self.features.modules():
            if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                m.eval() if freeze else m.train(self.training)
        if freeze_projector:
            for param in self.projector.parameters():
                param.requires_grad = not freeze
        elif not freeze:
            for param in self.projector.parameters():
                param.requires_grad = True

    def unfreeze_backbone(self, unfreeze_last_n_blocks: Optional[int] = None) -> None:
        """
        Mở khóa đóng băng MobileNetV3-Small:
        - unfreeze_last_n_blocks = None: mở khóa toàn bộ features.
        - unfreeze_last_n_blocks = K: chỉ mở khóa K block tích chập cuối của features.
        """
        if unfreeze_last_n_blocks is None:
            self.freeze_backbone(freeze=False, freeze_projector=False)
            return

        # Khóa toàn bộ trước
        self.freeze_backbone(freeze=True, freeze_projector=False)
        total_blocks = len(self.features)
        start_idx = max(0, total_blocks - unfreeze_last_n_blocks)
        for i in range(start_idx, total_blocks):
            for param in self.features[i].parameters():
                param.requires_grad = True

    @property
    def is_backbone_frozen(self) -> bool:
        """Kiểm tra xem tất cả các tham số của MobileNetV3 features có đang bị đóng băng hay không."""
        return not any(p.requires_grad for p in self.features.parameters())

    def train(self, mode: bool = True):
        super().train(mode)
        if mode and self.is_backbone_frozen:
            for m in self.features.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                    m.eval()
        return self

    def forward(self, frame_tensor: torch.Tensor) -> torch.Tensor:
        feat = self.features(frame_tensor)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        d = self.projector(feat)
        return d


class MultimodalFusionModule(nn.Module):
    """
    Dung hợp Đa phương thức giải quyết triệt để Rank-1 Bottleneck:
    1. Chiếu 6 kênh động học [x, y, dx, dy, d2x, d2y] lên không gian đặc trưng đa chiều D (Linear + LayerNorm + GELU).
    2. Cộng Node Positional Embedding [1, 1, 68, D] giúp phân biệt rõ vai trò giải phẫu của từng điểm mốc.
    3. Điều biến Gated FiLM modulation với vector ngữ cảnh thị giác d in R^{D}:
       H = H_kin * (1.0 + tanh(W_scale(d))) + W_shift(d)
    4. Tương thích hoàn hảo cả khi d=None (chế độ Fast Landmark-Only).
    """

    def __init__(
            self,
            num_features: int = 6,
            feature_dim: int = 64,
            num_nodes: int = 68,
            nose_idx: int = 30,
            left_eye_idx: int = 36,
            right_eye_idx: int = 45,
            eps: float = 1e-2,
            static_shape_ratio: float = 0.20,
            fps: float = 10.0,
    ):
        super().__init__()
        self.left_eye_idx = left_eye_idx
        self.right_eye_idx = right_eye_idx
        self.eps = eps
        self.feature_dim = feature_dim
        self.num_nodes = num_nodes

        # Khối tiền xử lý động học chuẩn hóa theo IOD
        self.kinematic = KinematicFeatureExtractor(
            nose_idx=nose_idx,
            left_eye_idx=left_eye_idx,
            right_eye_idx=right_eye_idx,
            eps=eps,
            static_shape_ratio=static_shape_ratio,
            fps=fps,
            num_nodes=num_nodes,
        )

        # Giữ lại w (đóng băng) để tương thích backward compatibility với checkpoint cũ
        self.w = nn.Parameter(torch.ones(num_features, 1) / float(num_features), requires_grad=False)

        # Phép chiếu động học đa kênh 6D -> D
        self.kin_proj = nn.Sequential(
            nn.Linear(num_features, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
        )

        # Vector nhúng vị trí đỉnh giải phẫu khả học cho 68 mốc mặt
        self.node_embed = nn.Parameter(torch.randn(1, 1, num_nodes, feature_dim) * 0.02)

        # Tầng điều biến FiLM từ vector ngữ cảnh thị giác d
        self.film_scale = nn.Linear(feature_dim, feature_dim)
        self.film_shift = nn.Linear(feature_dim, feature_dim)

    def reset_film_to_identity(self) -> None:
        """Khởi tạo zero-init cho FiLM scale và shift để bảo toàn 100% đặc trưng đồ thị tại thời điểm bắt đầu Stage 2."""
        nn.init.zeros_(self.film_scale.weight)
        nn.init.zeros_(self.film_scale.bias)
        nn.init.zeros_(self.film_shift.weight)
        nn.init.zeros_(self.film_shift.bias)

    def detect_missing_mask(self, coords: torch.Tensor) -> torch.Tensor:
        p_left = coords[:, :, self.left_eye_idx, :]
        p_right = coords[:, :, self.right_eye_idx, :]
        raw_iod = torch.norm(p_right - p_left, p=2, dim=-1, keepdim=True).unsqueeze(-1)

        all_ones = (coords >= 0.999).all(dim=-1, keepdim=True).all(dim=-2, keepdim=True)
        all_zeros = (coords <= 0.001).all(dim=-1, keepdim=True).all(dim=-2, keepdim=True)
        is_missing = (raw_iod < 1e-2) | all_ones | all_zeros
        return is_missing

    def forward(
            self,
            coords: torch.Tensor,
            d: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
            kin_state: Optional[Dict[str, torch.Tensor]] = None,
            return_kin_state: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, torch.Tensor]]]:
        """
        Args:
            coords: Chuỗi tọa độ mốc [B, T, N, 2] (hoặc [B, T, N, 3] từ cache cũ).
            d: Vector ngữ cảnh thị giác [B, T, D] hoặc None (Landmark-Only).
            mask: Temporal float mask [B, 1, T, 1].
            kin_state: Trạng thái baseline nhân quả truyền giữa các cửa sổ trượt.
            return_kin_state: True để trả kèm next_kin_state cho streaming/mobile.
        Returns:
            X: Tensor đặc trưng đồ thị [B, D, N, T] (hoặc tuple (X, next_kin_state)).
        """
        if coords.dim() == 4:
            if coords.size(-1) >= 2:
                coords_xy = coords[..., :2]
            elif coords.size(1) >= 2:
                coords_xy = coords[:, :2, :, :].permute(0, 2, 3, 1).contiguous()
            else:
                coords_xy = coords
        else:
            coords_xy = coords

        orig_dtype = coords.dtype
        dev = coords.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.autocast(device_type=dev_type, enabled=False):
            coords_32 = coords_xy.float()

            is_missing = self.detect_missing_mask(coords_32)
            valid_mask = (~is_missing).permute(0, 3, 1, 2).float()

            if mask is not None:
                temporal_mask = mask.float() * valid_mask
                is_missing = (temporal_mask == 0.0).permute(0, 2, 3, 1)
            else:
                temporal_mask = valid_mask

            x_in = coords_32.permute(0, 3, 1, 2).contiguous()  # [B, 2, T, N]
            feat_kin, next_kin_state = self.kinematic(
                x_in,
                mask=temporal_mask,
                kin_state=kin_state,
                return_state=True,
            )  # [B, 6, T, N]
            C_kin = feat_kin.permute(0, 2, 3, 1).contiguous()  # [B, T, N, 6]

            # Chiếu đa kênh 6D -> D và cộng nhúng vị trí giải phẫu
            H_kin = self.kin_proj(C_kin) + self.node_embed  # [B, T, N, D]

            # Điều biến FiLM nếu có bối cảnh thị giác d
            if d is not None:
                d_32 = d.float()
                scale = torch.tanh(self.film_scale(d_32)).unsqueeze(2)  # [B, T, 1, D]
                shift = self.film_shift(d_32).unsqueeze(2)  # [B, T, 1, D]
                H = H_kin * (1.0 + scale) + shift
            else:
                H = H_kin

            H = torch.nan_to_num(H, nan=0.0)
            # Giữ các khung hình đệm (padding) bằng 0 tuyệt đối trước khi vào STGL
            H = H * temporal_mask.permute(0, 2, 3, 1)

            # Đưa về chuẩn đồ thị STGL: [B, D, N, T]
            X = H.permute(0, 3, 2, 1).contiguous()

        out_x = X.to(orig_dtype)
        if return_kin_state:
            return out_x, next_kin_state
        return out_x


class SelfAdaptiveAdjacency(nn.Module):
    """
    Ma trận kề kết hợp Cấu trúc Giải phẫu Khuôn mặt (A_prior) và Tự thích nghi (A_adp):
        A_total = A_prior + alpha * softmax(ReLU(E1 @ E2^T / sqrt(d)))
    """

    def __init__(
            self,
            num_nodes: int = 68,
            embed_dim: int = 16,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.embed_dim = embed_dim

        # Xây dựng ma trận kề giải phẫu A_prior chuẩn hóa bậc đối xứng
        A_prior = self._build_anatomical_prior(num_nodes, vertical_pairs, cross_region_pairs)
        self.register_buffer("A_prior", A_prior)

        # Tham số học ma trận kề thích nghi
        self.E1 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(num_nodes, embed_dim) * 0.1)
        self.alpha = nn.Parameter(torch.tensor(0.5))

    @staticmethod
    def _build_anatomical_prior(
            num_nodes: int,
            vertical_pairs: Optional[List[Tuple[int, int]]],
            cross_region_pairs: Optional[List[Tuple[int, int]]],
    ) -> torch.Tensor:
        A = np.zeros((num_nodes, num_nodes), dtype=np.float32)

        # 1. Đường viền hàm dưới: 0..16
        for i in range(16):
            A[i, i + 1] = A[i + 1, i] = 1.0

        # 2. Lông mày trái (17..21) & phải (22..26)
        for i in range(17, 21):
            A[i, i + 1] = A[i + 1, i] = 1.0
        for i in range(22, 26):
            A[i, i + 1] = A[i + 1, i] = 1.0

        # 3. Sống mũi (27..30) & Cánh mũi (31..35)
        for i in range(27, 30):
            A[i, i + 1] = A[i + 1, i] = 1.0
        for i in range(31, 35):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[30, 33] = A[33, 30] = 1.0

        # 4. Mắt trái (36..41) & Mắt phải (42..47)
        for i in range(36, 41):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[41, 36] = A[36, 41] = 1.0
        for i in range(42, 47):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[47, 42] = A[42, 47] = 1.0

        # 5. Môi ngoài (48..59) & Môi trong (60..67)
        for i in range(48, 59):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[59, 48] = A[48, 59] = 1.0
        for i in range(60, 67):
            A[i, i + 1] = A[i + 1, i] = 1.0
        A[67, 60] = A[60, 67] = 1.0

        # 6. Các cặp dọc mở/nhắm mắt (EAR) & miệng (MAR)
        v_pairs = vertical_pairs or [
            (37, 41), (38, 40), (43, 47), (44, 46),
            (50, 58), (51, 57), (52, 56), (61, 67), (62, 66), (63, 65)
        ]
        for u, v in v_pairs:
            if u < num_nodes and v < num_nodes:
                A[u, v] = A[v, u] = 1.0

        # 7. Các cặp liên vùng khuôn mặt
        cr_pairs = cross_region_pairs or [
            (19, 37), (24, 44), (30, 39), (30, 42), (57, 8), (51, 33)
        ]
        for u, v in cr_pairs:
            if u < num_nodes and v < num_nodes:
                A[u, v] = A[v, u] = 1.0

        # Tự liên kết (Self-loops)
        A = A + np.eye(num_nodes, dtype=np.float32)

        # Chuẩn hóa bậc đối xứng: D^{-1/2} A D^{-1/2}
        degrees = np.sum(A, axis=1)
        d_inv_sqrt = np.power(np.maximum(degrees, 1e-8), -0.5)
        D_inv_sqrt = np.diag(d_inv_sqrt)
        A_norm = D_inv_sqrt @ A @ D_inv_sqrt

        return torch.from_numpy(A_norm).float()

    def forward(self) -> torch.Tensor:
        orig_dtype = self.E1.dtype
        scale = 1.0 / math.sqrt(self.embed_dim)
        E1_f32 = self.E1.float()
        E2_f32 = self.E2.float()
        spatial_weights = F.relu(torch.matmul(E1_f32, E2_f32.transpose(0, 1)) * scale)
        A_adp = F.softmax(spatial_weights, dim=-1)
        # Giới hạn alpha qua tanh để ma trận kề thích nghi không tăng trưởng vô hạn
        alpha_bounded = torch.tanh(self.alpha.float())
        A_total = self.A_prior.float() + alpha_bounded * A_adp
        return A_total.to(orig_dtype)


class SpatialGCNLayer(nn.Module):
    """
    Tầng Tích chập Đồ thị Không gian (GCN) không làm triệt tiêu gradient:
    Thay thế F.softmax bằng F.gelu(LayerNorm) kèm kết nối phần dư.
    """

    def __init__(
            self,
            in_dim: int = 64,
            hidden_dim: int = 64,
            out_dim: int = 64,
            dropout: float = 0.2,
    ):
        super().__init__()
        self.W0 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W1 = nn.Linear(hidden_dim, out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(p=dropout)
        self.res_proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x: torch.Tensor, A_adp: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        device_type = x.device.type
        # Bảo vệ số học: chạy phép nhân đồ thị và LayerNorm dưới float32 để chống tràn số float16 (max 65,504)
        with torch.amp.autocast(device_type, enabled=False):
            x_perm = x.float().permute(0, 3, 2, 1).contiguous()  # [B, T, N, D]
            A_f32 = A_adp.float()

            h1 = self.W0(x_perm)  # [B, T, N, H]
            # Nhân đồ thị chuẩn xác: đỉnh i tổng hợp đặc trưng từ các đỉnh láng giềng j theo A_adp[i, j]
            # torch.matmul(A_f32, h1) tự động broadcast qua [B, T], chạy bằng cuBLAS GEMM nhanh gấp 3-4 lần einsum
            h1 = torch.matmul(A_f32, h1)
            h1 = F.relu(h1)
            h1 = self.dropout(h1)

            h2 = self.W1(h1)  # [B, T, N, M]
            h2 = torch.matmul(A_f32, h2)
            z = F.gelu(self.norm(h2)) + self.res_proj(x_perm)

        return z.to(orig_dtype).permute(0, 3, 2, 1).contiguous()


class GatedDilatedCausalTCN(nn.Module):
    """
    Tầng Tích chập Thời gian Nhân quả Giãn nở có Cổng (Gated Dilated Causal TCN):
    h = tanh(Theta_1 * X) (x) sigmoid(Theta_2 * X).
    """

    def __init__(
            self,
            in_channels: int = 64,
            dilation_channels: int = 64,
            kernel_size: int = 2,
            dilation: int = 1,
    ):
        super().__init__()
        self.causal_padding = (kernel_size - 1) * dilation
        self.theta_1 = nn.Conv2d(
            in_channels,
            dilation_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            bias=True,
        )
        self.theta_2 = nn.Conv2d(
            in_channels,
            dilation_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            bias=True,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_pad = F.pad(x, (self.causal_padding, 0, 0, 0))
        tcn_a = torch.tanh(self.theta_1(x_pad))
        tcn_b = torch.sigmoid(self.theta_2(x_pad))
        return tcn_a * tcn_b


class STGLBlock(nn.Module):
    """
    Khối Không gian - Thời gian STGL Block kết hợp Gated TCN và Spatial GCN:
    Đường tắt skip_conv được tính SAU spatial_gcn để dòng gradient chảy xuyên suốt qua tầng đồ thị.
    """

    def __init__(
            self,
            residual_channels: int = 64,
            dilation_channels: int = 64,
            skip_channels: int = 128,
            kernel_size: int = 2,
            dilation: int = 1,
            dropout: float = 0.2,
            use_residual_out: bool = True,
    ):
        super().__init__()
        self.use_residual_out = bool(use_residual_out)
        self.gated_tcn = GatedDilatedCausalTCN(
            in_channels=residual_channels,
            dilation_channels=dilation_channels,
            kernel_size=kernel_size,
            dilation=dilation,
        )
        self.spatial_gcn = SpatialGCNLayer(
            in_dim=dilation_channels,
            hidden_dim=residual_channels,
            out_dim=residual_channels,
            dropout=dropout,
        )
        # Skip connection lấy từ h_gcn để truyền gradient trực tiếp qua GCN
        self.skip_conv = nn.Conv2d(residual_channels, skip_channels, kernel_size=(1, 1))
        self.bn = MaskedBatchNorm2d(residual_channels, momentum=0.01)
        self.res_scale = nn.Parameter(torch.ones(1))
        if not self.use_residual_out:
            self.bn.bn.weight.requires_grad_(False)
            self.bn.bn.bias.requires_grad_(False)
            self.res_scale.requires_grad_(False)

    def forward(
            self,
            x: torch.Tensor,
            A_adp: torch.Tensor,
            t_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        residual = x
        h_tcn = self.gated_tcn(x)
        if t_mask is not None:
            h_tcn = h_tcn * t_mask.to(dtype=h_tcn.dtype)
        h_gcn = self.spatial_gcn(h_tcn, A_adp)
        if t_mask is not None:
            h_gcn = h_gcn * t_mask.to(dtype=h_gcn.dtype)
        skip = self.skip_conv(h_gcn)
        if self.use_residual_out:
            x_res = self.bn(self.res_scale * h_gcn + residual, mask=t_mask)
        else:
            x_res = residual
        if t_mask is not None:
            skip = skip * t_mask.to(dtype=skip.dtype)
        return x_res, skip


class CrossWindowFeatureAccumulator(nn.Module):
    """
    Khối Tích lũy Đặc trưng Kép qua các Cửa sổ Trượt (Dual-Level Forward Feature Accumulation):
      1. Overlap Frame Gated Fusion: Dung hợp đặc trưng khung hình ở vùng chồng lấp (mặc định 30s = 300 frames)
         giữa cuối cửa sổ trước (w-1) và đầu cửa sổ hiện tại (w).
      2. Cross-Window Gated GRU Memory: Cập nhật trạng thái tích lũy cấp cửa sổ xuyên suốt chiều dài mẫu (>60s)
         kèm bảo vệ độ dài thực tế (active_mask) để padding không làm trôi trí nhớ mệt mỏi.
      3. Causal Window Attention: Tổng hợp chuỗi trạng thái các cửa sổ lịch sử thành biểu diễn toàn mẫu.
    """

    def __init__(
            self,
            feature_dim: int = 256,
            dropout: float = 0.2,
    ):
        super().__init__()
        self.feature_dim = feature_dim

        # 1. Nhánh dung hợp đặc trưng khung hình vùng chồng lấp 30s
        self.overlap_norm_cur = nn.LayerNorm(feature_dim)
        self.overlap_norm_prev = nn.LayerNorm(feature_dim)
        self.overlap_gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, feature_dim),
            nn.Sigmoid(),
        )

        # 2. Nhánh bộ nhớ trạng thái có cổng cấp cửa sổ (Gated GRU Memory)
        self.input_norm = nn.LayerNorm(feature_dim)
        self.state_norm = nn.LayerNorm(feature_dim)
        self.gru_cell = nn.GRUCell(feature_dim, feature_dim)
        self.mem_gate = nn.Sequential(
            nn.Linear(feature_dim * 3, feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim, feature_dim),
            nn.Sigmoid(),
        )
        self.out_norm = nn.LayerNorm(feature_dim)

        # 3. Nhánh tổng hợp chuỗi trạng thái các cửa sổ lịch sử
        self.window_attn_gate = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def fuse_overlap_frames(
            self,
            cur_frame_feat: torch.Tensor,
            prev_tail_feat: Optional[torch.Tensor],
            cur_lengths: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Dung hợp vùng khung hình chồng lấp giữa đầu cửa sổ hiện tại [B, T_w, D]
        và vùng tail đã tích lũy từ cửa sổ trước [B, T_ov, D].
        """
        if prev_tail_feat is None or prev_tail_feat.size(1) == 0 or cur_frame_feat.size(1) == 0:
            return cur_frame_feat

        L_ov = min(cur_frame_feat.size(1), prev_tail_feat.size(1))
        cur_ov = cur_frame_feat[:, :L_ov, :]
        prev_ov = prev_tail_feat[:, :L_ov, :]

        orig_dtype = cur_frame_feat.dtype
        dev = cur_frame_feat.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.amp.autocast(dev_type, enabled=False):
            cur_ov_32 = cur_ov.float()
            prev_ov_32 = prev_ov.float()
            gate_in = torch.cat([self.overlap_norm_cur(cur_ov_32), self.overlap_norm_prev(prev_ov_32)], dim=-1)
            gate = self.overlap_gate(gate_in)
            fused_ov = gate * cur_ov_32 + (1.0 - gate) * prev_ov_32

            if cur_lengths is not None:
                valid_ov = (
                    torch.arange(L_ov, device=dev).unsqueeze(0)
                    < cur_lengths.to(dev).unsqueeze(1)
                ).unsqueeze(-1).float()
                fused_ov = fused_ov * valid_ov + cur_ov_32 * (1.0 - valid_ov)

        fused_ov = fused_ov.to(orig_dtype)
        if L_ov < cur_frame_feat.size(1):
            return torch.cat([fused_ov, cur_frame_feat[:, L_ov:, :]], dim=1)
        return fused_ov

    def update_window_memory(
            self,
            window_feat: torch.Tensor,
            prev_memory: Optional[torch.Tensor] = None,
            active_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Cập nhật trạng thái tích lũy cấp cửa sổ h_k = GatedGRU(z_k, h_{k-1}).
        Nếu mẫu trong batch có active_mask == False (toàn bộ frame ở cửa sổ này là padding),
        giữ nguyên trạng thái prev_memory.
        """
        orig_dtype = window_feat.dtype
        dev = window_feat.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.amp.autocast(dev_type, enabled=False):
            w_32 = window_feat.float()
            if prev_memory is None:
                h_new = self.out_norm(w_32)
            else:
                p_32 = prev_memory.float()
                z = self.input_norm(w_32)
                h_prev = self.state_norm(p_32)
                h_gru = self.gru_cell(z, h_prev)
                alpha = self.mem_gate(torch.cat([z, h_prev, h_gru], dim=-1))
                h_new = self.out_norm(alpha * h_gru + (1.0 - alpha) * w_32)
                if active_mask is not None:
                    mask_col = active_mask.view(-1, 1).bool().to(dev)
                    h_new = torch.where(mask_col, h_new, p_32)

        return h_new.to(orig_dtype)

    def aggregate_window_states(
            self,
            window_states: torch.Tensor,
            window_valid_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Tổng hợp chuỗi trạng thái [B, W_num, D] qua các cửa sổ trượt thành vector đặc trưng cuối [B, D].
        Kết hợp trạng thái tích lũy cuối cùng hợp lệ (65%) và Causal Window Attention (35%).
        """
        B, W_num, _ = window_states.shape
        if W_num == 1:
            return window_states[:, 0, :]

        orig_dtype = window_states.dtype
        dev = window_states.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.amp.autocast(dev_type, enabled=False):
            ws_32 = window_states.float()
            if window_valid_mask is not None:
                valid_bool = window_valid_mask.bool().to(dev)
                last_idx = valid_bool.long().sum(dim=1).clamp(min=1) - 1
                batch_idx = torch.arange(B, device=dev)
                last_state = ws_32[batch_idx, last_idx, :]
            else:
                valid_bool = torch.ones((B, W_num), dtype=torch.bool, device=dev)
                last_state = ws_32[:, -1, :]

            valid_w_counts = valid_bool.float().sum(dim=1, keepdim=True).clamp(min=1.0)
            w_idx = torch.arange(W_num, device=dev, dtype=ws_32.dtype).unsqueeze(0)
            recency_bias = torch.where(
                valid_w_counts > 1.0,
                -0.5 + (w_idx / (valid_w_counts - 1.0).clamp(min=1.0)),
                torch.zeros((B, W_num), device=dev, dtype=ws_32.dtype),
            )
            scores = self.window_attn_gate(ws_32).squeeze(-1) + recency_bias
            scores = scores.masked_fill(~valid_bool, -1e4)
            attn_w = F.softmax(scores, dim=1) * valid_bool.float()
            attn_w = attn_w / attn_w.sum(dim=1, keepdim=True).clamp(min=1e-6)
            attn_state = (ws_32 * attn_w.unsqueeze(-1)).sum(dim=1)

            final_state = 0.65 * last_state + 0.35 * attn_state

        return final_state.to(orig_dtype)


class HybridKSSHead(nn.Module):
    """
    Đầu dự đoán điểm KSS (Karolinska Sleepiness Scale, thang 1-9) theo cơ chế Lai ghép:
      - Kết hợp Kỳ vọng phân phối 9 mức KSS [1..9] với Nhánh hồi quy liên tục có chặn [1.0, 9.0].
      - Trả về đồng thời điểm KSS liên tục và logits 9 lớp ở cả cấp khung hình và cấp đoạn/cửa sổ.
    """

    def __init__(
            self,
            in_channels: int = 256,
            num_classes: int = 9,
            kss_min: float = 1.0,
            kss_max: float = 9.0,
            dropout: float = 0.3,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.kss_min = float(kss_min)
        self.kss_max = float(kss_max)

        kss_levels = torch.linspace(self.kss_min, self.kss_max, steps=num_classes, dtype=torch.float32)
        self.register_buffer("kss_values", kss_levels)

        # Cấp khung hình (Frame-level)
        self.frame_cls_conv = nn.Conv2d(in_channels, num_classes, kernel_size=(1, 1))
        self.frame_reg_head = nn.Sequential(
            nn.Linear(in_channels, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

        # Cấp cửa sổ / toàn mẫu (Clip-level)
        self.clip_classifier = nn.Sequential(
            nn.Linear(in_channels, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )
        self.clip_reg_head = nn.Sequential(
            nn.Linear(in_channels, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
        self.kss_mix_logit = nn.Parameter(torch.tensor(0.5))

    def forward_frames(
            self,
            pooled_node_feat: torch.Tensor,
            frame_feat: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            pooled_node_feat: [B, D, T]
            frame_feat: [B, T, D]
        Returns:
            frame_kss: [B, T] trong đoạn [kss_min, kss_max]
            frame_logits: [B, T, num_classes]
        """
        orig_dtype = frame_feat.dtype
        dev = frame_feat.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        frame_logits = self.frame_cls_conv(pooled_node_feat.unsqueeze(2)).squeeze(2).permute(0, 2, 1).contiguous()
        with torch.amp.autocast(dev_type, enabled=False):
            logits_32 = frame_logits.float()
            probs = F.softmax(logits_32, dim=-1)
            kss_vals = self.kss_values.view(1, 1, -1).to(dev)
            kss_exp = (probs * kss_vals).sum(dim=-1)
            offset = 0.5 * torch.tanh(self.frame_reg_head(frame_feat.float()).squeeze(-1))
            frame_kss = torch.clamp(kss_exp + offset, min=self.kss_min, max=self.kss_max)

        return frame_kss.to(orig_dtype), frame_logits

    def forward_clip(
            self,
            clip_feat: torch.Tensor,
            attn_frame_logits: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            clip_feat: [B, D]
            attn_frame_logits: [B, num_classes] (tùy chọn)
        Returns:
            clip_kss: [B] trong đoạn [kss_min, kss_max]
            clip_logits: [B, num_classes]
        """
        orig_dtype = clip_feat.dtype
        dev = clip_feat.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        head_logits = self.clip_classifier(clip_feat)
        if attn_frame_logits is not None:
            clip_logits = 0.65 * head_logits + 0.35 * attn_frame_logits
        else:
            clip_logits = head_logits

        with torch.amp.autocast(dev_type, enabled=False):
            logits_32 = clip_logits.float()
            probs = F.softmax(logits_32, dim=-1)
            kss_vals = self.kss_values.view(1, -1).to(dev)
            kss_exp = (probs * kss_vals).sum(dim=-1)

            span = self.kss_max - self.kss_min
            kss_reg = self.kss_min + span * torch.sigmoid(self.clip_reg_head(clip_feat.float()).squeeze(-1))
            mix_w = torch.sigmoid(self.kss_mix_logit.float())
            clip_kss = torch.clamp(mix_w * kss_exp + (1.0 - mix_w) * kss_reg, min=self.kss_min, max=self.kss_max)

        return clip_kss.to(orig_dtype), clip_logits


class GradientReversalFn(torch.autograd.Function):
    """Lớp đảo ngược gradient (Gradient Reversal Layer) cho Domain-Adversarial Training."""

    @staticmethod
    def forward(ctx, x: torch.Tensor, alpha: float) -> torch.Tensor:
        ctx.alpha = float(alpha)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.alpha * grad_output, None


class GradientReversalLayer(nn.Module):
    """Module bọc GradientReversalFn hỗ trợ điều chỉnh hệ số lambda/alpha."""

    def __init__(self, alpha: float = 1.0):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, x: torch.Tensor, alpha: Optional[float] = None) -> torch.Tensor:
        eff_alpha = self.alpha if alpha is None else float(alpha)
        return GradientReversalFn.apply(x, eff_alpha)


class RegimeConditionedDomainDiscriminator(nn.Module):
    """
    Bộ phân biệt miền có điều kiện theo 3 vùng trạng thái KSS (Alert / Mid / Drowsy):
      - Chỉ kích hoạt trong lúc huấn luyện (zero overhead khi suy luận trên mobile).
      - Ghép đặc trưng cửa sổ/clip sau GRL với phân phối 3 vùng p_regime (detached)
        để tránh co cụm các mức KSS trung gian khi phân phối nhãn giữa các domain lệch nhau.
    """

    def __init__(
            self,
            in_dim: int = 256,
            num_regimes: int = 3,
            hidden_dim: int = 64,
            num_domains: int = 2,
            dropout: float = 0.2,
    ):
        super().__init__()
        self.in_dim = int(in_dim)
        self.num_regimes = int(num_regimes)
        self.num_domains = int(num_domains)
        self.grl = GradientReversalLayer(alpha=0.0)
        self.net = nn.Sequential(
            nn.LayerNorm(self.in_dim + self.num_regimes),
            nn.Linear(self.in_dim + self.num_regimes, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, self.num_domains),
        )

    @staticmethod
    def logits_to_regime_probs(logits: torch.Tensor) -> torch.Tensor:
        """Quy đổi logits 9 mức KSS [1..9] sang xác suất 3 vùng: Alert (1-4), Mid (5-6), Drowsy (7-9)."""
        probs = F.softmax(logits.detach().float(), dim=-1)
        p_alert = probs[..., :4].sum(dim=-1, keepdim=True)
        p_mid = probs[..., 4:6].sum(dim=-1, keepdim=True)
        p_drowsy = probs[..., 6:].sum(dim=-1, keepdim=True)
        return torch.cat([p_alert, p_mid, p_drowsy], dim=-1)

    def forward(
            self,
            feat: torch.Tensor,
            kss_logits: torch.Tensor,
            alpha: Optional[float] = None,
    ) -> torch.Tensor:
        dev = feat.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"
        with torch.amp.autocast(dev_type, enabled=False):
            feat_rev = self.grl(feat.float(), alpha=alpha)
            regime_probs = self.logits_to_regime_probs(kss_logits).to(dev)
            cond_in = torch.cat([feat_rev, regime_probs], dim=-1)
            domain_logits = self.net(cond_in)
        return domain_logits


class LiteFatV1(nn.Module):
    """
    Mô hình LiteFat_v1 Dự đoán Điểm Buồn ngủ KSS (thang 1-9):
      - Cửa sổ mặc định 60s ở 10 FPS (600 frames) và bước trượt mặc định 30s (300 frames).
      - Hỗ trợ tự động chạy Sliding Window với tích lũy đặc trưng kép (Overlap Frame Fusion +
        Cross-Window Gated GRU Memory) cho các mẫu có độ dài lớn hơn 60s.
      - Hỗ trợ linh hoạt cả 2 chế độ: Đa phương thức (Multimodal) và Chỉ dùng Landmark (Landmark-Only).
      - Đầu ra Hybrid KSS: hỗ trợ đồng thời điểm KSS liên tục [1.0, 9.0] và logits 9 mức KSS.
    """

    def __init__(
            self,
            num_nodes: int = 68,
            feature_dim: int = 64,
            mobilenet_proj_hidden: int = 192,
            node_embed_dim: int = 16,
            residual_channels: int = 64,
            dilation_channels: int = 64,
            skip_channels: int = 128,
            end_channels: int = 256,
            num_classes: int = 9,
            kss_min: float = 1.0,
            kss_max: float = 9.0,
            fps: int = 10,
            window_sec: float = 60.0,
            stride_sec: float = 30.0,
            num_blocks: int = 3,
            layers_per_block: int = 4,
            kernel_size: int = 3,
            pretrained_backbone: bool = True,
            dropout: float = 0.3,
            freeze_backbone: bool = False,
            vertical_pairs: Optional[List[Tuple[int, int]]] = None,
            cross_region_pairs: Optional[List[Tuple[int, int]]] = None,
            static_shape_ratio: float = 0.20,
            use_dann: bool = False,
            num_domains: int = 2,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.feature_dim = feature_dim
        self.end_channels = end_channels
        self.num_classes = num_classes
        self.kss_min = float(kss_min)
        self.kss_max = float(kss_max)
        self.fps = int(max(1, fps))
        self.window_sec = float(window_sec)
        self.stride_sec = float(stride_sec)
        self.window_size = max(1, int(round(self.window_sec * self.fps)))
        self.stride_size = max(1, int(round(self.stride_sec * self.fps)))
        self.overlap_size = max(0, self.window_size - self.stride_size)
        self.num_blocks = num_blocks
        self.layers_per_block = layers_per_block
        self.kernel_size = kernel_size
        self.static_shape_ratio = float(static_shape_ratio)
        self.use_dann = bool(use_dann)
        self.num_domains = int(max(2, num_domains))
        self.dann_alpha = 0.0

        # 1. MobileNetV3-Small trích xuất ngữ cảnh thị giác từ ảnh mặt
        self.context_encoder = MobileNetV3ContextEncoder(
            out_dim=feature_dim,
            proj_hidden_dim=mobilenet_proj_hidden,
            pretrained=pretrained_backbone,
            dropout=dropout,
            freeze_backbone=freeze_backbone,
        )

        # 2. Dung hợp đa phương thức FiLM chống Rank-1 Bottleneck (chuẩn hóa động học theo FPS)
        self.fusion = MultimodalFusionModule(
            num_features=6,
            feature_dim=feature_dim,
            num_nodes=num_nodes,
            static_shape_ratio=static_shape_ratio,
            fps=float(self.fps),
        )

        # 3. Ma trận kề giải phẫu + tự thích nghi
        self.adaptive_adj = SelfAdaptiveAdjacency(
            num_nodes=num_nodes,
            embed_dim=node_embed_dim,
            vertical_pairs=vertical_pairs,
            cross_region_pairs=cross_region_pairs,
        )

        # Tầng chiếu đầu vào đồ thị
        self.start_conv = nn.Conv2d(feature_dim, residual_channels, kernel_size=(1, 1))

        # 4. Các khối Không gian - Thời gian (STGL Blocks) mở rộng dilation 1, 2, 4, 8 cho cửa sổ 60s
        self.stgl_blocks = nn.ModuleList()
        total_stgl_layers = num_blocks * layers_per_block
        layer_global_idx = 0
        for _ in range(num_blocks):
            dilation = 1
            for _ in range(layers_per_block):
                self.stgl_blocks.append(
                    STGLBlock(
                        residual_channels=residual_channels,
                        dilation_channels=dilation_channels,
                        skip_channels=skip_channels,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        dropout=dropout,
                        use_residual_out=(layer_global_idx < total_stgl_layers - 1),
                    )
                )
                dilation *= 2
                layer_global_idx += 1

        # Chiếu đặc trưng đầu ra đồ thị
        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))

        # Attentive Node Pooling: Tự học trọng số chú ý trên đặc trưng 256D của 68 đỉnh khuôn mặt
        self.node_gate = nn.Sequential(
            nn.Conv2d(end_channels, 64, kernel_size=(1, 1)),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=(1, 1)),
        )
        # Prior giải phẫu ưu tiên vùng mắt (36..47: 12 đỉnh) hơn vùng miệng (48..67: 20 đỉnh)
        node_prior = torch.zeros(1, 1, num_nodes, 1)
        node_prior[:, :, 36:48, :] = 1.50  # Mắt
        node_prior[:, :, 48:68, :] = 0.75  # Miệng
        self.register_buffer("node_prior_bias", node_prior)

        # Attentive Temporal Pooling: Tự học trọng số chú ý từ vector đặc trưng khung hình (end_channels)
        self.temp_norm = nn.LayerNorm(end_channels)
        self.temporal_gate = nn.Sequential(
            nn.Linear(end_channels, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.temp_scale = nn.Parameter(torch.tensor(2.0))

        # 5. Khối tích lũy đặc trưng kép xuyên suốt các cửa sổ trượt 60s (stride 30s)
        self.accumulator = CrossWindowFeatureAccumulator(
            feature_dim=end_channels,
            dropout=dropout,
        )

        # 6. Đầu dự đoán KSS Lai ghép (Hybrid 9-level expectation + bounded regression [1.0, 9.0])
        self.kss_head = HybridKSSHead(
            in_channels=end_channels,
            num_classes=num_classes,
            kss_min=self.kss_min,
            kss_max=self.kss_max,
            dropout=dropout,
        )

        # 7. Nhánh Domain-Adversarial Training có điều kiện theo 3 vùng trạng thái (chỉ chạy khi train)
        self.domain_discriminator = RegimeConditionedDomainDiscriminator(
            in_dim=end_channels,
            num_regimes=3,
            hidden_dim=64,
            num_domains=self.num_domains,
            dropout=min(0.3, dropout),
        )
        self.set_dann_enabled(self.use_dann)

    @property
    def end_conv_2(self) -> nn.Conv2d:
        """Alias tương thích ngược trỏ tới tầng phân loại frame của kss_head."""
        return self.kss_head.frame_cls_conv

    @property
    def clip_classifier(self) -> nn.Sequential:
        """Alias tương thích ngược trỏ tới bộ phân loại clip của kss_head."""
        return self.kss_head.clip_classifier

    @staticmethod
    def _build_sliding_window_ranges(
            total_frames: int,
            window_size: int,
            stride_size: int,
            min_tail_frames: int = 50,
    ) -> List[Tuple[int, int]]:
        """
        Tạo danh sách các khoảng cửa sổ (start, end) cố định kích thước `window_size`
        (tương thích cơ chế Ring Buffer trên điện thoại di động) với bước trượt `stride_size`.
        """
        if total_frames <= window_size:
            return [(0, total_frames)]

        starts = list(range(0, max(1, total_frames - window_size + 1), stride_size))
        if not starts:
            starts = [0]

        rem = total_frames - (starts[-1] + window_size)
        if rem > 0:
            last_start = total_frames - window_size
            if rem >= int(max(1, min_tail_frames)):
                if last_start > starts[-1]:
                    starts.append(last_start)
            else:
                if len(starts) == 1 or last_start > starts[-2]:
                    starts[-1] = last_start

        return [(s, s + window_size) for s in starts]

    @staticmethod
    def _normalize_landmarks_shape(landmarks: torch.Tensor) -> torch.Tensor:
        """Chuẩn hóa tensor landmarks về dạng [B, T, N, C]."""
        if landmarks.dim() == 4:
            if landmarks.size(-1) in (2, 3):
                return landmarks
            if landmarks.size(1) in (2, 3):
                return landmarks.permute(0, 2, 3, 1).contiguous()
        return landmarks

    def _encode_visual_context(
            self,
            video_frames: torch.Tensor,
            target_length: int,
            max_chunk: int = 64,
    ) -> torch.Tensor:
        """Trích xuất vector ngữ cảnh thị giác theo từng chunk nhỏ và nội suy về target_length."""
        B, T_v, C_img, H, W = video_frames.shape
        frames_flat = video_frames.reshape(B * T_v, C_img, H, W)
        total_imgs = frames_flat.size(0)

        if total_imgs <= max_chunk:
            d_flat = self.context_encoder(frames_flat)
        else:
            chunks = [self.context_encoder(frames_flat[i:i + max_chunk]) for i in range(0, total_imgs, max_chunk)]
            d_flat = torch.cat(chunks, dim=0)

        d_sparse = d_flat.view(B, T_v, self.feature_dim)
        if T_v != target_length:
            return F.interpolate(
                d_sparse.permute(0, 2, 1),
                size=target_length,
                mode="linear",
                align_corners=False,
            ).permute(0, 2, 1)
        return d_sparse

    def _extract_window_frame_features(
            self,
            X: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Chạy đồ thị STGL và Attentive Spatial Node Pooling cho 1 cửa sổ."""
        A_adp = self.adaptive_adj()
        h = self.start_conv(X)

        B, _, _, T = X.shape
        t_mask = None
        if lengths is not None:
            t_idx = torch.arange(T, device=X.device).unsqueeze(0)  # [1, T]
            t_mask = (t_idx < lengths.to(X.device).unsqueeze(1)).float().view(B, 1, 1, T)

        skip_sum = None
        for block in self.stgl_blocks:
            h, skip = block(h, A_adp, t_mask=t_mask)
            skip_sum = skip if skip_sum is None else (skip_sum + skip)

        out = F.relu(skip_sum)
        h_end = F.relu(self.end_conv_1(out))  # [B, end_channels, N, T]

        # Attentive Spatial Pooling trên N=68 đỉnh dựa trên đặc trưng 256D và prior giải phẫu
        node_attn = self.node_gate(h_end) + self.node_prior_bias  # [B, 1, N, T]
        gate = F.softmax(node_attn, dim=2)  # [B, 1, N, T]

        pooled_node_feat = (h_end * gate).sum(dim=2)  # [B, end_channels, T]
        frame_feat = pooled_node_feat.permute(0, 2, 1).contiguous()  # [B, T, end_channels]
        return pooled_node_feat, frame_feat

    def _pool_window_temporal_features(
            self,
            frame_feat: torch.Tensor,
            frame_logits: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Gom tụ đặc trưng thời gian đa nhánh (Attn + Top-K + Mean) cho 1 cửa sổ."""
        B, T, _ = frame_feat.shape
        feat_norm = self.temp_norm(frame_feat)
        temp_scores = self.temporal_gate(feat_norm) * self.temp_scale  # [B, T, 1]

        if lengths is not None:
            valid_bool = (
                torch.arange(T, device=frame_logits.device).unsqueeze(0)
                < lengths.to(frame_logits.device).unsqueeze(1)
            ).unsqueeze(-1)
            valid_mask = valid_bool.type_as(frame_logits)
            denom = valid_mask.sum(dim=1).clamp(min=1.0)

            masked_scores = temp_scores.masked_fill(~valid_bool, -1e4)
            temp_weights = F.softmax(masked_scores, dim=1) * valid_mask
            temp_weights = temp_weights / temp_weights.sum(dim=1, keepdim=True).clamp(min=1e-6)

            feat_mean = (frame_feat * valid_mask).sum(dim=1) / denom
            feat_attn = (frame_feat * temp_weights).sum(dim=1)

            scores_2d = masked_scores.squeeze(-1)  # [B, T]
            k_per = (lengths.float() * 0.25).round().clamp(min=1.0).long()  # [B]
            ranks = scores_2d.argsort(dim=1, descending=True).argsort(dim=1)  # [B, T]
            topk_mask = (ranks < k_per.unsqueeze(1)) & valid_bool.squeeze(-1)  # [B, T]
            feat_topk = (
                (frame_feat * topk_mask.unsqueeze(-1).type_as(frame_feat)).sum(dim=1)
                / topk_mask.sum(dim=1, keepdim=True).clamp(min=1.0).type_as(frame_feat)
            )

            pooled_window_feat = 0.50 * feat_attn + 0.30 * feat_topk + 0.20 * feat_mean
            attn_frame_logits = (frame_logits * temp_weights).sum(dim=1)

            active_col = (lengths > 0).to(frame_logits.device).view(B, 1)
            pooled_window_feat = pooled_window_feat * active_col.type_as(pooled_window_feat)
            attn_frame_logits = attn_frame_logits * active_col.type_as(attn_frame_logits)
        else:
            temp_weights = F.softmax(temp_scores, dim=1)
            feat_mean = frame_feat.mean(dim=1)
            feat_attn = (frame_feat * temp_weights).sum(dim=1)

            k_val = max(1, int(round(T * 0.25)))
            scores_2d = temp_scores.squeeze(-1)
            topk_idx = torch.topk(scores_2d, k=k_val, dim=1).indices
            batch_indices = torch.arange(B, device=frame_feat.device).unsqueeze(1).expand(-1, k_val)
            feat_topk = frame_feat[batch_indices, topk_idx, :].mean(dim=1)

            pooled_window_feat = 0.50 * feat_attn + 0.30 * feat_topk + 0.20 * feat_mean
            attn_frame_logits = (frame_logits * temp_weights).sum(dim=1)

        return pooled_window_feat, attn_frame_logits, temp_weights

    def _extract_overlap_tail(
            self,
            frame_feat: torch.Tensor,
            lengths: Optional[torch.Tensor],
            stride_size: int,
            overlap_size: int,
            prev_tail: Optional[torch.Tensor] = None,
    ) -> Optional[torch.Tensor]:
        """Trích xuất vùng đặc trưng khung hình chồng lấp (từ stride_size trở đi) cho cửa sổ kế tiếp."""
        if overlap_size <= 0:
            return None

        B, T_w, D = frame_feat.shape
        dev = frame_feat.device

        if T_w > stride_size:
            raw_tail = frame_feat[:, stride_size:min(T_w, stride_size + overlap_size), :]
            cur_len = raw_tail.size(1)
            if cur_len < overlap_size:
                overlap_tail = F.pad(raw_tail, (0, 0, 0, overlap_size - cur_len))
            else:
                overlap_tail = raw_tail

            if lengths is not None:
                valid_tail_len = (lengths.to(dev) - stride_size).clamp(min=0, max=overlap_size)
                tail_mask = (
                    torch.arange(overlap_size, device=dev).unsqueeze(0)
                    < valid_tail_len.unsqueeze(1)
                ).unsqueeze(-1).type_as(overlap_tail)
                overlap_tail = overlap_tail * tail_mask
        else:
            overlap_tail = frame_feat.new_zeros((B, overlap_size, D))

        if prev_tail is not None and lengths is not None and prev_tail.size(1) == overlap_size:
            active_col = (lengths.to(dev) > 0).view(B, 1, 1)
            overlap_tail = torch.where(active_col, overlap_tail, prev_tail.to(overlap_tail.dtype))

        return overlap_tail

    def forward_stgl_from_fused(
            self,
            X: torch.Tensor,
            lengths: Optional[torch.Tensor] = None,
            prev_state: Optional[Dict[str, torch.Tensor]] = None,
            overlap_size: Optional[int] = None,
            stride_size: Optional[int] = None,
            kin_state: Optional[Dict[str, torch.Tensor]] = None,
            return_details: bool = False,
    ) -> Union[Tuple[torch.Tensor, torch.Tensor], Dict[str, Any]]:
        """
        Chạy phần đồ thị STGL từ tensor đặc trưng X [B, D, N, T_w] cho 1 cửa sổ,
        hỗ trợ tích lũy kép từ prev_state (vùng chồng lấp khung hình + bộ nhớ Gated GRU).
        """
        eff_stride = self.stride_size if stride_size is None else int(max(1, stride_size))
        eff_overlap = self.overlap_size if overlap_size is None else int(max(0, overlap_size))

        pooled_node_feat, frame_feat = self._extract_window_frame_features(X, lengths=lengths)

        prev_tail = prev_state.get("overlap_tail_feat") if prev_state is not None else None
        if prev_tail is not None:
            frame_feat = self.accumulator.fuse_overlap_frames(
                frame_feat,
                prev_tail,
                cur_lengths=lengths,
            )
            pooled_node_feat = frame_feat.permute(0, 2, 1).contiguous()

        frame_kss, frame_logits = self.kss_head.forward_frames(pooled_node_feat, frame_feat)
        pooled_window_feat, attn_frame_logits, _ = self._pool_window_temporal_features(
            frame_feat,
            frame_logits,
            lengths=lengths,
        )

        prev_mem = prev_state.get("memory_feat") if prev_state is not None else None
        active_mask = (lengths > 0) if lengths is not None else None
        memory_feat = self.accumulator.update_window_memory(
            pooled_window_feat,
            prev_memory=prev_mem,
            active_mask=active_mask,
        )

        clip_kss, clip_logits = self.kss_head.forward_clip(
            memory_feat,
            attn_frame_logits=attn_frame_logits,
        )

        win_domain_logits = None
        if self.use_dann and self.training:
            win_domain_logits = self.domain_discriminator(
                memory_feat,
                clip_logits,
                alpha=self.dann_alpha,
            )

        if return_details:
            overlap_tail_feat = self._extract_overlap_tail(
                frame_feat=frame_feat,
                lengths=lengths,
                stride_size=eff_stride,
                overlap_size=eff_overlap,
                prev_tail=prev_tail,
            )
            new_state: Dict[str, Any] = {
                "memory_feat": memory_feat,
                "overlap_tail_feat": overlap_tail_feat,
                "window_feat": pooled_window_feat,
            }
            if kin_state is not None:
                new_state["kin_state"] = kin_state
            elif prev_state is not None and "kin_state" in prev_state:
                new_state["kin_state"] = prev_state["kin_state"]
            return {
                "frame_kss": frame_kss,
                "clip_kss": clip_kss,
                "frame_logits": frame_logits,
                "clip_logits": clip_logits,
                "attn_frame_logits": attn_frame_logits,
                "domain_logits": win_domain_logits,
                "state": new_state,
            }

        return frame_kss, clip_kss

    def forward_sliding_windows(
            self,
            landmarks: torch.Tensor,
            video_frames: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
            window_size: Optional[int] = None,
            stride_size: Optional[int] = None,
            prev_state: Optional[Dict[str, torch.Tensor]] = None,
    ) -> Dict[str, Any]:
        """
        Chạy cơ chế cửa sổ trượt (mặc định 60s = 600 frames, stride 30s = 300 frames @ 10 FPS)
        với tích lũy đặc trưng kép phía trước và baseline động học nhân quả (streaming-safe).
        """
        lm_norm = self._normalize_landmarks_shape(landmarks)
        B, T, _, _ = lm_norm.shape
        dev = lm_norm.device

        win_size = self.window_size if window_size is None else int(max(1, window_size))
        str_size = self.stride_size if stride_size is None else int(max(1, stride_size))

        d_full = None
        if video_frames is not None and video_frames.numel() > 0:
            d_full = self._encode_visual_context(video_frames, target_length=T)

        if mask is None and lengths is not None:
            mask = KinematicFeatureExtractor.build_temporal_mask(
                lm_norm.permute(0, 3, 1, 2),
                lengths=lengths,
            )

        ranges = self._build_sliding_window_ranges(T, win_size, str_size)

        full_frame_kss = lm_norm.new_full((B, T), self.kss_min, dtype=torch.float32)
        full_frame_logits = lm_norm.new_zeros((B, T, self.num_classes), dtype=torch.float32)
        written_mask = torch.zeros((B, T), dtype=torch.bool, device=dev)

        state = prev_state
        kin_state = prev_state.get("kin_state") if prev_state is not None else None
        window_memories: List[torch.Tensor] = []
        window_kss_list: List[torch.Tensor] = []
        window_logits_list: List[torch.Tensor] = []
        window_attn_logits_list: List[torch.Tensor] = []
        window_valid_list: List[torch.Tensor] = []
        window_domain_logits_list: List[torch.Tensor] = []

        for w_idx, (s, e_valid) in enumerate(ranges):
            sub_len_span = e_valid - s
            sub_landmarks = lm_norm[:, s:e_valid]
            sub_d = d_full[:, s:e_valid] if d_full is not None else None

            if lengths is not None:
                sub_lengths = (lengths.to(dev) - s).clamp(min=0, max=sub_len_span)
                active_mask = sub_lengths > 0
            else:
                sub_lengths = None
                active_mask = torch.ones((B,), dtype=torch.bool, device=dev)

            if mask is not None:
                sub_mask = mask[:, :, s:e_valid, :]
            elif sub_lengths is not None:
                sub_mask = KinematicFeatureExtractor.build_temporal_mask(
                    sub_landmarks.permute(0, 3, 1, 2),
                    lengths=sub_lengths,
                )
            else:
                sub_mask = None

            X_sub, next_kin_state = self.fusion(
                sub_landmarks,
                d=sub_d,
                mask=sub_mask,
                kin_state=kin_state,
                return_kin_state=True,
            )
            kin_state = next_kin_state

            if w_idx + 1 < len(ranges):
                next_s = ranges[w_idx + 1][0]
                tail_stride = max(1, next_s - s)
                tail_overlap = max(0, e_valid - next_s)
            else:
                tail_stride = str_size
                tail_overlap = max(0, win_size - str_size)

            win_out = self.forward_stgl_from_fused(
                X_sub,
                lengths=sub_lengths,
                prev_state=state,
                overlap_size=tail_overlap,
                stride_size=tail_stride,
                kin_state=kin_state,
                return_details=True,
            )
            state = win_out["state"]

            w_frame_kss = win_out["frame_kss"].float()
            w_frame_logits = win_out["frame_logits"].float()

            if sub_lengths is not None:
                local_valid = (
                    torch.arange(sub_len_span, device=dev).unsqueeze(0)
                    < sub_lengths.unsqueeze(1)
                )
            else:
                local_valid = torch.ones((B, sub_len_span), dtype=torch.bool, device=dev)

            if w_idx == 0:
                if s > 0:
                    full_frame_kss[:, :s] = w_frame_kss[:, :1]
                    full_frame_logits[:, :s, :] = w_frame_logits[:, :1, :]
                    written_mask[:, :s] = True
                full_frame_kss[:, s:e_valid] = w_frame_kss
                full_frame_logits[:, s:e_valid, :] = w_frame_logits
                written_mask[:, s:e_valid] = True
            else:
                update_mask = local_valid | (~written_mask[:, s:e_valid])
                full_frame_kss[:, s:e_valid] = torch.where(
                    update_mask,
                    w_frame_kss,
                    full_frame_kss[:, s:e_valid],
                )
                full_frame_logits[:, s:e_valid, :] = torch.where(
                    update_mask.unsqueeze(-1),
                    w_frame_logits,
                    full_frame_logits[:, s:e_valid, :],
                )
                written_mask[:, s:e_valid] = True

            window_memories.append(state["memory_feat"])
            window_kss_list.append(win_out["clip_kss"])
            window_logits_list.append(win_out["clip_logits"])
            window_attn_logits_list.append(win_out["attn_frame_logits"])
            window_valid_list.append(active_mask)
            if win_out.get("domain_logits") is not None:
                window_domain_logits_list.append(win_out["domain_logits"])

        window_states = torch.stack(window_memories, dim=1)  # [B, W_num, D]
        window_kss = torch.stack(window_kss_list, dim=1)  # [B, W_num]
        window_logits = torch.stack(window_logits_list, dim=1)  # [B, W_num, num_classes]
        window_attn_logits = torch.stack(window_attn_logits_list, dim=1)  # [B, W_num, num_classes]
        window_valid_mask = torch.stack(window_valid_list, dim=1)  # [B, W_num]

        final_feat = self.accumulator.aggregate_window_states(
            window_states,
            window_valid_mask=window_valid_mask,
        )
        valid_w_float = window_valid_mask.float().unsqueeze(-1)
        final_attn_logits = (
            (window_attn_logits.float() * valid_w_float).sum(dim=1)
            / valid_w_float.sum(dim=1).clamp(min=1.0)
        ).to(final_feat.dtype)

        clip_kss, clip_logits = self.kss_head.forward_clip(
            final_feat,
            attn_frame_logits=final_attn_logits,
        )

        clip_domain_logits = None
        window_domain_logits = None
        if self.use_dann and self.training:
            clip_domain_logits = self.domain_discriminator(
                final_feat,
                clip_logits,
                alpha=self.dann_alpha,
            )
            if window_domain_logits_list:
                window_domain_logits = torch.stack(window_domain_logits_list, dim=1)

        orig_dtype = lm_norm.dtype
        return {
            "frame_kss": full_frame_kss.to(orig_dtype),
            "clip_kss": clip_kss,
            "frame_logits": full_frame_logits.to(orig_dtype),
            "clip_logits": clip_logits,
            "window_kss": window_kss,
            "window_logits": window_logits,
            "window_valid_mask": window_valid_mask,
            "domain_logits": clip_domain_logits,
            "window_domain_logits": window_domain_logits,
            "state": state,
        }

    def forward(
            self,
            video_frames: Optional[torch.Tensor] = None,
            landmarks: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
            return_clip_logits: bool = True,
            return_logits: bool = False,
            return_dict: bool = False,
            use_sliding_window: Optional[bool] = None,
            window_sec: Optional[float] = None,
            stride_sec: Optional[float] = None,
            prev_state: Optional[Dict[str, torch.Tensor]] = None,
            return_state: bool = False,
    ) -> Union[torch.Tensor, Tuple[Any, ...], Dict[str, Any]]:
        """
        Dự đoán điểm buồn ngủ KSS (thang 1-9) với cửa sổ mặc định 60s @ 10 FPS (600 frames)
        và tự động chạy Sliding Window (stride 30s = 300 frames) tích lũy đặc trưng phía trước
        khi chuỗi đầu vào dài hơn 60s.

        Args:
            video_frames: Tensor ảnh khuôn mặt [B, T_v, C, H, W] hoặc None (Landmark-Only).
            landmarks: Tensor mốc mặt [B, T, N, 2] (hoặc [B, 2, T, N]).
            lengths: Độ dài thực tế từng mẫu trong batch [B].
            mask: Temporal float mask [B, 1, T, 1].
            return_clip_logits: True để trả kèm đầu ra cấp đoạn/mẫu; False để chỉ trả cấp frame.
            return_logits: True để trả thêm frame_logits và clip_logits 9 mức KSS.
            return_dict: True để trả toàn bộ kết quả dưới dạng Dict đầy đủ.
            use_sliding_window: None (tự động bật khi T > window_size), True (ép chạy sliding),
                                False (ép chạy 1 cửa sổ).
            window_sec: Độ dài cửa sổ tùy chỉnh (giây); mặc định dùng self.window_sec (60.0s).
            stride_sec: Bước trượt cửa sổ tùy chỉnh (giây); mặc định dùng self.stride_sec (30.0s).
            prev_state: Trạng thái tích lũy từ cửa sổ/đoạn video trước.
            return_state: True để trả kèm trạng thái tích lũy `state` ở cuối tuple.
        """
        if landmarks is None:
            raise ValueError("Tham số 'landmarks' [B, T, N, 2] bắt buộc phải được cung cấp!")

        lm_norm = self._normalize_landmarks_shape(landmarks)
        T_target = lm_norm.shape[1]

        win_frames = self.window_size if window_sec is None else max(1, int(round(float(window_sec) * self.fps)))
        str_frames = self.stride_size if stride_sec is None else max(1, int(round(float(stride_sec) * self.fps)))
        ov_frames = max(0, win_frames - str_frames)

        run_sliding = (T_target > win_frames) if use_sliding_window is None else bool(use_sliding_window)

        if run_sliding:
            out_dict = self.forward_sliding_windows(
                landmarks=lm_norm,
                video_frames=video_frames,
                lengths=lengths,
                mask=mask,
                window_size=win_frames,
                stride_size=str_frames,
                prev_state=prev_state,
            )
        else:
            d = None
            if video_frames is not None and video_frames.numel() > 0:
                d = self._encode_visual_context(video_frames, target_length=T_target)

            if mask is None and lengths is not None:
                mask = KinematicFeatureExtractor.build_temporal_mask(
                    lm_norm.permute(0, 3, 1, 2),
                    lengths=lengths,
                )

            in_kin_state = prev_state.get("kin_state") if prev_state is not None else None
            X, next_kin_state = self.fusion(
                lm_norm,
                d=d,
                mask=mask,
                kin_state=in_kin_state,
                return_kin_state=True,
            )
            win_out = self.forward_stgl_from_fused(
                X,
                lengths=lengths,
                prev_state=prev_state,
                overlap_size=ov_frames,
                stride_size=str_frames,
                kin_state=next_kin_state,
                return_details=True,
            )
            out_dict = {
                "frame_kss": win_out["frame_kss"],
                "clip_kss": win_out["clip_kss"],
                "frame_logits": win_out["frame_logits"],
                "clip_logits": win_out["clip_logits"],
                "window_kss": win_out["clip_kss"].unsqueeze(1),
                "window_logits": win_out["clip_logits"].unsqueeze(1),
                "window_valid_mask": (
                    (lengths.to(lm_norm.device) > 0).unsqueeze(1)
                    if lengths is not None
                    else torch.ones((lm_norm.size(0), 1), dtype=torch.bool, device=lm_norm.device)
                ),
                "domain_logits": win_out.get("domain_logits"),
                "window_domain_logits": (
                    win_out["domain_logits"].unsqueeze(1)
                    if win_out.get("domain_logits") is not None
                    else None
                ),
                "state": win_out["state"],
            }

        if return_dict:
            return out_dict

        frame_kss = out_dict["frame_kss"]
        clip_kss = out_dict["clip_kss"]
        frame_logits = out_dict["frame_logits"]
        clip_logits = out_dict["clip_logits"]
        state = out_dict["state"]

        if return_logits:
            if return_clip_logits:
                base_out: Tuple[Any, ...] = (frame_kss, clip_kss, frame_logits, clip_logits)
            else:
                base_out = (frame_kss, frame_logits)
            return (*base_out, state) if return_state else base_out

        if return_clip_logits:
            base_out = (frame_kss, clip_kss)
            return (*base_out, state) if return_state else base_out

        return (frame_kss, state) if return_state else frame_kss

    @torch.no_grad()
    def forward_window_step(
            self,
            landmarks: torch.Tensor,
            video_frames: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
            mask: Optional[torch.Tensor] = None,
            prev_state: Optional[Dict[str, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, Any]]:
        """
        Suy luận cuốn chiếu (streaming) cho từng cửa sổ 60s mới (dịch bước 30s so với cửa sổ trước),
        tích lũy đặc trưng phía trước từ prev_state.

        Returns:
            clip_kss: Điểm KSS dự đoán [B] trong khoảng [1.0, 9.0].
            clip_probs: Phân phối xác suất 9 mức KSS [B, num_classes].
            new_state: Trạng thái tích lũy để truyền vào bước kế tiếp.
        """
        out = self.forward(
            video_frames=video_frames,
            landmarks=landmarks,
            lengths=lengths,
            mask=mask,
            return_dict=True,
            use_sliding_window=False,
            prev_state=prev_state,
        )
        clip_probs = F.softmax(out["clip_logits"].float(), dim=-1)
        return out["clip_kss"], clip_probs, out["state"]

    @torch.no_grad()
    def forward_asymmetric_step(
            self,
            landmark_window: torch.Tensor,
            d_history: torch.Tensor,
            current_frame: Optional[torch.Tensor] = None,
            prev_state: Optional[Dict[str, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, Dict[str, Any]]:
        if current_frame is not None:
            d_new = self.context_encoder(current_frame).unsqueeze(1)
        else:
            d_new = d_history[:, -1:, :]

        d_updated = torch.cat([d_history[:, 1:, :], d_new], dim=1)
        X = self.fusion(landmark_window, d=d_updated)
        win_out = self.forward_stgl_from_fused(
            X,
            prev_state=prev_state,
            return_details=True,
        )
        probs = F.softmax(win_out["clip_logits"].float(), dim=-1)
        return win_out["clip_kss"], probs, d_updated, win_out["state"]

    def freeze_backbone(self, freeze: bool = True, freeze_projector: bool = False) -> None:
        self.context_encoder.freeze_backbone(freeze=freeze, freeze_projector=freeze_projector)

    def unfreeze_backbone(self, unfreeze_last_n_blocks: Optional[int] = None) -> None:
        self.context_encoder.unfreeze_backbone(unfreeze_last_n_blocks=unfreeze_last_n_blocks)

    def set_backbone_trainable(self, trainable: bool, unfreeze_last_n_blocks: Optional[int] = 3) -> None:
        """Đóng/mở khóa backbone MobileNetV3 theo epoch huấn luyện."""
        if not trainable:
            self.freeze_backbone(freeze=True, freeze_projector=False)
        else:
            self.unfreeze_backbone(unfreeze_last_n_blocks=unfreeze_last_n_blocks)

    def set_visual_context_enabled(self, enabled: bool) -> None:
        """Đóng băng hoặc mở khóa toàn bộ nhánh thị giác (MobileNetV3 + Projector + FiLM)."""
        flag = bool(enabled)
        if not flag:
            self.context_encoder.freeze_backbone(freeze=True, freeze_projector=True)
        else:
            for param in self.context_encoder.projector.parameters():
                param.requires_grad = True
        for param in self.fusion.film_scale.parameters():
            param.requires_grad = flag
        for param in self.fusion.film_shift.parameters():
            param.requires_grad = flag

    def set_dann_enabled(self, enabled: bool) -> None:
        """Bật/tắt nhánh phân biệt miền có điều kiện (Regime-Conditioned DANN)."""
        self.use_dann = bool(enabled)
        for param in self.domain_discriminator.parameters():
            param.requires_grad = self.use_dann

    def set_dann_alpha(self, alpha: float) -> None:
        """Cập nhật hệ số đảo dấu gradient GRL (lambda) theo lịch trình Ganin."""
        self.dann_alpha = float(max(0.0, alpha))

    @property
    def is_backbone_frozen(self) -> bool:
        return self.context_encoder.is_backbone_frozen

    def get_trainable_params_count(self) -> Dict[str, Any]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        backbone_total = sum(p.numel() for p in self.context_encoder.features.parameters())
        backbone_trainable = sum(p.numel() for p in self.context_encoder.features.parameters() if p.requires_grad)
        return {
            "total_params": total,
            "trainable_params": trainable,
            "frozen_params": total - trainable,
            "backbone_total": backbone_total,
            "backbone_trainable": backbone_trainable,
            "is_backbone_frozen": self.is_backbone_frozen,
        }


LiteFat_v1 = LiteFatV1

In [10]:
# =====================================================================
# File: loss.py
# Dự án: Driver Guardian - Module LiteFat_v1 (KSS 1-9, Sliding 60s/30s)
# Tác vụ: Định nghĩa hàm mục tiêu huấn luyện LiteFatLoss (Hybrid KSS Loss)
#         kết hợp:
#           1. Clip Regression Loss (SmoothL1 + MSE trên thang điểm 1..9)
#           2. Clip Ordinal Gaussian Soft-Label + CrossEntropy (logits 9 mức)
#           3. Causal Time-Weighted Top-K MIL Frame Loss thích nghi mức KSS
#           4. Sliding-Window Causal & Smoothness Loss cho chuỗi dài > 60s
# =====================================================================
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F


class LiteFatLoss(nn.Module):
    """
    Hàm mất mát Hybrid KSS Loss cho mô hình LiteFat_v1 (Karolinska Sleepiness Scale 1..9):
      L_total = clip_reg_weight * L_clip_reg
              + clip_cls_weight * L_clip_cls
              + frame_loss_weight * L_frame_mil
              + window_loss_weight * L_window
              + window_smooth_weight * L_smooth

    Hỗ trợ linh hoạt mọi kiểu đầu ra của LiteFat_v1.forward():
      - Dict trả về khi `return_dict=True`
      - Tuple trả về khi `return_dict=False` (`(frame_kss, clip_kss)` hoặc kèm logits/state)
      - Truyền trực tiếp từng tensor hoặc theo chữ ký cũ `(frame_logits, targets, clip_logits=..., lengths=...)`.
    """

    def __init__(
            self,
            num_classes: int = 9,
            kss_min: float = 1.0,
            kss_max: float = 9.0,
            clip_reg_weight: float = 1.0,
            clip_cls_weight: float = 0.5,
            frame_loss_weight: float = 0.3,
            window_loss_weight: float = 0.25,
            window_smooth_weight: float = 0.05,
            dann_weight: float = 0.05,
            huber_beta: float = 0.5,
            ordinal_sigma: float = 0.85,
            label_smoothing: float = 0.05,
            alert_kss_threshold: float = 4.0,
            mid_kss_threshold: float = 6.0,
            topk_mid_ratio: float = 0.70,
            topk_drowsy_ratio: float = 0.45,
            fps: int = 10,
            window_sec: float = 60.0,
            stride_sec: float = 30.0,
            smooth_margin: float = 1.0,
    ):
        super().__init__()
        self.num_classes = int(num_classes)
        self.kss_min = float(kss_min)
        self.kss_max = float(kss_max)

        self.clip_reg_weight = float(clip_reg_weight)
        self.clip_cls_weight = float(clip_cls_weight)
        self.frame_loss_weight = float(frame_loss_weight)
        self.window_loss_weight = float(window_loss_weight)
        self.window_smooth_weight = float(window_smooth_weight)
        self.dann_weight = float(dann_weight)

        self.huber_beta = float(huber_beta)
        self.ordinal_sigma = float(max(1e-3, ordinal_sigma))
        self.label_smoothing = float(label_smoothing)
        self.alert_kss_threshold = float(alert_kss_threshold)
        self.mid_kss_threshold = float(mid_kss_threshold)
        self.topk_mid_ratio = float(topk_mid_ratio)
        self.topk_drowsy_ratio = float(topk_drowsy_ratio)

        self.fps = int(max(1, fps))
        self.window_sec = float(window_sec)
        self.stride_sec = float(stride_sec)
        self.window_size = max(1, int(round(self.window_sec * self.fps)))
        self.stride_size = max(1, int(round(self.stride_sec * self.fps)))
        self.smooth_margin = float(smooth_margin)

        kss_levels = torch.linspace(self.kss_min, self.kss_max, steps=self.num_classes, dtype=torch.float32)
        self.register_buffer("kss_levels", kss_levels)
        self.ce_raw = nn.CrossEntropyLoss(reduction="none", label_smoothing=self.label_smoothing)

    def _logits_to_kss_expectation(self, logits: torch.Tensor) -> torch.Tensor:
        """Quy đổi logits 9 mức KSS thành điểm kỳ vọng liên tục trong [kss_min, kss_max]."""
        probs = F.softmax(logits.float(), dim=-1)
        view_shape = [1] * (logits.dim() - 1) + [self.num_classes]
        levels = self.kss_levels.view(*view_shape).to(logits.device)
        return (probs * levels).sum(dim=-1)

    def _ordinal_soft_cross_entropy(
            self,
            logits: torch.Tensor,
            target_kss: torch.Tensor,
    ) -> torch.Tensor:
        """
        Tính Ordinal Gaussian Soft-Label Cross-Entropy giữa phân phối logits [..., C]
        và mục tiêu liên tục target_kss [...] trên lưới mức KSS [1..9].
        """
        view_shape = [1] * (logits.dim() - 1) + [self.num_classes]
        levels = self.kss_levels.view(*view_shape).to(logits.device)
        dist_sq = (levels - target_kss.float().unsqueeze(-1)).pow(2)
        soft_targets = F.softmax(-dist_sq / (2.0 * (self.ordinal_sigma ** 2)), dim=-1)
        log_probs = F.log_softmax(logits.float(), dim=-1)
        return -(soft_targets * log_probs).sum(dim=-1)

    def _classification_ordinal_loss(
            self,
            logits: torch.Tensor,
            target_kss: torch.Tensor,
            target_cls: torch.Tensor,
    ) -> torch.Tensor:
        """Kết hợp 65% Ordinal Gaussian Soft-Label CE + 35% CrossEntropy chuẩn."""
        ord_ce = self._ordinal_soft_cross_entropy(logits, target_kss)
        flat_logits = logits.float().reshape(-1, self.num_classes)
        flat_cls = target_cls.reshape(-1)
        std_ce = self.ce_raw(flat_logits, flat_cls).reshape(target_cls.shape)
        return 0.65 * ord_ce + 0.35 * std_ce

    def _unpack_predictions(
            self,
            predictions: Union[torch.Tensor, Dict[str, Any], Tuple[Any, ...]],
            clip_kss: Optional[torch.Tensor],
            frame_logits: Optional[torch.Tensor],
            clip_logits: Optional[torch.Tensor],
            window_kss: Optional[torch.Tensor],
            window_logits: Optional[torch.Tensor],
    ) -> Dict[str, Optional[torch.Tensor]]:
        """Chuẩn hóa đầu vào dự đoán từ dict, tuple hoặc các tensor riêng lẻ."""
        out: Dict[str, Optional[torch.Tensor]] = {
            "frame_kss": None,
            "clip_kss": clip_kss,
            "frame_logits": frame_logits,
            "clip_logits": clip_logits,
            "window_kss": window_kss,
            "window_logits": window_logits,
            "window_valid_mask": None,
            "domain_logits": None,
            "window_domain_logits": None,
        }

        if isinstance(predictions, dict):
            out["frame_kss"] = predictions.get("frame_kss", out["frame_kss"])
            out["clip_kss"] = predictions.get("clip_kss", out["clip_kss"])
            out["frame_logits"] = predictions.get("frame_logits", out["frame_logits"])
            out["clip_logits"] = predictions.get("clip_logits", out["clip_logits"])
            out["window_kss"] = predictions.get("window_kss", out["window_kss"])
            out["window_logits"] = predictions.get("window_logits", out["window_logits"])
            out["window_valid_mask"] = predictions.get("window_valid_mask", out["window_valid_mask"])
            out["domain_logits"] = predictions.get("domain_logits", out["domain_logits"])
            out["window_domain_logits"] = predictions.get("window_domain_logits", out["window_domain_logits"])
        elif isinstance(predictions, (tuple, list)):
            items = list(predictions)
            if items and isinstance(items[-1], dict):
                items = items[:-1]
            if len(items) == 1:
                t0 = items[0]
                if t0.dim() == 3:
                    out["frame_logits"] = t0
                else:
                    out["frame_kss"] = t0
            elif len(items) == 2:
                a, b = items
                if a.dim() == 2 and b.dim() == 1:
                    out["frame_kss"], out["clip_kss"] = a, b
                elif a.dim() == 2 and b.dim() == 3:
                    out["frame_kss"], out["frame_logits"] = a, b
                elif a.dim() == 3 and b.dim() == 2:
                    out["frame_logits"], out["clip_logits"] = a, b
                else:
                    out["frame_kss"] = a
                    if out["clip_kss"] is None and b.dim() == 1:
                        out["clip_kss"] = b
            elif len(items) >= 4:
                out["frame_kss"], out["clip_kss"], out["frame_logits"], out["clip_logits"] = items[:4]
        elif isinstance(predictions, torch.Tensor):
            if predictions.dim() == 3:
                out["frame_logits"] = predictions
            elif predictions.dim() == 2:
                out["frame_kss"] = predictions
            elif predictions.dim() == 1:
                out["clip_kss"] = predictions

        if out["frame_kss"] is None and out["frame_logits"] is not None:
            out["frame_kss"] = self._logits_to_kss_expectation(out["frame_logits"])
        if out["clip_kss"] is None and out["clip_logits"] is not None:
            out["clip_kss"] = self._logits_to_kss_expectation(out["clip_logits"])

        return out

    def _prepare_targets(
            self,
            targets: torch.Tensor,
            lengths: Optional[torch.Tensor],
            device: torch.device,
    ) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor], Optional[torch.Tensor]]:
        """
        Chuẩn hóa targets về cả KSS liên tục [1.0, 9.0] và chỉ số lớp 0..8.
        Hỗ trợ targets dạng [B], [B, 1], hoặc [B, T].
        """
        tgt = targets.to(device)
        if tgt.dim() == 2 and tgt.size(1) == 1:
            tgt = tgt.squeeze(1)

        if tgt.dim() == 1:
            clip_target_kss = tgt.float().clamp(min=self.kss_min, max=self.kss_max)
            clip_target_cls = (clip_target_kss.round().long() - int(round(self.kss_min))).clamp(
                0, self.num_classes - 1
            )
            return clip_target_kss, clip_target_cls, None, None

        # Trường hợp targets dạng chuỗi theo khung hình [B, T]
        B, T = tgt.shape
        if lengths is not None:
            last_idx = (lengths.to(device).long() - 1).clamp(min=0, max=T - 1)
            clip_raw = tgt[torch.arange(B, device=device), last_idx]
        else:
            clip_raw = tgt[:, -1]

        clip_target_kss = clip_raw.float().clamp(min=self.kss_min, max=self.kss_max)
        clip_target_cls = (clip_target_kss.round().long() - int(round(self.kss_min))).clamp(
            0, self.num_classes - 1
        )
        frame_target_kss = tgt.float().clamp(min=self.kss_min, max=self.kss_max)
        frame_target_cls = (frame_target_kss.round().long() - int(round(self.kss_min))).clamp(
            0, self.num_classes - 1
        )
        return clip_target_kss, clip_target_cls, frame_target_kss, frame_target_cls

    @staticmethod
    def _weighted_mean(
            values: torch.Tensor,
            sample_weights: Optional[torch.Tensor] = None,
            valid_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Tính trung bình có trọng số mẫu trên batch (hỗ trợ sample_weights và valid_mask)."""
        val = values.float()
        weights = torch.ones_like(val)
        if sample_weights is not None:
            weights = weights * sample_weights.float().clamp(min=0.0).to(val.device)
        if valid_mask is not None:
            weights = weights * valid_mask.float().to(val.device)
        denom = weights.sum().clamp(min=1e-6)
        return (val * weights).sum() / denom

    def _compute_frame_mil_loss(
            self,
            frame_kss: Optional[torch.Tensor],
            frame_logits: Optional[torch.Tensor],
            clip_target_kss: torch.Tensor,
            clip_target_cls: torch.Tensor,
            frame_target_kss: Optional[torch.Tensor],
            frame_target_cls: Optional[torch.Tensor],
            lengths: Optional[torch.Tensor],
            sample_weights: Optional[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Causal Time-Weighted Top-K MIL Frame Loss cho KSS 1..9:
          - Trọng số nhân quả tăng dần 0.5 -> 1.0 theo độ dài hợp lệ từng mẫu.
          - KSS <= 4: giám sát 100% frame hợp lệ.
          - 4 < KSS <= 6: giám sát Top-K (70%) frame xu hướng mệt mỏi cao nhất.
          - KSS > 6: giám sát Top-K (45%) frame xu hướng buồn ngủ cao nhất.
        """
        ref = frame_kss if frame_kss is not None else frame_logits
        assert ref is not None
        B, T = ref.shape[:2]
        dev = ref.device

        if lengths is not None:
            valid_lens = lengths.to(dev).long().clamp(min=0, max=T)
            t_idx = torch.arange(T, device=dev).unsqueeze(0)
            valid_mask = t_idx < valid_lens.unsqueeze(1)
            # Ramp nhân quả tính riêng trên độ dài hợp lệ của từng mẫu (bất biến với batch padding)
            ramp_denom = (valid_lens.float() - 1.0).clamp(min=1.0).unsqueeze(1)
            ramp = torch.where(
                valid_lens.unsqueeze(1) > 1,
                0.5 + 0.5 * (t_idx.float() / ramp_denom),
                torch.ones((B, T), device=dev, dtype=torch.float32),
            )
            weight_matrix = ramp * valid_mask.float()
        else:
            valid_lens = torch.full((B,), T, dtype=torch.long, device=dev)
            valid_mask = torch.ones((B, T), dtype=torch.bool, device=dev)
            ramp = (0.5 + 0.5 * torch.linspace(0.0, 1.0, T, device=dev, dtype=torch.float32)).unsqueeze(0).expand(B, T)
            weight_matrix = ramp.clone()

        # Điểm xu hướng buồn ngủ của từng frame dùng để xếp hạng MIL Top-K
        if frame_kss is not None:
            drowsy_scores = frame_kss.float()
        else:
            assert frame_logits is not None
            drowsy_scores = self._logits_to_kss_expectation(frame_logits)
        drowsy_scores = drowsy_scores.masked_fill(~valid_mask, -1e4)

        mil_mask = valid_mask.clone()
        for b in range(B):
            t_len = int(valid_lens[b].item())
            if t_len <= 1:
                continue
            y_kss = float(clip_target_kss[b].item())
            if y_kss <= self.alert_kss_threshold:
                ratio = 1.0
            elif y_kss <= self.mid_kss_threshold:
                ratio = self.topk_mid_ratio
            else:
                ratio = self.topk_drowsy_ratio

            if ratio < 0.999:
                k_b = max(1, min(t_len, int(round(t_len * ratio))))
                if k_b < t_len:
                    _, topk_idx = torch.topk(drowsy_scores[b, :t_len], k=k_b, dim=0)
                    sample_mask = torch.zeros(T, dtype=torch.bool, device=dev)
                    sample_mask[topk_idx] = True
                    mil_mask[b] = sample_mask

        weight_matrix = weight_matrix * mil_mask.float()
        norm_weights = weight_matrix / weight_matrix.sum(dim=1, keepdim=True).clamp(min=1e-6)

        tgt_f_kss = (
            frame_target_kss.to(dev)
            if frame_target_kss is not None
            else clip_target_kss.unsqueeze(1).expand(B, T)
        )
        tgt_f_cls = (
            frame_target_cls.to(dev)
            if frame_target_cls is not None
            else clip_target_cls.unsqueeze(1).expand(B, T)
        )

        loss_reg_per = None
        if frame_kss is not None:
            loss_reg_per = F.smooth_l1_loss(
                frame_kss.float(),
                tgt_f_kss,
                beta=self.huber_beta,
                reduction="none",
            )

        loss_cls_per = None
        if frame_logits is not None:
            loss_cls_per = self._classification_ordinal_loss(frame_logits, tgt_f_kss, tgt_f_cls)

        if loss_reg_per is not None and loss_cls_per is not None:
            per_frame_loss = 0.65 * loss_reg_per + 0.35 * loss_cls_per
        elif loss_reg_per is not None:
            per_frame_loss = loss_reg_per
        else:
            assert loss_cls_per is not None
            per_frame_loss = loss_cls_per

        per_sample_loss = (per_frame_loss * norm_weights).sum(dim=1)
        active_samples = valid_lens > 0
        total_frame_loss = self._weighted_mean(per_sample_loss, sample_weights=sample_weights, valid_mask=active_samples)
        return total_frame_loss, mil_mask

    def _compute_window_loss(
            self,
            window_kss: Optional[torch.Tensor],
            window_logits: Optional[torch.Tensor],
            clip_target_kss: torch.Tensor,
            clip_target_cls: torch.Tensor,
            lengths: Optional[torch.Tensor],
            sample_weights: Optional[torch.Tensor],
            window_valid_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Sliding-Window Causal & Smoothness Loss cho chuỗi dài > 60s:
          - Phạt sai số KSS/Ordinal trên từng cửa sổ trượt hợp lệ với trọng số tăng dần theo thời gian.
          - Phạt dao động KSS đột ngột vượt quá `smooth_margin` giữa 2 cửa sổ chồng lấp liên tiếp.
        """
        ref = window_kss if window_kss is not None else window_logits
        if ref is None or ref.size(1) == 0:
            dev = clip_target_kss.device
            zero = torch.tensor(0.0, device=dev, dtype=torch.float32)
            return zero, zero

        B, W_num = ref.shape[:2]
        dev = ref.device

        if window_valid_mask is not None and window_valid_mask.shape == (B, W_num):
            win_valid = window_valid_mask.to(dev).bool()
        elif lengths is not None:
            lens = lengths.to(dev).long()
            win_starts = torch.arange(W_num, device=dev) * self.stride_size
            win_valid = lens.unsqueeze(1) > win_starts.unsqueeze(0)
            win_valid[:, 0] = win_valid[:, 0] | (lens > 0)
        else:
            win_valid = torch.ones((B, W_num), dtype=torch.bool, device=dev)

        valid_w_counts = win_valid.float().sum(dim=1, keepdim=True).clamp(min=1.0)
        w_idx = torch.arange(W_num, device=dev, dtype=torch.float32).unsqueeze(0)
        w_ramp = torch.where(
            valid_w_counts > 1.0,
            0.6 + 0.4 * (w_idx / (valid_w_counts - 1.0).clamp(min=1.0)),
            torch.ones((B, W_num), device=dev, dtype=torch.float32),
        )
        w_weights = w_ramp * win_valid.float()
        w_norm = w_weights / w_weights.sum(dim=1, keepdim=True).clamp(min=1e-6)

        tgt_w_kss = clip_target_kss.unsqueeze(1).expand(B, W_num)
        tgt_w_cls = clip_target_cls.unsqueeze(1).expand(B, W_num)

        win_reg = None
        if window_kss is not None:
            win_reg = F.smooth_l1_loss(
                window_kss.float(),
                tgt_w_kss,
                beta=self.huber_beta,
                reduction="none",
            )

        win_cls = None
        if window_logits is not None:
            win_cls = self._classification_ordinal_loss(window_logits, tgt_w_kss, tgt_w_cls)

        if win_reg is not None and win_cls is not None:
            per_win_loss = 0.65 * win_reg + 0.35 * win_cls
        elif win_reg is not None:
            per_win_loss = win_reg
        else:
            assert win_cls is not None
            per_win_loss = win_cls

        per_sample_win_loss = (per_win_loss * w_norm).sum(dim=1)
        active_samples = win_valid.any(dim=1)
        loss_window = self._weighted_mean(
            per_sample_win_loss,
            sample_weights=sample_weights,
            valid_mask=active_samples,
        )

        # Điều chuẩn độ mượt liên cửa sổ (chỉ tính khi có ít nhất 2 cửa sổ hợp lệ)
        if W_num >= 2 and window_kss is not None and self.window_smooth_weight > 0:
            pair_valid = win_valid[:, 1:] & win_valid[:, :-1]
            if pair_valid.any():
                diff = torch.abs(window_kss.float()[:, 1:] - window_kss.float()[:, :-1])
                excess = torch.clamp(diff - self.smooth_margin, min=0.0).pow(2) * pair_valid.float()
                pair_counts = pair_valid.float().sum(dim=1).clamp(min=1.0)
                per_sample_smooth = excess.sum(dim=1) / pair_counts
                has_pairs = pair_valid.any(dim=1)
                loss_smooth = self._weighted_mean(
                    per_sample_smooth,
                    sample_weights=sample_weights,
                    valid_mask=has_pairs,
                )
            else:
                loss_smooth = torch.tensor(0.0, device=dev, dtype=torch.float32)
        else:
            loss_smooth = torch.tensor(0.0, device=dev, dtype=torch.float32)

        return loss_window, loss_smooth

    def forward(
            self,
            predictions: Union[torch.Tensor, Dict[str, Any], Tuple[Any, ...]],
            targets: torch.Tensor,
            clip_logits: Optional[torch.Tensor] = None,
            lengths: Optional[torch.Tensor] = None,
            clip_kss: Optional[torch.Tensor] = None,
            frame_logits: Optional[torch.Tensor] = None,
            window_kss: Optional[torch.Tensor] = None,
            window_logits: Optional[torch.Tensor] = None,
            sample_weights: Optional[torch.Tensor] = None,
            domain_targets: Optional[torch.Tensor] = None,
            return_dict: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict[str, float]]]:
        """
        Tính tổng hàm mất mát Hybrid KSS Loss cho một batch dự đoán từ LiteFat_v1.

        Args:
            predictions: Dict (`return_dict=True`), Tuple từ `LiteFat_v1(...)`,
                         hoặc `frame_kss` [B, T] / `frame_logits` [B, T, 9].
            targets: Nhãn KSS mục tiêu [B] (int64 1..9 hoặc float 1.0..9.0), hoặc [B, T].
            clip_logits: Logits toàn mẫu [B, 9] (tùy chọn).
            lengths: Độ dài thực tế từng mẫu trong batch [B].
            clip_kss: Điểm KSS dự đoán toàn mẫu [B] (tùy chọn).
            frame_logits: Logits từng frame [B, T, 9] (tùy chọn).
            window_kss: Điểm KSS từng cửa sổ trượt [B, W_num] (tùy chọn).
            window_logits: Logits từng cửa sổ trượt [B, W_num, 9] (tùy chọn).
            sample_weights: Trọng số cân bằng mẫu [B] từ manifest (tùy chọn).
            domain_targets: Nhãn miền [B] (0=UL-DD, 1=UTA-RLDD) dùng cho DANN (tùy chọn).
            return_dict: True để trả kèm dict các thành phần loss chi tiết.
        """
        unpacked = self._unpack_predictions(
            predictions=predictions,
            clip_kss=clip_kss,
            frame_logits=frame_logits,
            clip_logits=clip_logits,
            window_kss=window_kss,
            window_logits=window_logits,
        )

        f_kss = unpacked["frame_kss"]
        c_kss = unpacked["clip_kss"]
        f_logits = unpacked["frame_logits"]
        c_logits = unpacked["clip_logits"]
        w_kss = unpacked["window_kss"]
        w_logits = unpacked["window_logits"]
        w_valid_mask = unpacked.get("window_valid_mask")
        d_logits = unpacked.get("domain_logits")
        w_d_logits = unpacked.get("window_domain_logits")

        ref_tensor = next(t for t in (c_kss, c_logits, f_kss, f_logits, w_kss, w_logits) if t is not None)
        dev = ref_tensor.device
        dev_type = dev.type if dev.type in ("cuda", "cpu") else "cpu"

        with torch.amp.autocast(dev_type, enabled=False):
            clip_target_kss, clip_target_cls, frame_target_kss, frame_target_cls = self._prepare_targets(
                targets=targets,
                lengths=lengths,
                device=dev,
            )

            # Fallback khi caller chỉ trả về đầu ra cấp frame (return_clip_logits=False)
            if c_kss is None and f_kss is not None:
                if lengths is not None:
                    T_f = f_kss.size(1)
                    mask_f = (torch.arange(T_f, device=dev).unsqueeze(0) < lengths.to(dev).unsqueeze(1)).float()
                    c_kss = (f_kss.float() * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                else:
                    c_kss = f_kss.float().mean(dim=1)
            if c_logits is None and f_logits is not None:
                if lengths is not None:
                    T_f = f_logits.size(1)
                    mask_f = (
                        torch.arange(T_f, device=dev).unsqueeze(0) < lengths.to(dev).unsqueeze(1)
                    ).unsqueeze(-1).float()
                    c_logits = (f_logits.float() * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
                else:
                    c_logits = f_logits.float().mean(dim=1)

            # 1. Clip Regression Loss (SmoothL1 + MSE)
            if c_kss is not None and self.clip_reg_weight > 0:
                c_kss_32 = c_kss.float()
                smooth_l1 = F.smooth_l1_loss(c_kss_32, clip_target_kss, beta=self.huber_beta, reduction="none")
                mse = F.mse_loss(c_kss_32, clip_target_kss, reduction="none")
                loss_clip_reg = self._weighted_mean(0.75 * smooth_l1 + 0.25 * mse, sample_weights=sample_weights)
            else:
                loss_clip_reg = torch.tensor(0.0, device=dev, dtype=torch.float32)

            # 2. Clip Ordinal Gaussian Soft-Label + CrossEntropy Loss
            if c_logits is not None and self.clip_cls_weight > 0:
                cls_loss_vec = self._classification_ordinal_loss(c_logits, clip_target_kss, clip_target_cls)
                loss_clip_cls = self._weighted_mean(cls_loss_vec, sample_weights=sample_weights)
            else:
                loss_clip_cls = torch.tensor(0.0, device=dev, dtype=torch.float32)

            # 3. Causal Time-Weighted Top-K MIL Frame Loss
            if self.frame_loss_weight > 0 and (f_kss is not None or f_logits is not None):
                loss_frame, _ = self._compute_frame_mil_loss(
                    frame_kss=f_kss,
                    frame_logits=f_logits,
                    clip_target_kss=clip_target_kss,
                    clip_target_cls=clip_target_cls,
                    frame_target_kss=frame_target_kss,
                    frame_target_cls=frame_target_cls,
                    lengths=lengths,
                    sample_weights=sample_weights,
                )
            else:
                loss_frame = torch.tensor(0.0, device=dev, dtype=torch.float32)

            # 4. Sliding-Window Causal & Smoothness Loss
            if (self.window_loss_weight > 0 or self.window_smooth_weight > 0) and (
                w_kss is not None or w_logits is not None
            ):
                loss_window, loss_smooth = self._compute_window_loss(
                    window_kss=w_kss,
                    window_logits=w_logits,
                    clip_target_kss=clip_target_kss,
                    clip_target_cls=clip_target_cls,
                    lengths=lengths,
                    sample_weights=sample_weights,
                    window_valid_mask=w_valid_mask,
                )
            else:
                loss_window = torch.tensor(0.0, device=dev, dtype=torch.float32)
                loss_smooth = torch.tensor(0.0, device=dev, dtype=torch.float32)

            # 5. Regime-Conditioned Domain-Adversarial Loss (DANN)
            loss_domain = torch.tensor(0.0, device=dev, dtype=torch.float32)
            if self.dann_weight > 0 and domain_targets is not None and (
                d_logits is not None or w_d_logits is not None
            ):
                dom_tgt = domain_targets.to(dev).long().view(-1)
                dom_valid = dom_tgt >= 0
                if dom_valid.any():
                    loss_d_clip = None
                    if d_logits is not None:
                        per_sample_d_clip = F.cross_entropy(
                            d_logits.float(),
                            dom_tgt.clamp(min=0),
                            reduction="none",
                        )
                        loss_d_clip = self._weighted_mean(
                            per_sample_d_clip,
                            sample_weights=sample_weights,
                            valid_mask=dom_valid,
                        )

                    loss_d_win = None
                    if w_d_logits is not None and w_d_logits.size(1) > 0:
                        B_w, W_w, C_w = w_d_logits.shape
                        if w_valid_mask is not None and w_valid_mask.shape == (B_w, W_w):
                            win_v = w_valid_mask.to(dev).bool()
                        elif lengths is not None:
                            lens_w = lengths.to(dev).long()
                            ws_w = torch.arange(W_w, device=dev) * self.stride_size
                            win_v = lens_w.unsqueeze(1) > ws_w.unsqueeze(0)
                            win_v[:, 0] = win_v[:, 0] | (lens_w > 0)
                        else:
                            win_v = torch.ones((B_w, W_w), dtype=torch.bool, device=dev)

                        tgt_w_dom = dom_tgt.clamp(min=0).unsqueeze(1).expand(B_w, W_w)
                        per_win_d = F.cross_entropy(
                            w_d_logits.float().reshape(-1, C_w),
                            tgt_w_dom.reshape(-1),
                            reduction="none",
                        ).reshape(B_w, W_w)
                        per_sample_d_win = (per_win_d * win_v.float()).sum(dim=1) / win_v.float().sum(dim=1).clamp(min=1.0)
                        active_d_win = win_v.any(dim=1) & dom_valid
                        loss_d_win = self._weighted_mean(
                            per_sample_d_win,
                            sample_weights=sample_weights,
                            valid_mask=active_d_win,
                        )

                    if loss_d_clip is not None and loss_d_win is not None:
                        loss_domain = 0.5 * loss_d_clip + 0.5 * loss_d_win
                    elif loss_d_clip is not None:
                        loss_domain = loss_d_clip
                    elif loss_d_win is not None:
                        loss_domain = loss_d_win

            total_loss = (
                self.clip_reg_weight * loss_clip_reg
                + self.clip_cls_weight * loss_clip_cls
                + self.frame_loss_weight * loss_frame
                + self.window_loss_weight * loss_window
                + self.window_smooth_weight * loss_smooth
                + self.dann_weight * loss_domain
            )

        if return_dict:
            details = {
                "loss_total": float(total_loss.detach().item()),
                "loss_clip_reg": float(loss_clip_reg.detach().item()),
                "loss_clip_cls": float(loss_clip_cls.detach().item()),
                "loss_frame": float(loss_frame.detach().item()),
                "loss_window": float(loss_window.detach().item()),
                "loss_smooth": float(loss_smooth.detach().item()),
                "loss_domain": float(loss_domain.detach().item()),
            }
            return total_loss, details

        return total_loss


LiteFatV1Loss = LiteFatLoss

In [11]:
# =====================================================================
# File: train.py
# Dự án: Driver Guardian - Module LiteFat_v1 (KSS 1-9, Sliding 60s/30s)
# Tác vụ: Lớp LiteFatTrainer thực thi pipeline huấn luyện, lưu checkpoint,
#         đánh giá Validation/Test (KSS + phân rã dataset/subject) và
#         ghi log TensorBoard theo cấu hình TrainConfig trong config.py.
# =====================================================================
from __future__ import annotations

import json
import math
import multiprocessing as mp

try:
    mp.set_start_method("fork", force=True)
except Exception:
    pass

import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import cv2

try:
    cv2.setNumThreads(0)
except Exception:
    pass

import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

try:
    from config import TrainConfig
    from dataset import build_litefat_dataloaders
    from loss import LiteFatLoss
    from model import LiteFatV1
    from ultis import (
        ModelEMA,
        atomic_save,
        compute_kss_metrics,
        configure_logging,
        load_checkpoint,
        load_pretrained_weights,
        seed_everything,
    )
except ImportError:
    pass


class LiteFatTrainer:
    """
    Trình quản lý và thực thi huấn luyện mô hình LiteFatV1 (dự đoán điểm KSS 1..9):
      - Hỗ trợ Differential Learning Rate (Backbone MobileNetV3 vs Đồ thị STGL/KSS Head).
      - Đóng băng Backbone theo giai đoạn (Staged Backbone Freezing).
      - Linear Warmup + Cosine Annealing Learning Rate + GradScaler AMP.
      - Exponential Moving Average (EMA) cho đánh giá ổn định.
      - Lưu checkpoint an toàn (last.pt, best.pt, epoch_XX.pt) và theo dõi TensorBoard.
    """

    def __init__(
            self,
            config: Optional[TrainConfig] = None,
            dataloaders: Optional[Tuple[Any, Any, Any]] = None,
    ):
        self.cfg = config if config is not None else TrainConfig()

        # 1. Thiết lập hạt giống ngẫu nhiên
        seed_everything(self.cfg.seed)

        # 2. Thư mục lưu trữ, Logging & TensorBoard
        self.save_dir = Path(self.cfg.checkpoint_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.logger = configure_logging(self.save_dir)
        self.cfg.save_json(self.save_dir / "config.json")

        log_path = Path(self.cfg.log_dir)
        log_path.mkdir(parents=True, exist_ok=True)
        self.writer = SummaryWriter(log_dir=str(log_path))

        # 3. Thiết bị tính toán
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.logger.info("=" * 72)
        self.logger.info(f"KHỞI TẠO LITEFAT_V1 TRAINER (Thiết bị: {self.device})")
        self.logger.info(f"Ngữ cảnh thị giác (use_visual_context): {self.cfg.use_visual_context}")
        self.logger.info(
            f"Lấy mẫu: {self.cfg.fps} FPS (dt={self.cfg.time_interval}s) | "
            f"Model Window={self.cfg.window_sec}s / Stride={self.cfg.stride_sec}s | "
            f"Dataset Window={self.cfg.dataset_window_sec}"
        )
        self.grad_accum_steps = max(1, int(getattr(self.cfg, "grad_accum_steps", 1)))
        self.logger.info(
            f"Batch size: {self.cfg.batch_size} | Grad accum steps: {self.grad_accum_steps} | "
            f"Effective batch size: {self.cfg.batch_size * self.grad_accum_steps}"
        )
        self.logger.info(f"Bộ dữ liệu: {self.cfg.selected_datasets or 'Tất cả'} ({self.cfg.manifest_filename})")
        self.logger.info("=" * 72)

        # 4. Xây dựng DataLoaders (Train, Val, Test)
        if dataloaders is not None:
            self.train_loader, self.val_loader, self.test_loader = dataloaders
        else:
            self.logger.info("Đang khởi tạo DataLoaders...")
            self.train_loader, self.val_loader, self.test_loader = build_litefat_dataloaders(
                dataset_dir=self.cfg.dataset_dir,
                batch_size=self.cfg.batch_size,
                num_workers=self.cfg.num_workers,
                time_interval=self.cfg.time_interval,
                window_sec=self.cfg.dataset_window_sec,
                letterbox_size=self.cfg.letterbox_size,
                face_size=self.cfg.face_size,
                manifest_filename=self.cfg.manifest_filename,
                use_cache=self.cfg.use_cache,
                cache_dir=self.cfg.cache_dir,
                pin_memory=self.cfg.pin_memory,
                shuffle_train=True,
                device=str(self.device),
                pipnet_backbone=self.cfg.pipnet_backbone,
                conf_threshold=self.cfg.conf_threshold,
                nms_threshold=self.cfg.nms_threshold,
                bbox_margin=self.cfg.bbox_margin,
                box_smooth_alpha=self.cfg.box_smooth_alpha,
                extract_crops_only=(not self.cfg.use_visual_context),
                selected_datasets=self.cfg.selected_datasets,
                use_visual_context=self.cfg.use_visual_context,
                augment_train=self.cfg.augment_train,
                flip_prob=self.cfg.flip_prob,
                scale_jitter=self.cfg.scale_jitter,
                shift_jitter=self.cfg.shift_jitter,
                roll_deg=self.cfg.roll_deg,
                temporal_dropout=self.cfg.temporal_dropout,
                landmark_noise_std=self.cfg.landmark_noise_std,
                color_jitter=self.cfg.color_jitter,
                spatial_dropout=self.cfg.spatial_dropout,
                subject_balanced_sampling=self.cfg.subject_balanced_sampling,
                use_manifest_sample_weights=self.cfg.use_manifest_sample_weights,
                eval_window_mode=self.cfg.eval_window_mode,
                drop_single_class_subjects=self.cfg.drop_single_class_subjects,
                visual_frame_stride=self.cfg.visual_frame_stride,
            )

        if hasattr(self.train_loader, "dataset") and hasattr(self.train_loader.dataset, "return_meta"):
            self.train_loader.dataset.return_meta = True

        self.logger.info(
            f"Kích thước tập dữ liệu: Train={len(self.train_loader.dataset)} | "
            f"Val={len(self.val_loader.dataset)} | Test={len(self.test_loader.dataset)}"
        )

        # 5. Khởi tạo Mô hình LiteFatV1 & EMA
        self.logger.info("Đang khởi tạo mô hình LiteFatV1 (KSS 1-9)...")
        use_dann = bool(getattr(self.cfg, "use_dann", False))
        self.model = LiteFatV1(
            num_nodes=self.cfg.num_nodes,
            feature_dim=self.cfg.feature_dim,
            mobilenet_proj_hidden=self.cfg.mobilenet_proj_hidden,
            node_embed_dim=self.cfg.node_embed_dim,
            residual_channels=self.cfg.residual_channels,
            dilation_channels=self.cfg.dilation_channels,
            skip_channels=self.cfg.skip_channels,
            end_channels=self.cfg.end_channels,
            num_classes=self.cfg.num_classes,
            kss_min=self.cfg.kss_min,
            kss_max=self.cfg.kss_max,
            fps=self.cfg.fps,
            window_sec=self.cfg.window_sec,
            stride_sec=self.cfg.stride_sec,
            num_blocks=self.cfg.num_blocks,
            layers_per_block=self.cfg.layers_per_block,
            kernel_size=self.cfg.kernel_size,
            pretrained_backbone=self.cfg.pretrained_backbone,
            dropout=self.cfg.dropout,
            freeze_backbone=(self.cfg.freeze_backbone_epochs > 0),
            vertical_pairs=self.cfg.vertical_pairs,
            cross_region_pairs=self.cfg.cross_region_pairs,
            static_shape_ratio=self.cfg.static_shape_ratio,
            use_dann=use_dann,
        ).to(self.device)
        self.model.set_visual_context_enabled(self.cfg.use_visual_context)
        self.model.set_dann_enabled(use_dann)

        self.ema = ModelEMA(self.model, decay=self.cfg.ema_decay)
        self.early_stop_counter = 0

        stats = self.model.get_trainable_params_count()
        self.logger.info(
            f"Tổng số tham số: {stats['total_params']:,} | Khả học: {stats['trainable_params']:,} "
            f"(MobileNetV3: {stats['backbone_trainable']:,} / {stats['backbone_total']:,})"
        )

        # 6. Khởi tạo Hàm mất mát Hybrid KSS Loss
        self.criterion = LiteFatLoss(
            num_classes=self.cfg.num_classes,
            kss_min=self.cfg.kss_min,
            kss_max=self.cfg.kss_max,
            clip_reg_weight=self.cfg.clip_reg_weight,
            clip_cls_weight=self.cfg.clip_cls_weight,
            frame_loss_weight=self.cfg.frame_loss_weight,
            window_loss_weight=self.cfg.window_loss_weight,
            window_smooth_weight=self.cfg.window_smooth_weight,
            dann_weight=float(getattr(self.cfg, "dann_weight", 0.05)),
            huber_beta=self.cfg.huber_beta,
            ordinal_sigma=self.cfg.ordinal_sigma,
            label_smoothing=self.cfg.label_smoothing,
            alert_kss_threshold=self.cfg.alert_kss_threshold,
            mid_kss_threshold=self.cfg.mid_kss_threshold,
            topk_mid_ratio=self.cfg.topk_mid_ratio,
            topk_drowsy_ratio=self.cfg.topk_drowsy_ratio,
            fps=self.cfg.fps,
            window_sec=self.cfg.window_sec,
            stride_sec=self.cfg.stride_sec,
            smooth_margin=self.cfg.smooth_margin,
        ).to(self.device)

        # Biến theo dõi thực nghiệm
        self.start_epoch = 1
        self.best_val_score = -1.0
        self.best_val_mae = float("inf")
        self.best_val_f1 = -1.0
        self.best_epoch = -1
        self.history: List[Dict[str, Any]] = []
        self.pretrained_param_names: Set[str] = set()

        # Warm-start nếu được yêu cầu và không resume
        if self.cfg.pretrained_from and not self.cfg.resume:
            self.load_pretrained_weights(self.cfg.pretrained_from)

        # 7. Optimizer, Scheduler & GradScaler AMP
        self.optimizer = self._build_optimizer()
        self.scheduler = self._build_scheduler()
        amp_enabled = self.cfg.use_amp and self.device.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

        # Khôi phục checkpoint nếu có cfg.resume
        if self.cfg.resume:
            self.load_checkpoint(self.cfg.resume)

    def _build_optimizer(self) -> torch.optim.Optimizer:
        """Thiết lập AdamW với tốc độ học phân biệt (Differential LR)."""
        backbone_params = []
        matched_graph_params = []
        new_graph_params = []

        pretrained_scale = float(self.cfg.pretrained_lr_scale)
        has_pretrained = bool(self.pretrained_param_names) and (pretrained_scale < 1.0)

        for name, param in self.model.named_parameters():
            if "context_encoder.features" in name:
                backbone_params.append(param)
                continue
            if not param.requires_grad:
                continue
            if has_pretrained and name in self.pretrained_param_names:
                matched_graph_params.append(param)
            else:
                new_graph_params.append(param)

        param_groups = []
        if matched_graph_params:
            param_groups.append({
                "params": matched_graph_params,
                "lr": self.cfg.lr * pretrained_scale,
                "weight_decay": self.cfg.weight_decay,
                "name": "pretrained_graph",
            })
        if new_graph_params:
            param_groups.append({
                "params": new_graph_params,
                "lr": self.cfg.lr,
                "weight_decay": self.cfg.weight_decay,
                "name": "new_graph",
            })
        if backbone_params and self.cfg.use_visual_context:
            initial_backbone_lr = (
                self.cfg.backbone_lr if self.start_epoch > self.cfg.freeze_backbone_epochs else 0.0
            )
            param_groups.append({
                "params": backbone_params,
                "lr": initial_backbone_lr,
                "weight_decay": self.cfg.weight_decay,
                "name": "backbone",
            })

        return torch.optim.AdamW(param_groups)

    def _build_scheduler(self) -> torch.optim.lr_scheduler.LambdaLR:
        """Thiết lập lịch trình học Linear Warmup + Cosine Annealing."""
        warmup_epochs = max(0, int(self.cfg.warmup_epochs))
        total_epochs = max(1, int(self.cfg.epochs))

        def lr_lambda(epoch: int) -> float:
            if epoch < warmup_epochs:
                return float(epoch + 1) / float(max(1, warmup_epochs))
            progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        return torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_lambda=lr_lambda)

    def load_checkpoint(self, checkpoint_path: str | Path) -> Dict[str, Any]:
        """Nạp checkpoint thông qua hàm phụ trợ `load_checkpoint` trong ultis.py."""
        info = load_checkpoint(
            checkpoint_path=checkpoint_path,
            model=self.model,
            optimizer=self.optimizer,
            scheduler=self.scheduler,
            scaler=self.scaler,
            ema=self.ema,
            save_dir=self.save_dir,
            cfg=self.cfg,
            device=self.device,
            logger=self.logger,
        )
        self.start_epoch = info["start_epoch"]
        self.best_epoch = info["best_epoch"]
        self.best_val_score = info["best_val_score"]
        self.best_val_mae = info["best_val_mae"]
        self.best_val_f1 = info["best_val_f1"]
        if info.get("scheduler") is not None:
            self.scheduler = info["scheduler"]
        if isinstance(info.get("history"), list):
            self.history = info["history"]
        return info

    def load_pretrained_weights(self, pretrained_path: str | Path) -> Set[str]:
        """Nạp trọng số warm-start thông qua `load_pretrained_weights` trong ultis.py."""
        self.pretrained_param_names = load_pretrained_weights(
            pretrained_path=pretrained_path,
            model=self.model,
            use_visual_context=self.cfg.use_visual_context,
            device=self.device,
            logger=self.logger,
        )
        return self.pretrained_param_names

    @staticmethod
    def _unpack_batch(
            batch: Tuple[Any, ...],
    ) -> Tuple[
        Optional[torch.Tensor],
        torch.Tensor,
        torch.Tensor,
        Optional[torch.Tensor],
        Optional[torch.Tensor],
        Optional[torch.Tensor],
    ]:
        batch_frames = batch[0]
        batch_landmarks = batch[1]
        batch_labels = batch[2]
        batch_lengths = batch[3] if len(batch) >= 4 else None
        sample_weights = None
        domain_targets = None
        if len(batch) >= 5 and isinstance(batch[4], (list, tuple)) and batch[4] and isinstance(batch[4][0], dict):
            metas = batch[4]
            sample_weights = torch.tensor(
                [max(1e-6, float(m.get("sample_weight", 1.0))) for m in metas],
                dtype=torch.float32,
            )
            domain_targets = torch.tensor(
                [int(m.get("domain_id", -1)) for m in metas],
                dtype=torch.long,
            )
        return batch_frames, batch_landmarks, batch_labels, batch_lengths, sample_weights, domain_targets

    def _step_optimizer(self) -> None:
        """Thực hiện gradient clipping (nếu bật), bước tối ưu AMP và cập nhật EMA."""
        if self.cfg.grad_clip > 0:
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.cfg.grad_clip)

        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.ema.update(self.model)
        self.optimizer.zero_grad(set_to_none=True)

    def _train_epoch(self, epoch: int) -> Dict[str, float]:
        """Thực thi một epoch huấn luyện cho LiteFatV1 (hỗ trợ tích lũy gradient)."""
        self.model.train()

        # Đóng/mở khóa backbone MobileNetV3 theo epoch
        if epoch <= self.cfg.freeze_backbone_epochs:
            self.model.set_backbone_trainable(False)
            for g in self.optimizer.param_groups:
                if g.get("name") == "backbone":
                    g["lr"] = 0.0
        else:
            self.model.set_backbone_trainable(True, unfreeze_last_n_blocks=self.cfg.unfreeze_last_n_blocks)
            base_lr_ref = self.optimizer.param_groups[0]["lr"]
            base_ratio = base_lr_ref / max(1e-9, self.cfg.lr)
            for g in self.optimizer.param_groups:
                if g.get("name") == "backbone":
                    g["lr"] = self.cfg.backbone_lr * base_ratio

        total_loss = 0.0
        comp_sums: Dict[str, float] = {
            "loss_clip_reg": 0.0,
            "loss_clip_cls": 0.0,
            "loss_frame": 0.0,
            "loss_window": 0.0,
            "loss_smooth": 0.0,
            "loss_domain": 0.0,
        }
        all_targets: List[float] = []
        all_pred_kss: List[float] = []
        all_pred_logits: List[np.ndarray] = []

        total_batches = len(self.train_loader)
        num_batches = max(1, total_batches)
        total_train_steps = max(1, int(self.cfg.epochs) * num_batches)
        use_dann = bool(getattr(self.cfg, "use_dann", False))
        dann_max_alpha = float(getattr(self.cfg, "dann_max_alpha", 0.20))
        use_sample_weights = bool(getattr(self.cfg, "use_manifest_sample_weights", True))
        accum_steps = max(1, int(getattr(self.cfg, "grad_accum_steps", 1)))
        valid_steps = 0
        opt_steps = 0
        accum_count = 0
        current_accum_target = min(accum_steps, num_batches)
        start_time = time.time()

        self.optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(
            self.train_loader,
            desc=f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [Train]",
            dynamic_ncols=True,
            leave=False,
        )

        for batch_idx, batch in enumerate(pbar):
            # Tính kích thước cửa sổ tích lũy hiện tại để giữ đúng trung bình gradient ở nhóm lẻ cuối epoch
            if accum_count == 0:
                current_accum_target = min(accum_steps, max(1, total_batches - batch_idx))

            if use_dann:
                progress = min(1.0, max(0.0, float((epoch - 1) * num_batches + batch_idx) / float(total_train_steps)))
                dann_alpha = dann_max_alpha * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)
                self.model.set_dann_alpha(dann_alpha)
            else:
                self.model.set_dann_alpha(0.0)

            batch_frames, batch_landmarks, batch_labels, batch_lengths, batch_weights, batch_domains = self._unpack_batch(batch)
            batch_landmarks = batch_landmarks.to(self.device, non_blocking=True)
            batch_labels = batch_labels.to(self.device, non_blocking=True)
            if batch_frames is not None:
                batch_frames = batch_frames.to(self.device, non_blocking=True)
            if batch_lengths is not None:
                batch_lengths = batch_lengths.to(self.device, non_blocking=True)
            batch_weights = batch_weights.to(self.device, non_blocking=True) if (use_sample_weights and batch_weights is not None) else None
            batch_domains = batch_domains.to(self.device, non_blocking=True) if (use_dann and batch_domains is not None) else None

            with torch.amp.autocast(device_type=self.device.type, enabled=self.scaler.is_enabled()):
                out_dict = self.model(
                    video_frames=batch_frames,
                    landmarks=batch_landmarks,
                    lengths=batch_lengths,
                    return_dict=True,
                )
                loss, loss_details = self.criterion(
                    out_dict,
                    batch_labels,
                    lengths=batch_lengths,
                    sample_weights=batch_weights,
                    domain_targets=batch_domains,
                    return_dict=True,
                )

            if not torch.isfinite(loss):
                self.logger.warning(
                    f"Epoch {epoch:02d} | Batch {batch_idx + 1}/{num_batches}: Loss không hữu hạn ({loss.item()}). Bỏ qua bước cập nhật."
                )
                del out_dict, loss, loss_details, batch_frames, batch_landmarks, batch_labels, batch_lengths, batch_weights, batch_domains
                continue

            loss_to_backward = loss if current_accum_target == 1 else (loss / float(current_accum_target))
            self.scaler.scale(loss_to_backward).backward()
            accum_count += 1

            last_loss = float(loss.item())
            valid_steps += 1
            total_loss += last_loss
            for k in comp_sums:
                comp_sums[k] += float(loss_details.get(k, 0.0))

            clip_kss_np = out_dict["clip_kss"].detach().float().cpu().numpy()
            clip_logits_np = out_dict["clip_logits"].detach().float().cpu().numpy()
            targets_np = batch_labels.detach().float().cpu().numpy()

            all_pred_kss.extend(clip_kss_np.tolist())
            all_targets.extend(targets_np.tolist())
            all_pred_logits.append(clip_logits_np)

            del out_dict, loss, loss_details, loss_to_backward
            del batch_frames, batch_landmarks, batch_labels, batch_lengths, batch_weights, batch_domains

            if accum_count >= current_accum_target:
                self._step_optimizer()
                accum_count = 0
                opt_steps += 1

            current_lr = self.optimizer.param_groups[0]["lr"]
            postfix = {
                "loss": f"{last_loss:.4f}",
                "avg_loss": f"{total_loss / valid_steps:.4f}",
                "lr": f"{current_lr:.2e}",
            }
            if accum_steps > 1:
                postfix["opt_steps"] = opt_steps
            pbar.set_postfix(postfix)

        # Flush phần gradient hợp lệ còn lại nếu kết thúc epoch mà vẫn còn tích lũy
        if accum_count > 0:
            self._step_optimizer()
            opt_steps += 1

        if self.device.type == "cuda":
            torch.cuda.empty_cache()

        elapsed = time.time() - start_time
        logits_cat = np.concatenate(all_pred_logits, axis=0) if all_pred_logits else None
        metrics = compute_kss_metrics(
            np.array(all_targets),
            np.array(all_pred_kss),
            pred_logits=logits_cat,
            kss_min=self.cfg.kss_min,
            kss_max=self.cfg.kss_max,
        )
        denom = max(1, valid_steps)
        metrics["loss"] = total_loss / denom
        for k, v in comp_sums.items():
            metrics[k] = v / denom
        metrics["opt_steps"] = float(opt_steps)
        metrics["time"] = elapsed
        return metrics

    @torch.no_grad()
    def _evaluate_epoch(
            self,
            dataloader: Any,
            split_name: str = "val",
            epoch: Optional[int] = None,
            return_raw: bool = False,
            use_ema: bool = False,
    ) -> Any:
        """Đánh giá mô hình trên tập validation hoặc test (hỗ trợ EMA & phân rã theo dataset)."""
        backup = self.ema.apply_shadow(self.model) if (use_ema and hasattr(self, "ema")) else None
        self.model.eval()

        total_loss = 0.0
        all_targets: List[float] = []
        all_pred_kss: List[float] = []
        all_pred_logits: List[np.ndarray] = []
        num_batches = max(1, len(dataloader))
        start_time = time.time()

        desc = (
            f"Epoch {epoch:02d}/{self.cfg.epochs:02d} [{split_name.capitalize()}{'(EMA)' if use_ema else ''}]"
            if epoch is not None
            else f"[{split_name.capitalize()}{'(EMA)' if use_ema else ''}]"
        )
        pbar = tqdm(dataloader, desc=desc, dynamic_ncols=True, leave=False)

        try:
            for batch_idx, batch in enumerate(pbar):
                batch_frames, batch_landmarks, batch_labels, batch_lengths, _, _ = self._unpack_batch(batch)
                batch_landmarks = batch_landmarks.to(self.device, non_blocking=True)
                batch_labels = batch_labels.to(self.device, non_blocking=True)
                if batch_frames is not None:
                    batch_frames = batch_frames.to(self.device, non_blocking=True)
                if batch_lengths is not None:
                    batch_lengths = batch_lengths.to(self.device, non_blocking=True)

                with torch.amp.autocast(device_type=self.device.type, enabled=self.scaler.is_enabled()):
                    out_dict = self.model(
                        video_frames=batch_frames,
                        landmarks=batch_landmarks,
                        lengths=batch_lengths,
                        return_dict=True,
                    )
                    loss = self.criterion(
                        out_dict,
                        batch_labels,
                        lengths=batch_lengths,
                    )

                total_loss += float(loss.item())
                all_pred_kss.extend(out_dict["clip_kss"].float().cpu().numpy().tolist())
                all_targets.extend(batch_labels.float().cpu().numpy().tolist())
                all_pred_logits.append(out_dict["clip_logits"].float().cpu().numpy())

                pbar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "avg_loss": f"{total_loss / (batch_idx + 1):.4f}",
                })
        finally:
            if backup is not None:
                self.ema.restore(self.model, backup)

        elapsed = time.time() - start_time
        targets_np = np.array(all_targets, dtype=np.float64)
        pred_kss_np = np.array(all_pred_kss, dtype=np.float64)
        logits_np = np.concatenate(all_pred_logits, axis=0) if all_pred_logits else np.empty((0, self.cfg.num_classes))

        metrics = compute_kss_metrics(
            targets_np,
            pred_kss_np,
            pred_logits=logits_np if len(logits_np) > 0 else None,
            kss_min=self.cfg.kss_min,
            kss_max=self.cfg.kss_max,
        )
        metrics["loss"] = total_loss / max(1, len(dataloader))
        metrics["time"] = elapsed

        # Phân rã chỉ số theo từng bộ dữ liệu con (UL-DD, UTA-RLDD, ...)
        samples = getattr(dataloader.dataset, "samples", [])
        per_dataset_metrics: Dict[str, Dict[str, float]] = {}
        if len(samples) == len(targets_np) and len(samples) > 0:
            ds_arr = np.array([str(s.get("dataset", "unknown")) for s in samples])
            for d_name in np.unique(ds_arr):
                m_d = (ds_arr == d_name)
                d_metrics = compute_kss_metrics(
                    targets_np[m_d],
                    pred_kss_np[m_d],
                    pred_logits=logits_np[m_d] if len(logits_np) == len(targets_np) else None,
                    kss_min=self.cfg.kss_min,
                    kss_max=self.cfg.kss_max,
                )
                per_dataset_metrics[str(d_name)] = {
                    "mae": d_metrics["mae"],
                    "rmse": d_metrics["rmse"],
                    "within_1_acc": d_metrics["within_1_acc"],
                    "qwk": d_metrics["qwk"],
                    "auc": d_metrics["auc"],
                    "f1": d_metrics["f1"],
                    "num_samples": int(np.sum(m_d)),
                }
        metrics["per_dataset_metrics"] = per_dataset_metrics

        if return_raw:
            return metrics, targets_np, pred_kss_np, logits_np
        return metrics

    def _log_tensorboard(self, epoch: int, train_m: Dict[str, float], val_m: Dict[str, float], lr: float) -> None:
        """Ghi toàn bộ chỉ số KSS & loss vào TensorBoard."""
        self.writer.add_scalar("Loss/Train", train_m["loss"], epoch)
        self.writer.add_scalar("Loss/Val", val_m["loss"], epoch)
        for comp in ("loss_clip_reg", "loss_clip_cls", "loss_frame", "loss_window", "loss_smooth", "loss_domain"):
            if comp in train_m:
                self.writer.add_scalar(f"Loss/{comp}", train_m[comp], epoch)

        self.writer.add_scalar("KSS_MAE/Train", train_m["mae"], epoch)
        self.writer.add_scalar("KSS_MAE/Val", val_m["mae"], epoch)
        self.writer.add_scalar("KSS_RMSE/Train", train_m["rmse"], epoch)
        self.writer.add_scalar("KSS_RMSE/Val", val_m["rmse"], epoch)
        self.writer.add_scalar("KSS_Within1/Train", train_m["within_1_acc"], epoch)
        self.writer.add_scalar("KSS_Within1/Val", val_m["within_1_acc"], epoch)
        self.writer.add_scalar("KSS_ExactAcc/Val", val_m["exact_acc"], epoch)
        self.writer.add_scalar("KSS_QWK/Train", train_m["qwk"], epoch)
        self.writer.add_scalar("KSS_QWK/Val", val_m["qwk"], epoch)
        self.writer.add_scalar("KSS_RegimeAcc/Val", val_m["regime_acc"], epoch)
        self.writer.add_scalar("Drowsy_F1/Val", val_m["f1"], epoch)
        self.writer.add_scalar("Drowsy_AUC/Val", val_m["auc"], epoch)
        self.writer.add_scalar("Score/Val", val_m["val_score"], epoch)
        self.writer.add_scalar("LR", lr, epoch)

        for d_name, d_m in val_m.get("per_dataset_metrics", {}).items():
            self.writer.add_scalar(f"KSS_MAE/Val_{d_name}", d_m["mae"], epoch)
            self.writer.add_scalar(f"KSS_Within1/Val_{d_name}", d_m["within_1_acc"], epoch)
            self.writer.add_scalar(f"KSS_QWK/Val_{d_name}", d_m["qwk"], epoch)

    def train(self) -> Dict[str, float]:
        """Vòng lặp huấn luyện chính qua toàn bộ các epoch."""
        if self.start_epoch > self.cfg.epochs:
            self.logger.warning(
                f"Checkpoint đã hoàn thành Epoch {self.start_epoch - 1}, trong khi cfg.epochs={self.cfg.epochs}. "
                "Vui lòng tăng 'epochs' trong TrainConfig để tiếp tục train."
            )
            test_m = self.evaluate_test()
            self.close()
            return test_m

        self.logger.info("\n>>> BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN LITEFAT_V1 (KSS 1-9) <<<")
        if self.start_epoch > 1:
            self.logger.info(f"Tiếp tục huấn luyện từ Epoch {self.start_epoch:02d} đến {self.cfg.epochs:02d}")

        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            train_m = self._train_epoch(epoch)
            val_m = self._evaluate_epoch(
                self.val_loader,
                split_name="val",
                epoch=epoch,
                use_ema=self.cfg.use_ema_eval,
            )

            current_lr = self.optimizer.param_groups[0]["lr"]
            self.scheduler.step()

            per_ds_str = " | ".join(
                f"{k}(MAE={v['mae']:.3f}, W1={v['within_1_acc']*100:.1f}%, QWK={v['qwk']:.3f})"
                for k, v in val_m.get("per_dataset_metrics", {}).items()
            )
            self.logger.info(
                f"Epoch [{epoch:02d}/{self.cfg.epochs:02d}]\n"
                f"  Train: Loss={train_m['loss']:.4f} (Dom={train_m.get('loss_domain', 0.0):.3f}) | MAE={train_m['mae']:.4f} | RMSE={train_m['rmse']:.4f} | "
                f"Within1={train_m['within_1_acc']*100:.1f}% | QWK={train_m['qwk']:.4f} | "
                f"Pred=[{train_m['pred_min']:.2f}..{train_m['pred_max']:.2f}, {train_m['pred_mean']:.2f}±{train_m['pred_std']:.2f}]\n"
                f"  Val  : Loss={val_m['loss']:.4f} | MAE={val_m['mae']:.4f} | RMSE={val_m['rmse']:.4f} | "
                f"Within1={val_m['within_1_acc']*100:.1f}% | Exact={val_m['exact_acc']*100:.1f}% | "
                f"QWK={val_m['qwk']:.4f} | RegimeAcc={val_m['regime_acc']*100:.1f}% | "
                f"DrowsyAUC={val_m['auc']:.4f} | Score={val_m['val_score']:.4f} | "
                f"Pred=[{val_m['pred_min']:.2f}..{val_m['pred_max']:.2f}, {val_m['pred_mean']:.2f}±{val_m['pred_std']:.2f}]\n"
                f"  Val Per-Dataset: {per_ds_str or 'N/A'}\n"
                f"  LR: {current_lr:.2e} | Time: {train_m['time']:.1f}s"
            )

            self._log_tensorboard(epoch, train_m, val_m, current_lr)

            self.history.append({
                "epoch": epoch,
                "train": train_m,
                "val": val_m,
                "lr": current_lr,
            })

            cur_val_score = float(val_m["val_score"])
            is_new_best = cur_val_score > self.best_val_score
            if is_new_best:
                self.best_val_score = cur_val_score
                self.best_val_mae = float(val_m["mae"])
                self.best_val_f1 = float(val_m["f1"])
                self.best_epoch = epoch
                self.early_stop_counter = 0
            else:
                self.early_stop_counter += 1

            checkpoint_payload = {
                "epoch": epoch,
                "model_state_dict": self.model.state_dict(),
                "ema_state_dict": self.ema.shadow,
                "ema_num_updates": self.ema.num_updates,
                "optimizer_state_dict": self.optimizer.state_dict(),
                "scheduler_state_dict": self.scheduler.state_dict(),
                "scaler_state_dict": self.scaler.state_dict(),
                "best_epoch": self.best_epoch,
                "best_val_score": self.best_val_score,
                "best_val_mae": self.best_val_mae,
                "best_val_f1": self.best_val_f1,
                "val_metrics": val_m,
                "train_metrics": train_m,
                "history": self.history,
                "config": self.cfg.to_dict(),
            }
            atomic_save(checkpoint_payload, self.save_dir / "last.pt")

            if self.cfg.save_freq > 0 and (epoch % self.cfg.save_freq == 0):
                atomic_save(checkpoint_payload, self.save_dir / f"epoch_{epoch:02d}.pt")

            if is_new_best:
                atomic_save(checkpoint_payload, self.save_dir / "best.pt")
                with open(self.save_dir / "best_metrics.json", "w", encoding="utf-8") as f:
                    json.dump({"best_epoch": epoch, "metrics": val_m}, f, indent=2)
                self.logger.info(
                    f"===> ĐÃ LƯU BEST.PT TẠI EPOCH {epoch} "
                    f"(Score: {self.best_val_score:.4f} | MAE: {self.best_val_mae:.4f} | QWK: {val_m['qwk']:.4f})"
                )

            if self.early_stop_counter >= int(self.cfg.early_stop_patience):
                self.logger.info(
                    f"===> DỪNG SỚM (Early Stopping) tại Epoch {epoch} "
                    f"vì val_score không cải thiện sau {self.early_stop_counter} epoch liên tiếp."
                )
                break

        self.logger.info("\n>>> HOÀN THÀNH HUẤN LUYỆN! <<<")
        self.logger.info(
            f"Checkpoint tốt nhất: Epoch {self.best_epoch} "
            f"(Score: {self.best_val_score:.4f} | MAE: {self.best_val_mae:.4f})"
        )

        test_metrics = self.evaluate_test()
        self.close()
        return test_metrics

    def close(self) -> None:
        """Đóng TensorBoard writer và các file handler của logger."""
        if hasattr(self, "writer") and self.writer is not None:
            try:
                self.writer.close()
            except Exception:
                pass
        if hasattr(self, "logger") and self.logger is not None:
            for h in list(self.logger.handlers):
                try:
                    h.close()
                except Exception:
                    pass
                self.logger.removeHandler(h)

    def evaluate_test(self, checkpoint_path: Optional[str | Path] = None) -> Dict[str, Any]:
        """Nạp checkpoint tốt nhất (best.pt) và đánh giá trên tập Test độc lập."""
        ckpt_target = Path(checkpoint_path) if checkpoint_path is not None else (self.save_dir / "best.pt")
        if not ckpt_target.is_file():
            fallback_last = self.save_dir / "last.pt"
            if fallback_last.is_file():
                ckpt_target = fallback_last
            else:
                self.logger.warning(f"Không tìm thấy checkpoint {ckpt_target} để đánh giá test.")
                return {}

        checkpoint = torch.load(ckpt_target, map_location=self.device, weights_only=False)
        eval_epoch = int(checkpoint.get("epoch", self.best_epoch))

        if self.cfg.use_ema_eval and "ema_state_dict" in checkpoint:
            ema_sd = checkpoint["ema_state_dict"]
            if isinstance(ema_sd, dict) and "shadow" in ema_sd and isinstance(ema_sd["shadow"], dict):
                ema_sd = ema_sd["shadow"]
            self.model.load_state_dict(ema_sd, strict=False)
            self.logger.info(f"\n--- ĐÁNH GIÁ TEST BẰNG EMA CHECKPOINT ({ckpt_target.name}, Epoch {eval_epoch}) ---")
        elif "model_state_dict" in checkpoint:
            self.model.load_state_dict(checkpoint["model_state_dict"], strict=False)
            self.logger.info(f"\n--- ĐÁNH GIÁ TEST BẰNG CHECKPOINT ({ckpt_target.name}, Epoch {eval_epoch}) ---")

        test_metrics, test_targets, test_pred_kss, test_logits = self._evaluate_epoch(
            self.test_loader,
            split_name="test",
            return_raw=True,
            use_ema=False,
        )

        self.logger.info(
            f"[TEST] Loss={test_metrics['loss']:.4f} | MAE={test_metrics['mae']:.4f} | "
            f"RMSE={test_metrics['rmse']:.4f} | Within1={test_metrics['within_1_acc']*100:.2f}% | "
            f"Exact={test_metrics['exact_acc']*100:.2f}% | QWK={test_metrics['qwk']:.4f} | "
            f"Pearson={test_metrics['pearson_r']:.4f} | RegimeAcc={test_metrics['regime_acc']*100:.2f}% | "
            f"DrowsyAUC={test_metrics['auc']:.4f} | DrowsyF1={test_metrics['f1']:.4f}"
        )

        test_samples = getattr(self.test_loader.dataset, "samples", [])
        per_sub_test: Dict[str, Dict[str, float]] = {}
        if len(test_samples) == len(test_targets) and len(test_samples) > 0:
            for d_name, d_m in test_metrics.get("per_dataset_metrics", {}).items():
                self.logger.info(
                    f"   [Dataset {d_name}] MAE={d_m['mae']:.4f} | RMSE={d_m['rmse']:.4f} | "
                    f"Within1={d_m['within_1_acc']*100:.2f}% | QWK={d_m['qwk']:.4f} | "
                    f"AUC={d_m['auc']:.4f} (N={d_m['num_samples']})"
                )

            test_ds_arr = np.array([str(s.get("dataset", "unknown")) for s in test_samples])
            test_sub_arr = np.array([str(s.get("subject_id", "unknown")) for s in test_samples])
            for sub_id in np.unique(test_sub_arr):
                m_s = (test_sub_arr == sub_id)
                s_ds = test_ds_arr[m_s][0]
                s_m = compute_kss_metrics(
                    test_targets[m_s],
                    test_pred_kss[m_s],
                    pred_logits=test_logits[m_s] if len(test_logits) == len(test_targets) else None,
                    kss_min=self.cfg.kss_min,
                    kss_max=self.cfg.kss_max,
                )
                per_sub_test[f"{s_ds}:{sub_id}"] = {
                    "mae": s_m["mae"],
                    "within_1_acc": s_m["within_1_acc"],
                    "qwk": s_m["qwk"],
                    "auc": s_m["auc"],
                    "num_samples": int(np.sum(m_s)),
                }

        output_payload = {
            "best_epoch": self.best_epoch if self.best_epoch > 0 else eval_epoch,
            "checkpoint": str(ckpt_target),
            "test_metrics": test_metrics,
            "per_subject_test": per_sub_test,
        }
        with open(self.save_dir / "test_metrics.json", "w", encoding="utf-8") as f:
            json.dump(output_payload, f, indent=2)

        return test_metrics


def main() -> None:
    """Điểm vào chính khi chạy `python train.py` với cấu hình mặc định từ TrainConfig."""
    config = TrainConfig()
    trainer = LiteFatTrainer(config)
    trainer.train()

In [12]:
# =====================================================================
# Cell 9: Cấu hình & Thực thi Pipeline Huấn luyện LiteFat_v1 trên Colab
# =====================================================================
# Giảm batch_size và tăng grad_accum_steps (vd: batch_size=2, grad_accum_steps=2
# hoặc batch_size=1, grad_accum_steps=4) để giảm đỉnh VRAM và tránh CUDA OOM.
config = TrainConfig()
trainer = LiteFatTrainer(config)
test_metrics = trainer.train()

[2026-11-17 13:27:43] [INFO] ========================================================================
[2026-11-17 13:27:43] [INFO] KHỞI TẠO LITEFAT_V1 TRAINER (Thiết bị: cuda)
[2026-11-17 13:27:43] [INFO] Ngữ cảnh thị giác (use_visual_context): False
[2026-11-17 13:27:43] [INFO] Lấy mẫu: 10 FPS (dt=0.1s) | Model Window=60.0s / Stride=50.0s | Dataset Window=None
[2026-11-17 13:27:43] [INFO] Batch size: 2 | Grad accum steps: 8 | Effective batch size: 16
[2026-11-17 13:27:43] [INFO] Bộ dữ liệu: Tất cả (dataset_merged_split.csv)
[2026-11-17 13:27:43] [INFO] ========================================================================
[2026-11-17 13:27:43] [INFO] Đang khởi tạo DataLoaders...
[2026-11-17 13:27:44] [INFO] [Cache] Đã nạp 518 mẫu từ train.pt vào Shared Cache Pool (Tổng: 518).
[2026-11-17 13:27:44] [INFO] [Cache] Đã nạp 66 mẫu từ val.pt vào Shared Cache Pool (Tổng: 584).
[2026-11-17 13:27:44] [INFO] [Cache] Đã nạp 65 mẫu từ test.pt vào Shared Cache Pool (Tổng: 649).
[2026-11-17 13:27

100%|██████████| 9.83M/9.83M [00:00<00:00, 221MB/s]


[2026-11-17 13:27:44] [INFO] Tổng số tham số: 2,593,496 | Khả học: 1,666,488 (MobileNetV3: 0 / 927,008)
[2026-11-17 13:27:44] [INFO] 
>>> BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN LITEFAT_V1 (KSS 1-9) <<<


[2026-11-17 13:32:14] [INFO] Epoch [01/30]
  Train: Loss=5.3551 | MAE=2.2477 | RMSE=2.5681 | Within1=24.9% | QWK=0.0341 | Pred=[4.30..5.88, 5.34±0.27]
  Val  : Loss=5.4480 | MAE=2.3036 | RMSE=2.6233 | Within1=19.7% | Exact=12.1% | QWK=-0.0409 | RegimeAcc=22.7% | DrowsyAUC=0.4693 | Score=0.2287 | Pred=[4.46..5.80, 5.22±0.42]
  Val Per-Dataset: UL-DD(MAE=2.451, W1=10.0%, QWK=-0.030) | UTA-RLDD(MAE=2.181, W1=27.8%, QWK=-0.063)
  LR: 1.50e-04 | Time: 260.1s


[2026-11-17 13:32:14] [INFO] ===> ĐÃ LƯU BEST.PT TẠI EPOCH 1 (Score: 0.2287 | MAE: 2.3036 | QWK: -0.0409)


[2026-11-17 13:36:44] [INFO] Epoch [02/30]
  Train: Loss=5.2327 | MAE=2.1835 | RMSE=2.5590 | Within1=28.0% | QWK=0.0507 | Pred=[4.06..6.38, 5.42±0.50]
  Val  : Loss=5.4508 | MAE=2.2851 | RMSE=2.6444 | Within1=16.7% | Exact=15.2% | QWK=0.0571 | RegimeAcc=33.3% | DrowsyAUC=0.4888 | Score=0.2386 | Pred=[3.86..6.16, 5.16±0.67]
  Val Per-Dataset: UL-DD(MAE=2.471, W1=6.7%, QWK=-0.001) | UTA-RLDD(MAE=2.130, W1=25.0%, QWK=0.102)
  LR: 3.00e-04 | Time: 260.3s


... (35 outputs left out)

[2026-11-17 15:38:05] [INFO] Epoch [29/30]
  Train: Loss=3.8150 | MAE=1.5585 | RMSE=1.9219 | Within1=39.2% | QWK=0.5858 | Pred=[2.10..8.17, 5.20±1.59]
  Val  : Loss=4.8296 | MAE=1.9277 | RMSE=2.5030 | Within1=39.4% | Exact=22.7% | QWK=0.3512 | RegimeAcc=48.5% | DrowsyAUC=0.7727 | Score=0.4308 | Pred=[2.71..8.06, 6.07±1.39]
  Val Per-Dataset: UL-DD(MAE=1.923, W1=43.3%, QWK=0.429) | UTA-RLDD(MAE=1.931, W1=36.1%, QWK=0.261)
  LR: 3.76e-06 | Time: 257.8s


[2026-11-17 15:42:35] [INFO] Epoch [30/30]
  Train: Loss=3.7297 | MAE=1.5136 | RMSE=1.8866 | Within1=41.1% | QWK=0.5932 | Pred=[2.18..8.26, 5.21±1.58]
  Val  : Loss=4.4895 | MAE=1.7833 | RMSE=2.3528 | Within1=43.9% | Exact=22.7% | QWK=0.4023 | RegimeAcc=56.1% | DrowsyAUC=0.8020 | Score=0.4742 | Pred=[2.47..8.03, 5.61±1.64]
  Val Per-Dataset: UL-DD(MAE=1.606, W1=50.0%, QWK=0.519) | UTA-RLDD(MAE=1.931, W1=38.9%, QWK=0.267)
  LR: 9.43e-07 | Time: 259.4s


[2026-11-17 15:42:35] [INFO] 
>>> HOÀN THÀNH HUẤN LUYỆN! <<<
[2026-11-17 15:42:35] [INFO] Checkpoint tốt nhất: Epoch 28 (Score: 0.4762 | MAE: 1.8331)
[2026-11-17 15:42:35] [INFO] 
--- ĐÁNH GIÁ TEST BẰNG EMA CHECKPOINT (best.pt, Epoch 28) ---


[2026-11-17 15:42:45] [INFO] [TEST] Loss=4.8275 | MAE=1.9957 | RMSE=2.4197 | Within1=27.69% | Exact=10.77% | QWK=0.2215 | Pearson=0.2435 | RegimeAcc=40.00% | DrowsyAUC=0.5944 | DrowsyF1=0.4286
[2026-11-17 15:42:45] [INFO]    [Dataset UL-DD] MAE=1.4215 | RMSE=1.6178 | Within1=27.59% | QWK=0.5609 | AUC=0.7262 (N=29)
[2026-11-17 15:42:45] [INFO]    [Dataset UTA-RLDD] MAE=2.4582 | RMSE=2.9091 | Within1=27.78% | QWK=0.0205 | AUC=0.4653 (N=36)


In [13]:
# =====================================================================
# Cell 10: Theo dõi quá trình huấn luyện bằng TensorBoard trên Colab
# =====================================================================
%load_ext tensorboard
%tensorboard --logdir ./checkpoints/LiteFat_v1/logs

<IPython.core.display.Javascript object>

In [14]:
# =====================================================================
# Cell 11: Nén Checkpoints & Logs thành .zip + Tùy chọn sao lưu Google Drive
# =====================================================================
import shutil
from pathlib import Path

_ckpt_dir = Path(config.checkpoint_dir if "config" in globals() else "./checkpoints/LiteFat_v1")
_zip_name = "litefat_v1_results"

if _ckpt_dir.exists():
    archive_path = shutil.make_archive(
        _zip_name,
        "zip",
        root_dir=_ckpt_dir.parent,
        base_dir=_ckpt_dir.name,
    )
    print(f"Đã nén toàn bộ checkpoints & logs tại: {archive_path}")

    # Tùy chọn: Sao lưu sang Google Drive nếu đang chạy trên Colab
    BACKUP_TO_DRIVE = True  # Đổi thành False nếu chỉ muốn lưu cục bộ trên runtime Colab
    if BACKUP_TO_DRIVE:
        try:
            from google.colab import drive

            drive_root = Path("/content/drive/MyDrive")
            if not drive_root.exists():
                drive.mount("/content/drive")
            dest_dir = drive_root / "LiteFat_v1"
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest_zip = dest_dir / Path(archive_path).name
            shutil.copy2(archive_path, dest_zip)
            print(f"Đã sao lưu lên Google Drive: {dest_zip}")
        except ImportError:
            print("Không phải môi trường Google Colab, bỏ qua bước sao lưu Google Drive.")
else:
    print(f"Không tìm thấy thư mục checkpoint: {_ckpt_dir}")

Đã nén toàn bộ checkpoints & logs tại: /content/litefat_v1_results.zip
Đã sao lưu lên Google Drive: /content/drive/MyDrive/LiteFat_v1/litefat_v1_results.zip
